# PREBORN: MSDS to OMOP

This notebook turns one site's **Maternity Services Data Set (MSDS v2)** tables into the PREBORN OMOP
dataset: OMOP CDM v5.4 tables plus the PREBORN `pregnancy` and `infant` extension tables. It works for
Barts (site R1H) or any other site that holds the national MSDS tables. The Barts extension
(`20_preborn_barts`) starts from the dataset this notebook publishes.

**How it runs, top to bottom**

1. Settings: which site, where its MSDS tables are, and where to publish.
2. Small helpers used everywhere: id minting, SQL quoting.
3. The shape of every output table.
4. The MSDS mapping spec: one rule per MSDS column (or a written reason to skip it).
5. Concept lookup: turn SNOMED codes and the spec's questions and answers into standard OMOP concepts.
6. People, pregnancies and babies.
7. Clinical facts: every coded MSDS row and every spec rule becomes one row in a working table.
8. Visits, ward and neonatal stays, deaths and home areas (LSOA).
9. Cleaning: decide which staged rows cannot be published, record why, and remove them.
10. Assemble the OMOP tables in a run-specific "pub" schema.
11. Derived tables: observation periods, CDM source row, the bundled vocabulary.
12. Quality checks on the staged tables.
13. Publish to the target (locked, all-or-nothing, rolled back if the checks fail afterwards).
14. Rebuild the target's bundled vocabulary and tidy up.

**Working schemas.** Each run writes to two scratch schemas named after the run tag:
`{target}_pub_{run_tag}` holds the finished OMOP tables before publishing, and
`{target}_pub_{run_tag}_stg` holds working tables. Both are dropped at the end of a successful run.

**Another site.** Set `site_id` and `source_schema`. The 12 core MSDS tables must exist; any other MSDS
table the site does not hold is skipped, and the rules that read it are reported.

## 1. Settings

In [0]:
import json
import re
import uuid
from datetime import date, datetime, timezone

dbutils.widgets.text("site_id", "R1H")
dbutils.widgets.text("source_schema", "4_prod.raw")     # where this site's msd* tables live
dbutils.widgets.text("target", "5_projects.preborn_msds")    # the published PREBORN MSDS schema
dbutils.widgets.text("release_date", "")                # blank = today; facts dated after it are dropped
dbutils.widgets.text("run_tag", "")                     # blank = a new random tag
dbutils.widgets.dropdown("update_mapping_caches", "no", ["no", "yes"])  # see sections 5d and 5e
dbutils.widgets.text("adjudication_table", "3_lookup.embeddings.llm_adjudications")   # see section 5e
dbutils.widgets.dropdown("keep_working_schemas", "no", ["no", "yes"])   # yes = keep pub/stg for inspection

SITE_ID = dbutils.widgets.get("site_id").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET = dbutils.widgets.get("target").strip()
RELEASE_DATE = dbutils.widgets.get("release_date").strip() or date.today().isoformat()
UPDATE_MAPPING_CACHES = dbutils.widgets.get("update_mapping_caches") == "yes"
ADJUDICATION_TABLE = dbutils.widgets.get("adjudication_table").strip()
KEEP_WORKING_SCHEMAS = dbutils.widgets.get("keep_working_schemas") == "yes"

assert SITE_ID, "set the site_id widget"
assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", RELEASE_DATE), f"release_date must be YYYY-MM-DD, got {RELEASE_DATE!r}"

# The run tag names this run's working schemas, so two runs never share scratch tables.
RUN_TAG = dbutils.widgets.get("run_tag").strip().lower()
if not RUN_TAG:
    site_tag = re.sub(r"[^a-z0-9]+", "_", SITE_ID.lower()).strip("_") or "site"
    RUN_TAG = f"{site_tag}_{uuid.uuid4().hex[:12]}"
assert re.fullmatch(r"[a-z0-9_]+", RUN_TAG), f"run_tag may only use a-z, 0-9 and _: {RUN_TAG!r}"

PUB = f"{TARGET}_pub_{RUN_TAG}"     # finished OMOP tables for this run, before publishing
STG = f"{PUB}_stg"                  # working tables for this run

# The OMOP vocabulary every concept lookup reads.
VOCAB = "4_prod.omop"

print(f"site {SITE_ID} | source {SOURCE_SCHEMA} -> target {TARGET} | release {RELEASE_DATE} | run tag {RUN_TAG}")

In [0]:
# Every MSDS v2 table. Each column of each table is either mapped by a rule, used by the core build, or
# skipped with a written reason (section 4; tests/test_msds_spec.py checks that nothing is left out).
MSDS_TABLES = [
    "msd001motherdemo", "msd002gp", "msd003socperscircum", "msd004overseasvischarcat",
    "msd101pregbook", "msd102matcareplan", "msd103datingscan", "msd104codedscoreasspreg",
    "msd105provdiagnosispreg", "msd106diagnosispreg", "msd107medhistory", "msd109findobsmother",
    "msd201carecontactpreg", "msd202careactivitypreg",
    "msd301labdel", "msd302careactlabdel",
    "msd401babydemo", "msd402neoadmiss", "msd403provdiagneo", "msd404diagneo", "msd405careactbaby",
    "msd501hospprovspell", "msd502hospspellcomm", "msd503wardstay", "msd504assigncareprof",
]

# The tables the core build reads directly. A site must hold all of these.
MSDS_CORE_TABLES = [
    "msd001motherdemo", "msd101pregbook", "msd106diagnosispreg", "msd107medhistory",
    "msd109findobsmother", "msd201carecontactpreg", "msd202careactivitypreg", "msd301labdel",
    "msd302careactlabdel", "msd401babydemo", "msd404diagneo", "msd405careactbaby",
]


def find_source_tables():
    """Map each MSDS table name to its full source name, or to None when the site does not hold it."""
    catalog, schema = SOURCE_SCHEMA.split(".")
    present = {row.tableName for row in spark.sql(f"SHOW TABLES IN `{catalog}`.`{schema}`").collect()}
    src = {}
    for table in MSDS_TABLES:
        if table in present:
            src[table] = f"{SOURCE_SCHEMA}.{table}"
        else:
            src[table] = None
    return src


SRC = find_source_tables()

missing_core = [t for t in MSDS_CORE_TABLES if SRC[t] is None]
assert not missing_core, f"{SOURCE_SCHEMA} is missing core MSDS tables: {missing_core}"
not_held = [t for t in MSDS_TABLES if SRC[t] is None]
print("optional MSDS tables not held (their rules are skipped):", not_held or "none")

# Ethnicity crosswalk override for another site, as a SQL CASE expression with a {col} placeholder.
# None = the NHS ethnic category crosswalk in section 6.
RACE_CASE_OVERRIDE = None

## 2. Small helpers

**Ids.** Every PREBORN id (person, pregnancy, visit, fact, ...) is *calculated* from the source key rather
than counted. The same MSDS row therefore gets the same id on every run and at every site's release, which
downstream products (the synthetic data, linked extracts) rely on. Ids must never change, so the recipe in
`mint_id` must never change either.

In [0]:
def sql_str(value):
    """Escape a Python value for use inside a SQL '...' string literal (doubles any single quote)."""
    return str(value).replace("'", "''")


SITE = sql_str(SITE_ID)   # the site id, ready to drop into SQL as '{SITE}'


def _normalised_key_part(sql_expr):
    """SQL for one id key part: trimmed text, with blank or NULL replaced by '~' so it still hashes."""
    return f"coalesce(nullif(trim(cast({sql_expr} as string)), ''), '~')"


def mint_id(kind, *key_sql):
    """SQL expression for a deterministic PREBORN id.

    The id is SHA-256 of 'kind|site|key1|key2|...', with the first 15 hex digits read as a number.
    `kind` names what is being identified ('person', 'pregnancy', 'visit', ...), so different kinds of thing
    never share an id. `key_sql` are SQL expressions (column names, or quoted literals such as "'X1'"),
    not Python values.
    """
    site_literal = "'" + sql_str(SITE_ID) + "'"
    parts = [f"'{kind}'", _normalised_key_part(site_literal)]
    for key in key_sql:
        parts.append(_normalised_key_part(key))
    joined = ", ".join(parts)
    return f"conv(substr(sha2(concat_ws('|', {joined}), 256), 1, 15), 16, 10)"


def check_mint_id_matches_python():
    """Stop the run if Spark's id recipe ever stops matching the same recipe done in plain Python."""
    import hashlib
    key = f"pregnancy|{SITE_ID}|TESTKEY"
    expected = int(hashlib.sha256(key.encode()).hexdigest()[:15], 16)
    got = spark.sql(f"SELECT {mint_id('pregnancy', chr(39) + 'TESTKEY' + chr(39))} AS v").first()["v"]
    assert int(got) == expected, f"id minting has changed: Spark gave {got}, Python gave {expected}"
    print("id minting check passed for site", SITE_ID)


def scalar(sql):
    """Run a query that returns one value and return that value."""
    return spark.sql(sql).first()[0]


def show(sql):
    """Run a query and print the result as a small text table."""
    print(spark.sql(sql).toPandas().to_string(index=False))


def schema_exists(schema):
    """True when catalog.schema exists."""
    catalog, name = schema.split(".", 1)
    return spark.sql(f"SHOW SCHEMAS IN `{catalog}` LIKE '{sql_str(name)}'").count() > 0


def table_columns(schema, table):
    """The column names of schema.table, in table order."""
    catalog, name = schema.split(".", 1)
    rows = spark.sql(f"""
      SELECT column_name FROM {catalog}.information_schema.columns
      WHERE table_schema = '{name}' AND table_name = '{table}'
      ORDER BY ordinal_position""").collect()
    columns = [row["column_name"] for row in rows]
    if not columns:
        raise RuntimeError(f"no columns found for {schema}.{table}")
    return columns


def latest_version(full_table_name):
    """The table's current Delta version."""
    return int(spark.sql(f"DESCRIBE HISTORY {full_table_name} LIMIT 1").first()["version"])


def assert_unique(table, id_column, where="true"):
    """Stop the run if id_column is not unique in table (an id collision would corrupt every join)."""
    rows, distinct_ids = spark.sql(
        f"SELECT count(*), count(DISTINCT {id_column}) FROM {table} WHERE {where}").first()
    assert rows == distinct_ids, f"{table}.{id_column} is not unique: {rows} rows, {distinct_ids} ids"
    return rows


check_mint_id_matches_python()

## 3. The shape of every output table

The standard OMOP CDM v5.4 columns (from the official SQL Server DDL, with `integer` as BIGINT because
minted ids exceed 32 bits), plus PREBORN lineage columns:

- `site_id` on every clinical table, so several sites can share one schema;
- `pregnancy_id` and `source_table` on fact tables, `person_role` (mother / infant) on `person`.

The `pregnancy` and `infant` tables are the PREBORN maternity extension. The vocabulary tables are created
empty here and filled in section 11.

In [0]:
CDM_TABLES = {
    "person": """
      person_id BIGINT, gender_concept_id BIGINT, year_of_birth BIGINT,
      month_of_birth BIGINT, day_of_birth BIGINT, birth_datetime TIMESTAMP,
      race_concept_id BIGINT, ethnicity_concept_id BIGINT, location_id BIGINT,
      provider_id BIGINT, care_site_id BIGINT, person_source_value STRING,
      gender_source_value STRING, gender_source_concept_id BIGINT,
      race_source_value STRING, race_source_concept_id BIGINT,
      ethnicity_source_value STRING, ethnicity_source_concept_id BIGINT,
      site_id STRING, person_role STRING""",
    "observation_period": """
      observation_period_id BIGINT, person_id BIGINT,
      observation_period_start_date DATE, observation_period_end_date DATE,
      period_type_concept_id BIGINT, site_id STRING""",
    "condition_occurrence": """
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, condition_start_datetime TIMESTAMP,
      condition_end_date DATE, condition_end_datetime TIMESTAMP,
      condition_type_concept_id BIGINT, condition_status_concept_id BIGINT,
      stop_reason STRING, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, condition_source_value STRING,
      condition_source_concept_id BIGINT, condition_status_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING""",
    "procedure_occurrence": """
      procedure_occurrence_id BIGINT, person_id BIGINT, procedure_concept_id BIGINT,
      procedure_date DATE, procedure_datetime TIMESTAMP,
      procedure_end_date DATE, procedure_end_datetime TIMESTAMP,
      procedure_type_concept_id BIGINT, modifier_concept_id BIGINT, quantity BIGINT,
      provider_id BIGINT, visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      procedure_source_value STRING, procedure_source_concept_id BIGINT,
      modifier_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING""",
    "measurement": """
      measurement_id BIGINT, person_id BIGINT, measurement_concept_id BIGINT,
      measurement_date DATE, measurement_datetime TIMESTAMP, measurement_time STRING,
      measurement_type_concept_id BIGINT, operator_concept_id BIGINT,
      value_as_number DOUBLE, value_as_concept_id BIGINT, unit_concept_id BIGINT,
      range_low DOUBLE, range_high DOUBLE, provider_id BIGINT,
      visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      measurement_source_value STRING, measurement_source_concept_id BIGINT,
      unit_source_value STRING, unit_source_concept_id BIGINT, value_source_value STRING,
      measurement_event_id BIGINT, meas_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING""",
    "observation": """
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      observation_date DATE, observation_datetime TIMESTAMP,
      observation_type_concept_id BIGINT, value_as_number DOUBLE,
      value_as_string STRING, value_as_concept_id BIGINT, qualifier_concept_id BIGINT,
      unit_concept_id BIGINT, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, observation_source_value STRING,
      observation_source_concept_id BIGINT, unit_source_value STRING,
      qualifier_source_value STRING, value_source_value STRING,
      observation_event_id BIGINT, obs_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING""",
    "visit_occurrence": """
      visit_occurrence_id BIGINT, person_id BIGINT, visit_concept_id BIGINT, visit_start_date DATE,
      visit_start_datetime TIMESTAMP, visit_end_date DATE, visit_end_datetime TIMESTAMP,
      visit_type_concept_id BIGINT, provider_id BIGINT, care_site_id BIGINT, visit_source_value STRING,
      visit_source_concept_id BIGINT, admitted_from_concept_id BIGINT, admitted_from_source_value STRING,
      discharged_to_concept_id BIGINT, discharged_to_source_value STRING,
      preceding_visit_occurrence_id BIGINT, site_id STRING, pregnancy_id BIGINT""",
    "visit_detail": """
      visit_detail_id BIGINT, person_id BIGINT, visit_detail_concept_id BIGINT,
      visit_detail_start_date DATE, visit_detail_start_datetime TIMESTAMP, visit_detail_end_date DATE,
      visit_detail_end_datetime TIMESTAMP, visit_detail_type_concept_id BIGINT, provider_id BIGINT,
      care_site_id BIGINT, visit_detail_source_value STRING, visit_detail_source_concept_id BIGINT,
      admitted_from_concept_id BIGINT, admitted_from_source_value STRING, discharged_to_source_value STRING,
      discharged_to_concept_id BIGINT, preceding_visit_detail_id BIGINT, parent_visit_detail_id BIGINT,
      visit_occurrence_id BIGINT, site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING,
      source_table STRING""",
    "death": """
      person_id BIGINT, death_date DATE, death_datetime TIMESTAMP, death_type_concept_id BIGINT,
      cause_concept_id BIGINT, cause_source_value STRING, cause_source_concept_id BIGINT, site_id STRING""",
    "location": """
      location_id BIGINT, address_1 STRING, address_2 STRING, city STRING, state STRING, zip STRING,
      county STRING, location_source_value STRING, country_concept_id BIGINT, country_source_value STRING,
      latitude DOUBLE, longitude DOUBLE, site_id STRING""",
    "episode": """
      episode_id BIGINT, person_id BIGINT, episode_concept_id BIGINT,
      episode_start_date DATE, episode_start_datetime TIMESTAMP,
      episode_end_date DATE, episode_end_datetime TIMESTAMP, episode_parent_id BIGINT,
      episode_number BIGINT, episode_object_concept_id BIGINT,
      episode_type_concept_id BIGINT, episode_source_value STRING,
      episode_source_concept_id BIGINT, pregnancy_id BIGINT, site_id STRING""",
    "episode_event": """
      episode_id BIGINT, event_id BIGINT, episode_event_field_concept_id BIGINT,
      source_table STRING, site_id STRING""",
    "fact_relationship": """
      domain_concept_id_1 BIGINT, fact_id_1 BIGINT, domain_concept_id_2 BIGINT,
      fact_id_2 BIGINT, relationship_concept_id BIGINT, site_id STRING""",
    "cdm_source": """
      cdm_source_name STRING, cdm_source_abbreviation STRING, cdm_holder STRING,
      source_description STRING, source_documentation_reference STRING,
      cdm_etl_reference STRING, source_release_date DATE, cdm_release_date DATE,
      cdm_version STRING, cdm_version_concept_id BIGINT, vocabulary_version STRING""",
    # PREBORN maternity extension: one row per pregnancy, one row per infant.
    "pregnancy": """
      person_id BIGINT, pregnancy_id BIGINT,
      pregnancy_start_date DATE, pregnancy_end_date DATE,
      gestational_length_in_day INT,
      pregnancy_outcome_concept_id BIGINT, pregnancy_mode_delivery_concept_id BIGINT,
      pregnancy_single_concept_id BIGINT, pregnancy_marital_status_concept_id BIGINT,
      pregnancy_number_fetuses INT, pregnancy_number_liveborn INT,
      prev_pregnancy_parity_concept_id BIGINT, prev_pregnancy_gravidity INT,
      prev_livebirth_number INT, prev_still_misc_number INT,
      pre_pregnancy_bmi DOUBLE, pregnancy_folic_concept_id BIGINT,
      pregnancy_outcome_source_value STRING, pregnancy_mode_delivery_source_value STRING,
      site_id STRING,
      pre_pregnancy_bmi_source STRING, pregnancy_marital_status_source_value STRING""",
    "infant": """
      pregnancy_id BIGINT, infant_id BIGINT,
      birth_outcome_concept_id BIGINT, birth_weight DOUBLE,
      birth_con_malformation_concept_id BIGINT, birth_apgar INT,
      labour_delivery_id STRING, site_id STRING,
      birth_weight_source STRING, birth_apgar_source STRING""",
    # The bundled vocabulary (filled in section 11).
    "concept": """
      concept_id BIGINT, concept_name STRING, domain_id STRING, vocabulary_id STRING,
      concept_class_id STRING, standard_concept STRING, concept_code STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING""",
    "concept_relationship": """
      concept_id_1 BIGINT, concept_id_2 BIGINT, relationship_id STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING""",
    "concept_ancestor": """
      ancestor_concept_id BIGINT, descendant_concept_id BIGINT,
      min_levels_of_separation BIGINT, max_levels_of_separation BIGINT""",
    "concept_synonym": """
      concept_id BIGINT, concept_synonym_name STRING, language_concept_id BIGINT""",
    "vocabulary": """
      vocabulary_id STRING, vocabulary_name STRING, vocabulary_reference STRING,
      vocabulary_version STRING, vocabulary_concept_id BIGINT""",
    "domain": """
      domain_id STRING, domain_name STRING, domain_concept_id BIGINT""",
    "concept_class": """
      concept_class_id STRING, concept_class_name STRING, concept_class_concept_id BIGINT""",
    "relationship": """
      relationship_id STRING, relationship_name STRING, is_hierarchical STRING,
      defines_ancestry STRING, reverse_relationship_id STRING, relationship_concept_id BIGINT""",
    # How every MSDS spec question, answer and unit was mapped to a concept, with its method (section 5).
    "_msds_concept_map": """
      site_id STRING, rule_id STRING, role STRING, src_code STRING, src_text STRING,
      concept_id BIGINT, domain_id STRING, method STRING, model_version STRING,
      cosine DOUBLE, margin DOUBLE, rule_ref STRING""",
}


def column_names(table):
    """The column names of a CDM_TABLES entry, in order."""
    return [column.split()[0] for column in CDM_TABLES[table].split(",")]


def create_cdm_tables(schema):
    """Create schema and every CDM_TABLES table in it (tables that already exist are left alone)."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    for table, columns in CDM_TABLES.items():
        spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.{table} ({columns}) USING DELTA")


# The tables this notebook publishes, per site, to the target.
MSDS_PUBLISH_TABLES = [
    "person", "observation_period",
    "condition_occurrence", "procedure_occurrence", "measurement", "observation",
    "visit_occurrence", "visit_detail", "death", "location",
    "episode", "episode_event", "fact_relationship",
    "pregnancy", "infant", "_msds_concept_map",
]

# The bundled vocabulary tables (rebuilt for the whole target, not per site).
VOCAB_TABLES = [
    "concept", "concept_relationship", "concept_ancestor", "concept_synonym",
    "vocabulary", "domain", "concept_class", "relationship",
]

**Working tables.** The build first stages people, pregnancies and infants in the working schema (`STG`)
in a narrower shape, and records every row it declines to publish in `STG._dropped`
(rule, row key, reason). Nothing is ever dropped without a row here.

In [0]:
STAGING_TABLES = {
    "person": """
      person_id BIGINT, person_source_value STRING, person_role STRING,
      gender_source_value STRING, birth_datetime TIMESTAMP,
      race_concept_id BIGINT, race_source_value STRING, site_id STRING""",
    "pregnancy": """
      person_id BIGINT, pregnancy_id BIGINT,
      pregnancy_start_date DATE, pregnancy_end_date DATE,
      gestational_length_in_day INT,
      pregnancy_outcome_concept_id BIGINT, pregnancy_mode_delivery_concept_id BIGINT,
      pregnancy_single_concept_id BIGINT, pregnancy_marital_status_concept_id BIGINT,
      pregnancy_number_fetuses INT, pregnancy_number_liveborn INT,
      prev_pregnancy_parity_concept_id BIGINT, prev_pregnancy_gravidity INT,
      prev_livebirth_number INT, prev_still_misc_number INT,
      pre_pregnancy_bmi DOUBLE, pregnancy_folic_concept_id BIGINT,
      pregnancy_outcome_source_value STRING, pregnancy_mode_delivery_source_value STRING,
      site_id STRING""",
    "infant": """
      pregnancy_id BIGINT, infant_id BIGINT,
      birth_outcome_concept_id BIGINT, birth_weight DOUBLE,
      birth_con_malformation_concept_id BIGINT, birth_apgar INT,
      labour_delivery_id STRING, site_id STRING""",
    "_dropped": """
      rule_id STRING, fact_natural_key STRING, reason STRING""",
}


def record_dropped(select_sql):
    """Append rows to STG._dropped. select_sql must return (rule_id, fact_natural_key, reason)."""
    spark.sql(f"INSERT INTO {STG}._dropped (rule_id, fact_natural_key, reason) {select_sql}")

In [0]:
# Create this run's scratch schemas. A run tag is used once: an existing schema means a clash.
assert not schema_exists(PUB) and not schema_exists(STG), f"working schemas for run tag {RUN_TAG} already exist"
create_cdm_tables(PUB)
spark.sql(f"CREATE SCHEMA {STG}")
for table, columns in STAGING_TABLES.items():
    spark.sql(f"CREATE TABLE {STG}.{table} ({columns}) USING DELTA")
print("created", PUB, "and", STG)

## 4. The MSDS mapping spec

Every column of every MSDS table has exactly one written decision:

- **a rule** in `MSDS_COLUMN_SPEC` turns it into OMOP rows;
- **core**: `MSDS_CORE_USED` lists the columns the core build reads directly (keys, dates, the coded
  diagnosis/finding/activity tables, the pregnancy and infant extension columns);
- **skip**: `MSDS_SKIP` gives the reason a column is deliberately not carried.

`tests/test_msds_spec.py` fails if any MSDS column has no decision.

**What a rule says** (one dict per rule):

| Field | Meaning |
|---|---|
| `rule_id` | stable name `MSDS-<table>-<label>`; stored on every row the rule produces |
| `table`, `columns` | where the value comes from (the first column is the value) |
| `kind` | the shape of the fact (below) |
| `grain` | whose fact and at what level: mother, pregnancy, carecontact, labourdelivery, baby, spell. Baby rules land on the infant unless `person` says 'mother' |
| `date` | the event-date column, or a list tried in order; `date_fallback: baby_birth` adds the delivery's birth time as a last resort |
| `target` | the *question* concept as `{vocab, code, name}`; resolved and checked in section 5, never a typed concept_id |
| `code_map` | NHS Data Dictionary (May 2024) code -> `{text}`, optionally with the SNOMED code it means |
| `aux` | helper columns: `qualifier` (decoded via a code_map), `scheme` (ignored, always SNOMED), others kept as `name=value` |
| `only_codes` | emit only these raw codes |
| `unit`, `scale` | UCUM unit, and a multiplier when MSDS uses another unit (height in metres x 100 = cm) |
| `answer_roots` | opt-in to the embedding fallback for answers: candidate concepts are the standard descendants of these roots |
| `condition_status` | condition_status concept for diagnosis rules |
| `no_target_reason` | why a fact rule has no question concept (it is published with concept 0) |

**Kinds**

| Kind | OMOP result |
|---|---|
| `obs_coded` | observation: concept = the question, value_as_concept = the decoded answer, value_source_value = the raw code |
| `obs_flag` | observation: Y/N decoded to Yes 4188539 / No 4188540 |
| `obs_number` | observation with value_as_number |
| `obs_date` | observation whose answer is a date (value_as_string, yyyy-MM-dd) |
| `meas_number` | measurement with value_as_number and unit (times `scale`) |
| `obs_snomed`, `meas_snomed` | the column itself is a SNOMED code; its concept's domain picks the table |
| `condition` | condition_occurrence from a SNOMED diagnosis column |
| `procedure` | procedure_occurrence |
| `visit`, `visit_detail`, `death`, `location` | structural rows, built in section 8 |

In [0]:
# >>> SPEC
MSDS_COLUMN_SPEC = [{'rule_id': 'MSDS-001-DEATH',
  'table': 'msd001motherdemo',
  'columns': ['PERSONDEATHDATETIMEMOTHER'],
  'kind': 'death',
  'grain': 'mother',
  'date': None,
  'target': None,
  'code_map': None,
  'note': 'death_type 32817; death date = PERSONDEATHDATETIMEMOTHER'},
 {'rule_id': 'MSDS-001-LOC',
  'table': 'msd001motherdemo',
  'columns': ['POSTCODE'],
  'kind': 'location',
  'grain': 'mother',
  'date': None,
  'target': None,
  'code_map': None,
  'note': 'postcode -> 3_lookup.ons.onspd -> LSOA only (D6); postcode itself never published'},
 {'rule_id': 'MSDS-003-SOC',
  'table': 'msd003socperscircum',
  'columns': ['SOCPERSNOMED'],
  'kind': 'obs_snomed',
  'grain': 'mother',
  'date': 'SOCPERDATE',
  'target': None,
  'code_map': None,
  'note': 'SNOMED social/personal circumstance used as-is (standard or Maps to)'},
 {'rule_id': 'MSDS-004-OVS',
  'table': 'msd004overseasvischarcat',
  'columns': ['OVSVISCHCAT'],
  'kind': 'obs_coded',
  'grain': 'mother',
  'date': 'OVSVISCHCATAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '198311000000106', 'name': 'Overseas visitor'},
  'code_map': {'A': {'text': 'Standard NHS-funded PATIENT'},
               'B': {'text': 'Immigration Health Surcharge payee'},
               'C': {'text': 'Charge-exempt Overseas Visitor (European Economic Area)'},
               'D': {'text': 'Chargeable European Economic Area PATIENT'},
               'E': {'text': 'Charge-exempt Overseas Visitor (non- European Economic Area)'},
               'F': {'text': 'Chargeable non- European Economic Area PATIENT'},
               'P': {'text': 'Decision Pending on OVERSEAS VISITOR CHARGING CATEGORY'}}},
 {'rule_id': 'MSDS-101-EDD',
  'table': 'msd101pregbook',
  'columns': ['EDDAGREED'],
  'kind': 'obs_date',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'LOINC', 'code': '11778-8', 'name': 'Delivery date Estimated'},
  'code_map': {'01': {'text': 'Last Menstrual Period (LMP) date as stated by the mother'},
               '02': {'text': 'Last Menstrual Period (LMP) date confirmed by Ultrasound Scan In Pregnancy'},
               '03': {'text': 'Ultrasound Scan In Pregnancy dating measurements'},
               '04': {'text': 'Clinical assessment'}},
  'aux': {'qualifier': 'EDDMETHOD'},
  'note': 'value_as_datetime = EDDAGREED; EDD method rides as qualifier_source_value / qualifier via code_map'},
 {'rule_id': 'MSDS-101-FIRSTCON',
  'table': 'msd101pregbook',
  'columns': ['PREGFIRSTCONDATE'],
  'kind': 'obs_date',
  'grain': 'pregnancy',
  'date': 'PREGFIRSTCONDATE',
  'target': {'vocab': 'SNOMED', 'code': '424525001', 'name': 'Antenatal care'},
  'code_map': None,
  'note': 'first contact with maternity services; observation dated on the contact'},
 {'rule_id': 'MSDS-101-REFSRC',
  'table': 'msd101pregbook',
  'columns': ['SOURCEREFMAT'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '397663001', 'name': 'Referral source'},
  'answer_roots': [{'vocab': 'SNOMED', 'code': '397663001', 'name': 'Referral source'}],
  'code_map': {'01': {'text': 'General Medical Practitioner Practice'},
               '02': {'text': 'Self referral'},
               '03': {'text': 'Other Maternity Service'},
               '04': {'text': 'Early Pregnancy Unit (EPU)'},
               '05': {'text': 'Emergency Care Department (including Minor Injuries Units, Walk In Centres and Urgent '
                              'Treatment Centres)'},
               '06': {'text': 'School'},
               '07': {'text': 'Prison'},
               '08': {'text': 'Social Services'},
               '09': {'text': 'Health Visiting Service'},
               '98': {'text': 'Other (not listed)'}}},
 {'rule_id': 'MSDS-101-FCP',
  'table': 'msd101pregbook',
  'columns': ['PREGFIRSTCONTACTCAREPROFTYPE'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': ['PREGFIRSTCONDATE', 'ANTENATALAPPDATE'],
  'target': {'vocab': 'LOINC', 'code': '74571-1', 'name': 'Healthcare professional type [AHRQ]'},
  'code_map': {'01': {'text': 'NHS CONSULTANT Obstetrician'},
               '02': {'text': 'NHS Obstetrician - non- CONSULTANT grade'},
               '03': {'text': 'Private CONSULTANT Obstetrician'},
               '04': {'text': 'Private Obstetrician - non- CONSULTANT grade'},
               '05': {'text': 'NHS employed MIDWIFE'},
               '06': {'text': 'MIDWIFE employed by private company contracted to the NHS'},
               '07': {'text': 'Independent MIDWIFE'},
               '08': {'text': 'Midwifery support staff'},
               '09': {'text': 'Gynaecologist'},
               '10': {'text': 'GENERAL MEDICAL PRACTITIONER'},
               '11': {'text': 'NURSE'},
               '12': {'text': 'Health Visitor'},
               '98': {'text': 'Other CARE PROFESSIONAL (not listed)'}}},
 {'rule_id': 'MSDS-101-LATE',
  'table': 'msd101pregbook',
  'columns': ['REASONLATEBOOKING'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '424525001', 'name': 'Antenatal care'},
  'code_map': {'01': {'text': 'Mother unaware of pregnancy'},
               '02': {'text': 'Maternal choice'},
               '03': {'text': 'Concealed pregnancy'},
               '04': {'text': 'Transferred in from other maternity Health Care Provider'},
               '05': {'text': 'SERVICE capacity'},
               '06': {'text': 'Awaiting availability of interpreter'},
               '07': {'text': 'Did not attend one or more Antenatal Booking Appointments'},
               '08': {'text': 'Recently moved to area - no previous Antenatal Booking Appointment with other '
                              'maternity Health Care Provider'},
               '98': {'text': 'Other (not listed)'}},
  'note': "reason for late booking as value, qualifier_source_value 'late_booking_reason' (no standard late-booking "
          'question concept); empty at Barts'},
 {'rule_id': 'MSDS-101-DISAB',
  'table': 'msd101pregbook',
  'columns': ['DISABILITYINDMOTHER'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '21134002', 'name': 'Disability'},
  'code_map': None},
 {'rule_id': 'MSDS-101-LANG',
  'table': 'msd101pregbook',
  'columns': ['LANGCODE'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '370157003', 'name': 'Main spoken language'},
  'code_map': None,
  'note': 'ISO 639-1 code kept in value_source_value; value concept 0 unless a language concept resolves'},
 {'rule_id': 'MSDS-101-MH',
  'table': 'msd101pregbook',
  'columns': ['MHPREDICTIONDETECTIONINDMOTHER'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '18301000000102', 'name': 'Mental health assessment status'},
  'code_map': None,
  'note': 'Y = mental health prediction and detection questions asked at booking'},
 {'rule_id': 'MSDS-101-CSF',
  'table': 'msd101pregbook',
  'columns': ['COMPLEXSOCIALFACTORSIND'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '56098000', 'name': 'Social problem not due to a mental disorder'},
  'code_map': None,
  'note': 'complex social factors at booking (NICE CG110); Y/N'},
 {'rule_id': 'MSDS-101-EMPM',
  'table': 'msd101pregbook',
  'columns': ['EMPLOYMENTSTATUSMOTHER'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '224362002', 'name': 'Employment status'},
  'code_map': {'01': {'text': 'Employed', 'vocab': 'SNOMED', 'code': '224363007'},
               '02': {'text': 'Unemployed and actively seeking work', 'vocab': 'SNOMED', 'code': '73438004'},
               '03': {'text': 'Undertaking full (at least 16 hours per week) or part-time (less than 16 hours per '
                              'week) education or training as a student and not working or actively seeking work'},
               '04': {'text': 'Long-term sick or disabled, those receiving government sickness and disability '
                              'benefits'},
               '05': {'text': 'Looking after the family or home as a homemaker and not working or actively seeking '
                              'work'},
               '06': {'text': 'Not receiving government sickness and disability benefits and not working or actively '
                              'seeking work'},
               '07': {'text': 'Unpaid voluntary work and not working or actively seeking work'},
               '08': {'text': 'Retired'},
               'ZZ': {'text': 'Not Stated (PERSON asked but declined to provide a response)'},
               'UU': {'text': 'Not Known (Not Recorded)'}}},
 {'rule_id': 'MSDS-101-EMPP',
  'table': 'msd101pregbook',
  'columns': ['EMPLOYMENTSTATUSPARTNER'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '1287981006', 'name': 'Employment status of partner'},
  'code_map': {'01': {'text': 'Employed', 'vocab': 'SNOMED', 'code': '224363007'},
               '02': {'text': 'Unemployed and actively seeking work', 'vocab': 'SNOMED', 'code': '73438004'},
               '03': {'text': 'Undertaking full (at least 16 hours per week) or part-time (less than 16 hours per '
                              'week) education or training as a student and not working or actively seeking work'},
               '04': {'text': 'Long-term sick or disabled, those receiving government sickness and disability '
                              'benefits'},
               '05': {'text': 'Looking after the family or home as a homemaker and not working or actively seeking '
                              'work'},
               '06': {'text': 'Not receiving government sickness and disability benefits and not working or actively '
                              'seeking work'},
               '07': {'text': 'Unpaid voluntary work and not working or actively seeking work'},
               '08': {'text': 'Retired'},
               'ZZ': {'text': 'Not Stated (PERSON asked but declined to provide a response)'},
               'UU': {'text': 'Not Known (Not Recorded)'}}},
 {'rule_id': 'MSDS-101-SUPP',
  'table': 'msd101pregbook',
  'columns': ['SUPPORTSTATUSINDMOTHER'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '315042007', 'name': 'Social support'},
  'code_map': None,
  'note': 'Y/N/Z (Z = not stated -> value 0, raw kept)'},
 {'rule_id': 'MSDS-101-PREVCS',
  'table': 'msd101pregbook',
  'columns': ['PREVIOUSCAESAREANSECTIONS'],
  'kind': 'obs_number',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'LOINC', 'code': '68497-7', 'name': 'Previous cesarean deliveries #'},
  'code_map': None},
 {'rule_id': 'MSDS-101-SUBST',
  'table': 'msd101pregbook',
  'columns': ['SUBSTANCEUSESTATUS'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '1187057000', 'name': 'Substance use behavior'},
  'code_map': None,
  'note': 'empty at Barts; value kept raw'},
 {'rule_id': 'MSDS-101-SMOK',
  'table': 'msd101pregbook',
  'columns': ['SMOKINGSTATUS'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'LOINC', 'code': '72166-2', 'name': 'Tobacco smoking status'},
  'code_map': None,
  'note': 'empty at Barts (MSDS v2 carries smoking as SNOMED in msd109); value kept raw'},
 {'rule_id': 'MSDS-101-CIG',
  'table': 'msd101pregbook',
  'columns': ['CIGARETTESPERDAY'],
  'kind': 'obs_number',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'LOINC', 'code': '92275-7', 'name': 'Cigarettes smoked per day --during pregnancy'},
  'code_map': None,
  'unit': '{cigarettes}/d'},
 {'rule_id': 'MSDS-101-ALC',
  'table': 'msd101pregbook',
  'columns': ['ALCOHOLUNITSPERWEEK'],
  'kind': 'obs_number',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '1082641000000106', 'name': 'Alcohol units consumed per week'},
  'code_map': None,
  'unit': "[alcohol'U]/w"},
 {'rule_id': 'MSDS-101-WT',
  'table': 'msd101pregbook',
  'columns': ['PERSONWEIGHT'],
  'kind': 'meas_number',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '27113001', 'name': 'Body weight'},
  'code_map': None,
  'unit': 'kg',
  'note': 'with PERSONHEIGHT also fills pregnancy.pre_pregnancy_bmi'},
 {'rule_id': 'MSDS-101-HT',
  'table': 'msd101pregbook',
  'columns': ['PERSONHEIGHT'],
  'kind': 'meas_number',
  'grain': 'pregnancy',
  'date': 'ANTENATALAPPDATE',
  'target': {'vocab': 'SNOMED', 'code': '1153637007', 'name': 'Body height'},
  'code_map': None,
  'unit': 'cm',
  'scale': 100,
  'note': 'MSDS carries metres'},
 {'rule_id': 'MSDS-101-DISCH',
  'table': 'msd101pregbook',
  'columns': ['DISCHREASON'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'DISCHARGEDATEMATSERVICE',
  'target': {'vocab': 'SNOMED', 'code': '938851000000106', 'name': 'Discharged from maternity unit'},
  'code_map': {'01': {'text': 'Discharge following Delivery'},
               '02': {'text': 'Transfer to other Health Care Provider'},
               '03': {'text': 'Miscarriage'},
               '04': {'text': 'Termination of Pregnancy less than 24 weeks'},
               '05': {'text': 'Termination of Pregnancy greater than or equal to 24 weeks'},
               '06': {'text': 'No further contact from mother'},
               '07': {'text': 'Maternal death'}}},
 {'rule_id': 'MSDS-102-PLAN',
  'table': 'msd102matcareplan',
  'columns': ['CAREPLANTYPE'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED', 'code': '802321000000106', 'name': 'Care plan agreed'},
  'code_map': {'05': {'text': 'Antenatal Care Plan'},
               '06': {'text': 'Birth Care Plan'},
               '07': {'text': 'Postpartum Care Plan'}},
  'note': 'plan type (antenatal/birth/postpartum) as value'},
 {'rule_id': 'MSDS-102-PERS',
  'table': 'msd102matcareplan',
  'columns': ['MATPERSCAREPLANIND'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED',
             'code': '905181000000105',
             'name': 'Healthcare professional actions in care plan agreed'},
  'code_map': None},
 {'rule_id': 'MSDS-102-COC',
  'table': 'msd102matcareplan',
  'columns': ['CONTCAREPATHIND'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED', 'code': '423215004', 'name': 'Provision of continuity of care'},
  'code_map': None},
 {'rule_id': 'MSDS-102-SETTING',
  'table': 'msd102matcareplan',
  'columns': ['PLANNEDDELIVERYSETTING'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '05': {'text': 'Home (Private care)'},
               '06': {'text': 'Private hospital'},
               '96': {'text': 'Undecided'},
               '97': {'text': 'Not discussed'},
               '98': {'text': 'Other (not listed)'},
               '99': {'text': 'Not known'}},
  'aux': {'qualifier': 'CAREPLANTYPE'},
  'note': "planned (not actual) place: qualifier_source_value = 'planned|' + care plan type"},
 {'rule_id': 'MSDS-102-MIDW',
  'table': 'msd102matcareplan',
  'columns': ['PLACETYPEINTENDEDMIDWIFERY'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '05': {'text': 'Home (Private care)'},
               '06': {'text': 'Private hospital'},
               '96': {'text': 'Undecided'},
               '97': {'text': 'Not discussed'},
               '98': {'text': 'Other (not listed)'},
               '99': {'text': 'Not known'}},
  'note': "intended midwifery setting; qualifier_source_value 'intended_midwifery'"},
 {'rule_id': 'MSDS-102-LEAD',
  'table': 'msd102matcareplan',
  'columns': ['LEADPROFESSIONALTYPE'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'LOINC', 'code': '74571-1', 'name': 'Healthcare professional type [AHRQ]'},
  'code_map': None,
  'note': 'lead professional; empty at Barts'},
 {'rule_id': 'MSDS-102-CHG',
  'table': 'msd102matcareplan',
  'columns': ['REASONCHANGEDELSETTINGANT'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'CAREPLANDATE',
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'Change of ADDRESS'},
               '02': {'text': 'Maternal preference'},
               '03': {'text': 'Clinical reason maternal'},
               '04': {'text': 'Clinical reason fetal'},
               '05': {'text': 'Unit closure'},
               '06': {'text': 'SERVICE capacity'}},
  'note': 'reason planned setting changed; empty at Barts'},
 {'rule_id': 'MSDS-103-SCAN',
  'table': 'msd103datingscan',
  'columns': ['PROCEDUREDATEDATINGULTRASOUND'],
  'kind': 'procedure',
  'grain': 'pregnancy',
  'date': 'PROCEDUREDATEDATINGULTRASOUND',
  'target': {'vocab': 'SNOMED', 'code': '169229007', 'name': 'Dating/booking US scan'},
  'code_map': None},
 {'rule_id': 'MSDS-103-GA',
  'table': 'msd103datingscan',
  'columns': ['GESTATIONDATINGULTRASOUND'],
  'kind': 'meas_number',
  'grain': 'pregnancy',
  'date': 'PROCEDUREDATEDATINGULTRASOUND',
  'target': {'vocab': 'LOINC', 'code': '11884-4', 'name': 'Gestational age Estimated'},
  'code_map': None,
  'unit': 'd'},
 {'rule_id': 'MSDS-103-NFET',
  'table': 'msd103datingscan',
  'columns': ['NOFETUSESDATINGULTRASOUND'],
  'kind': 'meas_number',
  'grain': 'pregnancy',
  'date': 'PROCEDUREDATEDATINGULTRASOUND',
  'target': {'vocab': 'SNOMED', 'code': '246435002', 'name': 'Number of fetuses'},
  'code_map': None},
 {'rule_id': 'MSDS-103-ABN',
  'table': 'msd103datingscan',
  'columns': ['ABNORMALITYDATINGULTRASOUND'],
  'kind': 'obs_flag',
  'grain': 'pregnancy',
  'date': 'PROCEDUREDATEDATINGULTRASOUND',
  'target': {'vocab': 'SNOMED', 'code': '405996001', 'name': 'Presence of fetal anomaly in specimen'},
  'code_map': None,
  'note': 'Y/N fetal abnormality at the dating scan'},
 {'rule_id': 'MSDS-103-OFFER',
  'table': 'msd103datingscan',
  'columns': ['OFFERSTATUSDATINGULTRASOUND'],
  'kind': 'obs_coded',
  'grain': 'pregnancy',
  'date': 'ACTIVITYOFFERDATEULTRASOUND',
  'target': {'vocab': 'SNOMED', 'code': '1104661000000106', 'name': 'Antenatal ultrasound scan offered'},
  'code_map': {'01': {'text': 'Offered and Undecided'},
               '02': {'text': 'Offered and Declined'},
               '03': {'text': 'Offered and Accepted'},
               '04': {'text': 'Not Offered'}}},
 {'rule_id': 'MSDS-104-SCORE',
  'table': 'msd104codedscoreasspreg',
  'columns': ['TOOLTYPESNOMED', 'SCORE'],
  'kind': 'meas_snomed',
  'grain': 'pregnancy',
  'date': 'COMPDATE',
  'target': None,
  'code_map': None,
  'note': 'concept = SNOMED assessment tool, value_as_number = SCORE; empty at Barts'},
 {'rule_id': 'MSDS-105-PROVDX',
  'table': 'msd105provdiagnosispreg',
  'columns': ['PROVDIAG'],
  'kind': 'condition',
  'grain': 'pregnancy',
  'date': 'PROVDIAGDATE',
  'target': None,
  'code_map': None,
  'aux': {'scheme': 'DIAGSCHEME'},
  'condition_status': {'vocab': 'Condition Status', 'code': 'OMOP4976969', 'name': 'Preliminary diagnosis'},
  'note': 'resolved on the msd106 path (map_nomenclature / SNOMED Maps to)'},
 {'rule_id': 'MSDS-201-VISIT',
  'table': 'msd201carecontactpreg',
  'columns': ['ATTENDCODE',
              'MEDIUM',
              'CONSULTTYPE',
              'LOCCODE',
              'CONTACTDURATION',
              'ADMINCATCODE',
              'CCSUBJECT',
              'GPTHERAPYIND'],
  'kind': 'visit',
  'grain': 'carecontact',
  'date': 'CCONTACTDATETIME',
  'target': None,
  'code_map': {'01': {'text': 'Face to face communication'},
               '02': {'text': 'Telephone'},
               '03': {'text': 'Telemedicine'},
               '04': {'text': 'Talk type for a PERSON unable to speak'},
               '05': {'text': 'Email'},
               '06': {'text': 'Short Message Service (SMS) - Text Messaging'},
               '07': {'text': 'On-line Triage'},
               '08': {'text': 'Online Instant Messaging'},
               '98': {'text': 'Other (not listed)'}},
  'note': 'D4: only ATTENDCODE 5/6 (held) become visits; MEDIUM 01 -> 9202, 02/03 -> telehealth; visit_source_value '
          '= CONSULTTYPE|CCSUBJECT|LOCCODE|ADMINCATCODE; CONTACTDURATION sets visit_end'},
 {'rule_id': 'MSDS-201-DNA',
  'table': 'msd201carecontactpreg',
  'columns': ['ATTENDCODE'],
  'kind': 'obs_coded',
  'grain': 'carecontact',
  'date': 'CCONTACTDATETIME',
  'target': {'vocab': 'SNOMED', 'code': '410543007', 'name': 'Did not attend'},
  'code_map': {'5': {'text': 'Attended on time or, if late, before the relevant CARE PROFESSIONAL was ready to see '
                             'the PATIENT'},
               '6': {'text': 'Arrived late, after the relevant CARE PROFESSIONAL was ready to see the PATIENT, but '
                             'was seen'},
               '7': {'text': 'PATIENT arrived late and could not be seen'},
               '2': {'text': 'APPOINTMENT cancelled by, or on behalf of, the PATIENT'},
               '3': {'text': 'Did not attend - no advance warning given'},
               '4': {'text': 'APPOINTMENT cancelled or postponed by the Health Care Provider'},
               '0': {'text': 'Not applicable - APPOINTMENT occurs in the future'}},
  'only_codes': ['3'],
  'note': 'existing curated DNA observation; other non-held codes counted in _qa_results as contact_not_held'},
 {'rule_id': 'MSDS-301-ADM',
  'table': 'msd301labdel',
  'columns': ['STARTDATETIMEMOTHERDELIVERYHPS',
              'DISCHARGEDATETIMEMOTHERHSP',
              'ADMMETHCODEMOTHDELHSP',
              'DISCHMETHCODEMOTHPOSTDELHSP',
              'DISCHDESTCODEMOTHPOSTDELHSP'],
  'kind': 'visit',
  'grain': 'labourdelivery',
  'date': 'STARTDATETIMEMOTHERDELIVERYHPS',
  'target': None,
  'code_map': None,
  'note': 'visit 9201; deduped against MSDS-501-SPELL (a 301 stay inside a 501 spell uses the spell)'},
 {'rule_id': 'MSDS-301-ONSET',
  'table': 'msd301labdel',
  'columns': ['LABOURONSETMETHOD'],
  'kind': 'obs_coded',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'LOINC', 'code': '73774-2', 'name': 'Onset of labor [US Standard Certificate of Live Birth]'},
  'code_map': {'1': {'text': 'Spontaneous; the onset of regular contractions whether or not preceded by spontaneous '
                             'rupture of the membranes',
                     'vocab': 'SNOMED',
                     'code': '84457005'},
               '2': {'text': 'Any Caesarean Section carried out before the onset of Labour or a planned elective '
                             'Caesarean Section carried out immediately following the onset of Labour, when the '
                             'decision was made before Labour'},
               '3': {'text': 'Surgical induction; by amniotomy', 'vocab': 'SNOMED', 'code': '236958009'},
               '4': {'text': 'Medical induction; including administration of agents either orally, intravenously or '
                             'intravaginally with the intention of initiating Labour',
                     'vocab': 'SNOMED',
                     'code': '236958009'},
               '5': {'text': 'Combination of surgical induction and medical induction',
                     'vocab': 'SNOMED',
                     'code': '236958009'}},
  'note': 'codes 3/4/5 resolve to Induction of labor as value_as_concept (no separate procedure row)'},
 {'rule_id': 'MSDS-301-CS',
  'table': 'msd301labdel',
  'columns': ['CAESAREANDATETIME'],
  'kind': 'procedure',
  'grain': 'labourdelivery',
  'date': 'CAESAREANDATETIME',
  'target': {'vocab': 'SNOMED', 'code': '11466000', 'name': 'Cesarean section'},
  'code_map': None,
  'note': 'only when no msd302 caesarean procedure exists for the delivery'},
 {'rule_id': 'MSDS-301-DTD',
  'table': 'msd301labdel',
  'columns': ['DECISIONTODELIVERDATETIME'],
  'kind': 'obs_date',
  'grain': 'labourdelivery',
  'date': 'DECISIONTODELIVERDATETIME',
  'target': None,
  'code_map': None,
  'no_target_reason': 'no standard SNOMED/LOINC decision-to-deliver concept (searched 2026-09-30); '
                      'observation_concept 0, observation_source_value = rule_id, value_as_string = decision '
                      'datetime (decision-to-delivery interval)'},
 {'rule_id': 'MSDS-301-SETTING',
  'table': 'msd301labdel',
  'columns': ['SETTINGINTRACARE'],
  'kind': 'obs_coded',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '07': {'text': 'Maternity assessment or triage unit/ area'},
               '08': {'text': 'NHS WARD/health care setting without Delivery facilities'},
               '98': {'text': 'Other (not listed)'}},
  'note': "qualifier_source_value 'intrapartum_start'"},
 {'rule_id': 'MSDS-301-CHG',
  'table': 'msd301labdel',
  'columns': ['REASONCHANGEDELSETTINGLAB'],
  'kind': 'obs_coded',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'02': {'text': 'Maternal preference'},
               '03': {'text': 'Clinical reason maternal'},
               '04': {'text': 'Clinical reason fetal'},
               '05': {'text': 'Unit closure'},
               '06': {'text': 'SERVICE capacity'},
               '07': {'text': 'Unintentional change'}},
  'note': "reason setting changed in labour; qualifier_source_value 'change_reason_labour'"},
 {'rule_id': 'MSDS-301-ROM',
  'table': 'msd301labdel',
  'columns': ['ROMDATETIME'],
  'kind': 'obs_date',
  'grain': 'labourdelivery',
  'date': 'ROMDATETIME',
  'target': {'vocab': 'LOINC', 'code': '56829-5', 'name': 'Time membranes ruptured'},
  'code_map': None,
  'aux': {'method': 'ROMMETHOD', 'reason': 'ROMREASON'},
  'note': 'method/reason raw in qualifier_source_value as method=..|reason=..; empty at Barts'},
 {'rule_id': 'MSDS-301-STAGE2',
  'table': 'msd301labdel',
  'columns': ['LABOURONSETSECONDSTAGEDATETIME'],
  'kind': 'condition',
  'grain': 'labourdelivery',
  'date': 'LABOURONSETSECONDSTAGEDATETIME',
  'target': {'vocab': 'SNOMED', 'code': '42857002', 'name': 'Second stage of labor'},
  'code_map': None,
  'note': 'condition_start_datetime = onset of second stage; empty at Barts'},
 {'rule_id': 'MSDS-301-STAGE3',
  'table': 'msd301labdel',
  'columns': ['LABOURTHIRDSTAGEENDDATETIME'],
  'kind': 'condition',
  'grain': 'labourdelivery',
  'date': 'LABOURTHIRDSTAGEENDDATETIME',
  'target': {'vocab': 'SNOMED', 'code': '15898009', 'name': 'Third stage of labor'},
  'code_map': None,
  'note': 'condition dated on the end of the third stage; empty at Barts'},
 {'rule_id': 'MSDS-301-PLAC',
  'table': 'msd301labdel',
  'columns': ['PLACENTADELIVERYMETHOD'],
  'kind': 'procedure',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'SNOMED', 'code': '236994008', 'name': 'Placental delivery procedure'},
  'code_map': None,
  'note': 'method raw in procedure_source_value; empty at Barts'},
 {'rule_id': 'MSDS-301-EPIS',
  'table': 'msd301labdel',
  'columns': ['EPISIOTOMYREASON'],
  'kind': 'procedure',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'SNOMED', 'code': '85548006', 'name': 'Episiotomy'},
  'code_map': None,
  'note': 'reason raw in procedure_source_value; empty at Barts'},
 {'rule_id': 'MSDS-301-PRES',
  'table': 'msd301labdel',
  'columns': ['LABOURONSETPRESENTATION'],
  'kind': 'obs_coded',
  'grain': 'labourdelivery',
  'date': ['LABOURONSETDATETIME', 'STARTDATETIMEMOTHERDELIVERYHPS'],
  'date_fallback': 'baby_birth',
  'target': {'vocab': 'SNOMED', 'code': '271692001', 'name': 'Presentation of fetus'},
  'code_map': {'01': {'text': 'Cephalic'},
               '02': {'text': 'Breech', 'vocab': 'SNOMED', 'code': '6096002'},
               '03': {'text': 'Transverse/oblique', 'vocab': 'SNOMED', 'code': '73161006'},
               'XX': {'text': 'Other (not listed)'}},
  'note': 'empty at Barts'},
 {'rule_id': 'MSDS-302-CRIT',
  'table': 'msd302careactlabdel',
  'columns': ['MATCRITIND'],
  'kind': 'obs_flag',
  'grain': 'labourdelivery',
  'date': 'CLININTERDATETIMEMOTHER',
  'target': {'vocab': 'SNOMED', 'code': '405644001', 'name': 'Critical incident'},
  'code_map': None,
  'note': 'maternal critical incident; empty at Barts'},
 {'rule_id': 'MSDS-401-PRES',
  'table': 'msd401babydemo',
  'columns': ['FETUSPRESENTATION'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '271692001', 'name': 'Presentation of fetus'},
  'code_map': {'01': {'text': 'Cephalic'},
               '02': {'text': 'Breech', 'vocab': 'SNOMED', 'code': '6096002'},
               '03': {'text': 'Transverse/oblique', 'vocab': 'SNOMED', 'code': '73161006'},
               'XX': {'text': 'Other (not listed)'},
               '04': {'text': 'Other (retired MSDS v1 code)'}}},
 {'rule_id': 'MSDS-401-WATER',
  'table': 'msd401babydemo',
  'columns': ['WATERDELIVERYIND'],
  'kind': 'procedure',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '177185001', 'name': 'Water birth delivery'},
  'code_map': None,
  'only_codes': ['Y'],
  'person': 'mother',
  'note': 'emitted on the mother; empty at Barts'},
 {'rule_id': 'MSDS-401-PLACE',
  'table': 'msd401babydemo',
  'columns': ['SETTINGPLACEBIRTH'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'answer_roots': [{'vocab': 'SNOMED', 'code': '366344009', 'name': 'Place of birth - finding'}],
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '05': {'text': 'Home (Private care)'},
               '06': {'text': 'Private hospital'},
               '07': {'text': 'Maternity assessment or triage unit/ area'},
               '08': {'text': 'NHS WARD/health care setting without Delivery facilities'},
               '09': {'text': 'In transit (with NHS Ambulance Services)'},
               '10': {'text': 'In transit (with private Ambulance Services)'},
               '11': {'text': 'In transit (without healthcare services present)'},
               '12': {'text': 'Non-domestic and non-health care'},
               '98': {'text': 'Other (not listed)'},
               '99': {'text': 'Not known'}}},
 {'rule_id': 'MSDS-401-PLACEDEL',
  'table': 'msd401babydemo',
  'columns': ['PLACETYPEACTUALDELIVERY'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '05': {'text': 'Home (Private care)'},
               '06': {'text': 'Private hospital'},
               '07': {'text': 'Maternity assessment or triage unit/ area'},
               '08': {'text': 'NHS WARD/health care setting without Delivery facilities'},
               '09': {'text': 'In transit (with NHS Ambulance Services)'},
               '10': {'text': 'In transit (with private Ambulance Services)'},
               '11': {'text': 'In transit (without healthcare services present)'},
               '12': {'text': 'Non-domestic and non-health care'},
               '98': {'text': 'Other (not listed)'},
               '99': {'text': 'Not known'}},
  'note': 'empty at Barts'},
 {'rule_id': 'MSDS-401-PLACEMIDW',
  'table': 'msd401babydemo',
  'columns': ['PLACETYPEACTUALMIDWIFERY'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '169812000', 'name': 'Place of birth'},
  'code_map': {'01': {'text': 'NHS Obstetric Unit (including theatre)'},
               '02': {'text': 'NHS Alongside Midwifery Unit'},
               '03': {'text': 'NHS Freestanding Midwifery Unit (FMU)'},
               '04': {'text': 'Home (NHS care)'},
               '05': {'text': 'Home (Private care)'},
               '06': {'text': 'Private hospital'},
               '07': {'text': 'Maternity assessment or triage unit/ area'},
               '08': {'text': 'NHS WARD/health care setting without Delivery facilities'},
               '09': {'text': 'In transit (with NHS Ambulance Services)'},
               '10': {'text': 'In transit (with private Ambulance Services)'},
               '11': {'text': 'In transit (without healthcare services present)'},
               '12': {'text': 'Non-domestic and non-health care'},
               '98': {'text': 'Other (not listed)'},
               '99': {'text': 'Not known'}},
  'note': 'empty at Barts'},
 {'rule_id': 'MSDS-401-FEED',
  'table': 'msd401babydemo',
  'columns': ['BABYFIRSTFEEDINDCODE'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['BABYFIRSTFEEDDATETIME', 'PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '412727007', 'name': 'Infant feeding at birth'},
  'code_map': {'01': {'text': 'Yes - Maternal breast milk', 'vocab': 'SNOMED', 'code': '226789007'},
               '02': {'text': 'Yes - Donor breast milk'},
               '03': {'text': 'No - Not breast milk'}}},
 {'rule_id': 'MSDS-401-FEEDBM',
  'table': 'msd401babydemo',
  'columns': ['BABYFIRSTFEEDBREASTMILKSTATUS'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': ['BABYFIRSTFEEDDATETIME', 'PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '412727007', 'name': 'Infant feeding at birth'},
  'code_map': {'01': {'text': 'Yes - Maternal breast milk', 'vocab': 'SNOMED', 'code': '226789007'},
               '02': {'text': 'Yes - Donor breast milk'},
               '03': {'text': 'No - Not breast milk'}},
  'note': 'empty at Barts'},
 {'rule_id': 'MSDS-401-FEEDDIS',
  'table': 'msd401babydemo',
  'columns': ['BABYBREASTMILKSTATUSDISCHARGE'],
  'kind': 'obs_coded',
  'grain': 'baby',
  'date': 'DISCHARGEDATETIMEBABYHSP',
  'target': {'vocab': 'SNOMED', 'code': '169740003', 'name': 'Infant feeding method'},
  'code_map': {'01': {'text': 'Yes - Maternal breast milk', 'vocab': 'SNOMED', 'code': '226789007'},
               '02': {'text': 'Yes - Donor breast milk'},
               '03': {'text': 'No - Not breast milk'}},
  'note': 'feeding at discharge; empty at Barts'},
 {'rule_id': 'MSDS-401-S2S',
  'table': 'msd401babydemo',
  'columns': ['SKINTOSKINCONTACT1HOURIND'],
  'kind': 'obs_flag',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '386342003', 'name': 'Kangaroo care'},
  'code_map': None,
  'note': 'skin-to-skin within 1 hour, Y/N; empty at Barts'},
 {'rule_id': 'MSDS-401-VISIT',
  'table': 'msd401babydemo',
  'columns': ['DISCHARGEDATETIMEBABYHSP'],
  'kind': 'visit',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': None,
  'code_map': None,
  'note': 'birth admission 9201 for the baby: start = birth datetime, end = discharge'},
 {'rule_id': 'MSDS-401-DEATH',
  'table': 'msd401babydemo',
  'columns': ['PERSONDEATHDATETIMEBABY'],
  'kind': 'death',
  'grain': 'baby',
  'date': None,
  'target': None,
  'code_map': None},
 {'rule_id': 'MSDS-401-BW',
  'table': 'msd401babydemo',
  'columns': ['BIRTHWEIGHT'],
  'kind': 'meas_number',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'SNOMED', 'code': '364589006', 'name': 'Birth weight'},
  'code_map': None,
  'unit': 'g',
  'note': 'also infant_col (core); empty at Barts (SP-3)'},
 {'rule_id': 'MSDS-401-APGAR',
  'table': 'msd401babydemo',
  'columns': ['APGARSCORE5'],
  'kind': 'meas_number',
  'grain': 'baby',
  'date': ['PERSONBIRTHDATETIMEBABY', 'BABYBIRTHDATETIME'],
  'target': {'vocab': 'LOINC', 'code': '9274-2', 'name': '5 minute Apgar Score'},
  'code_map': None,
  'note': 'also infant_col (core); empty at Barts (SP-3)'},
 {'rule_id': 'MSDS-402-NNU',
  'table': 'msd402neoadmiss',
  'columns': ['NEONATALTRANSFERSTARTDATETIME', 'NEOCRITCAREIND'],
  'kind': 'visit_detail',
  'grain': 'baby',
  'date': 'NEONATALTRANSFERSTARTDATETIME',
  'target': None,
  'code_map': None,
  'note': 'under the birth visit; 32037 when NEOCRITCAREIND=Y else 9201'},
 {'rule_id': 'MSDS-403-PROVDX',
  'table': 'msd403provdiagneo',
  'columns': ['PROVDIAG'],
  'kind': 'condition',
  'grain': 'baby',
  'date': 'PROVDIAGDATE',
  'target': None,
  'code_map': None,
  'aux': {'scheme': 'DIAGSCHEME'},
  'condition_status': {'vocab': 'Condition Status', 'code': 'OMOP4976969', 'name': 'Preliminary diagnosis'}},
 {'rule_id': 'MSDS-405-CRIT',
  'table': 'msd405careactbaby',
  'columns': ['NNCRITINCIND'],
  'kind': 'obs_flag',
  'grain': 'baby',
  'date': 'CLININTERDATETIMEBABY',
  'target': {'vocab': 'SNOMED', 'code': '405644001', 'name': 'Critical incident'},
  'code_map': None,
  'note': 'neonatal critical incident; empty at Barts'},
 {'rule_id': 'MSDS-501-SPELL',
  'table': 'msd501hospprovspell',
  'columns': ['HOSPPROVSPELLNUM',
              'PREGNANCYID',
              'DISCHDATETIME',
              'SOURCEADMCODE',
              'PATIENTCLASSCODE',
              'ADMMETHCODE',
              'DISCHMETHCODE',
              'DISCHDESTCODE'],
  'kind': 'visit',
  'grain': 'spell',
  'date': 'STARTDATETIME',
  'target': None,
  'code_map': None,
  'note': 'visit 9201 per spell (mother via PREGNANCYID); admitted_from <- SOURCEADMCODE, discharged_to <- '
          'DISCHDESTCODE; ADMMETHCODE|PATIENTCLASSCODE|DISCHMETHCODE -> visit_source_value'},
 {'rule_id': 'MSDS-503-WARD',
  'table': 'msd503wardstay',
  'columns': ['HOSPPROVSPELLNUM', 'ENDDATETIMEWARDSTAY', 'WARDCODE'],
  'kind': 'visit_detail',
  'grain': 'spell',
  'date': 'STARTDATETIMEWARDSTAY',
  'target': None,
  'code_map': None,
  'note': 'visit_detail under the 501 visit; visit_detail_source_value = WARDCODE'}]


# Columns deliberately not carried, each with the reason.
MSDS_SKIP = {('msd001motherdemo', 'NHSNUMBERSTATUSMOTHER'): 'NHS number trace status: identifier metadata, not published',
 ('msd001motherdemo', 'ORGIDRESIDENCERESP'): 'responsible commissioner organisation/site code: administrative, no '
                                             'OMOP research meaning without an org dimension (D6)',
 ('msd002gp', 'ENDDATEGMPREG'): 'GP registration period: only meaningful with the skipped practice code',
 ('msd002gp', 'LPIDMOTHER'): 'key of a skipped table (msd002 GP registration)',
 ('msd002gp', 'ORGCODEGMPMOTHER'): 'GP practice organisation/site code: administrative, no OMOP research meaning '
                                   'without an org dimension (D6)',
 ('msd002gp', 'ORGIDGPPRAC'): 'GP practice responsibility organisation/site code: administrative, no OMOP research '
                              'meaning without an org dimension (D6)',
 ('msd002gp', 'STARTDATEGMPREG'): 'GP registration period: only meaningful with the skipped practice code',
 ('msd101pregbook', 'ORGIDCOMM'): 'commissioner organisation/site code: administrative, no OMOP research meaning '
                                  'without an org dimension (D6)',
 ('msd101pregbook', 'ORGIDPROVORIGIN'): 'provider of origin organisation/site code: administrative, no OMOP research '
                                        'meaning without an org dimension (D6)',
 ('msd101pregbook', 'ORGIDRECV'): 'receiving provider organisation/site code: administrative, no OMOP research '
                                  'meaning without an org dimension (D6)',
 ('msd101pregbook', 'ORGSITEIDBOOKING'): 'booking site organisation/site code: administrative, no OMOP research '
                                         'meaning without an org dimension (D6)',
 ('msd102matcareplan', 'CAREPROFLID'): 'care professional / team local identifier: staff identity, no provider table '
                                       'in PREBORN',
 ('msd102matcareplan', 'ORGSITEIDPLANNEDDELIVERY'): 'planned delivery site organisation/site code: administrative, '
                                                    'no OMOP research meaning without an org dimension (D6)',
 ('msd102matcareplan', 'TEAMLOCALID'): 'care professional / team local identifier: staff identity, no provider table '
                                       'in PREBORN',
 ('msd103datingscan', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                     'mother/pregnancy',
 ('msd103datingscan', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                       'mother/pregnancy',
 ('msd103datingscan', 'ORGIDDATINGULTRASOUND'): 'scan provider organisation/site code: administrative, no OMOP '
                                                'research meaning without an org dimension (D6)',
 ('msd105provdiagnosispreg', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to '
                                            'the mother/pregnancy',
 ('msd105provdiagnosispreg', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to '
                                              'the mother/pregnancy',
 ('msd106diagnosispreg', 'COMPLICATINGDIAGIND'): 'complicating-diagnosis modifier of core msd106 rows; empty at '
                                                 'every current site, revisit when populated',
 ('msd106diagnosispreg', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                        'mother/pregnancy',
 ('msd106diagnosispreg', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                          'mother/pregnancy',
 ('msd109findobsmother', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                        'mother/pregnancy',
 ('msd109findobsmother', 'FINDINGSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                           'resolves the code as SNOMED and QA flags other schemes',
 ('msd109findobsmother', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                          'mother/pregnancy',
 ('msd109findobsmother', 'OBSSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                       'resolves the code as SNOMED and QA flags other schemes',
 ('msd201carecontactpreg', 'CANCELDATE'): 'cancelled contact: not held, not published (D4); counted in _qa_results '
                                          'as contact_not_held',
 ('msd201carecontactpreg', 'CANCELREASON'): 'cancelled contact: not held, not published (D4); counted in _qa_results '
                                            'as contact_not_held',
 ('msd201carecontactpreg', 'ORGIDCOMM'): 'commissioner organisation/site code: administrative, no OMOP research '
                                         'meaning without an org dimension (D6)',
 ('msd201carecontactpreg', 'ORGSITEIDOFTREAT'): 'site of treatment organisation/site code: administrative, no OMOP '
                                                'research meaning without an org dimension (D6)',
 ('msd201carecontactpreg', 'REPLAPPTDATE'): 'replacement-appointment admin for a cancelled contact (D4): not '
                                            'published',
 ('msd201carecontactpreg', 'REPLAPPTOFFDATE'): 'replacement-appointment admin for a cancelled contact (D4): not '
                                               'published',
 ('msd202careactivitypreg', 'ACTIVITYDURATION'): 'care-activity duration: no OMOP field for activity duration; the '
                                                 'owning visit carries timing',
 ('msd202careactivitypreg', 'CAREPROFLID'): 'care professional / team local identifier: staff identity, no provider '
                                            'table in PREBORN',
 ('msd202careactivitypreg', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to '
                                           'the mother/pregnancy',
 ('msd202careactivitypreg', 'FINDINGSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                              'resolves the code as SNOMED and QA flags other schemes',
 ('msd202careactivitypreg', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to '
                                             'the mother/pregnancy',
 ('msd202careactivitypreg', 'OBSSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                          'resolves the code as SNOMED and QA flags other schemes',
 ('msd202careactivitypreg', 'PROCEDURESCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); '
                                                'core resolves the code as SNOMED and QA flags other schemes',
 ('msd202careactivitypreg', 'TEAMLOCALID'): 'care professional / team local identifier: staff identity, no provider '
                                            'table in PREBORN',
 ('msd301labdel', 'ORGIDPOSTNATALPATHLEADPROVIDER'): 'postnatal lead provider organisation/site code: '
                                                     'administrative, no OMOP research meaning without an org '
                                                     'dimension (D6)',
 ('msd301labdel', 'ORGSITEIDINTRA'): 'intrapartum site organisation/site code: administrative, no OMOP research '
                                     'meaning without an org dimension (D6)',
 ('msd302careactlabdel', 'ACTIVITYDURATION'): 'care-activity duration: no OMOP field for activity duration; the '
                                              'owning visit carries timing',
 ('msd302careactlabdel', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                        'mother/pregnancy',
 ('msd302careactlabdel', 'FINDINGSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                           'resolves the code as SNOMED and QA flags other schemes',
 ('msd302careactlabdel', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                          'mother/pregnancy',
 ('msd302careactlabdel', 'OBSSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                       'resolves the code as SNOMED and QA flags other schemes',
 ('msd302careactlabdel', 'PROCEDURESCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                             'resolves the code as SNOMED and QA flags other schemes',
 ('msd302careactlabdel', 'TEAMLOCALID'): 'care professional / team local identifier: staff identity, no provider '
                                         'table in PREBORN',
 ('msd401babydemo', 'CAREPROFLIDDEL'): 'care professional / team local identifier: staff identity, no provider table '
                                       'in PREBORN',
 ('msd401babydemo', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                     'mother/pregnancy',
 ('msd401babydemo', 'NHSNUMBERBABY'): 'direct identifier: not published in PREBORN; linkage uses ids, never NHS '
                                      'numbers',
 ('msd401babydemo', 'NHSNUMBERSTATUSBABY'): 'NHS number trace status: identifier metadata, not published',
 ('msd401babydemo', 'ORGSITEIDACTUALDELIVERY'): 'actual delivery site organisation/site code: administrative, no '
                                                'OMOP research meaning without an org dimension (D6)',
 ('msd402neoadmiss', 'ORGSITEADMITTINGNEONATAL'): 'admitting neonatal unit organisation/site code: administrative, '
                                                  'no OMOP research meaning without an org dimension (D6)',
 ('msd405careactbaby', 'ACTIVITYDURATION'): 'care-activity duration: no OMOP field for activity duration; the owning '
                                            'visit carries timing',
 ('msd405careactbaby', 'CAREPROFLID'): 'care professional / team local identifier: staff identity, no provider table '
                                       'in PREBORN',
 ('msd405careactbaby', 'FETALORDER'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                      'mother/pregnancy',
 ('msd405careactbaby', 'FINDINGSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                         'resolves the code as SNOMED and QA flags other schemes',
 ('msd405careactbaby', 'LOCALFETALID'): 'fetus-local identifier/ordinal: no OMOP fetus entity; facts attach to the '
                                        'mother/pregnancy',
 ('msd405careactbaby', 'OBSSCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core resolves '
                                     'the code as SNOMED and QA flags other schemes',
 ('msd405careactbaby', 'ORGIDBLOODSCREENINGLAB'): 'newborn blood-spot lab organisation/site code: administrative, no '
                                                  'OMOP research meaning without an org dimension (D6)',
 ('msd405careactbaby', 'PROCEDURESCHEME'): 'scheme discriminator: every current row is SNOMED CT (06/04/03); core '
                                           'resolves the code as SNOMED and QA flags other schemes',
 ('msd405careactbaby', 'TEAMLOCALID'): 'care professional / team local identifier: staff identity, no provider table '
                                       'in PREBORN',
 ('msd502hospspellcomm', 'ENDDATEORGCODECOMM'): 'spell commissioner assignment: organisation/site code: '
                                                'administrative, no OMOP research meaning without an org dimension '
                                                '(D6)',
 ('msd502hospspellcomm', 'HOSPPROVSPELLNUM'): 'spell commissioner assignment: organisation/site code: '
                                              'administrative, no OMOP research meaning without an org dimension '
                                              '(D6)',
 ('msd502hospspellcomm', 'ORGIDCOMM'): 'spell commissioner assignment: organisation/site code: administrative, no '
                                       'OMOP research meaning without an org dimension (D6)',
 ('msd502hospspellcomm', 'STARTDATEORGCODECOMM'): 'spell commissioner assignment: organisation/site code: '
                                                  'administrative, no OMOP research meaning without an org dimension '
                                                  '(D6)',
 ('msd503wardstay', 'ORGSITEIDOFTREAT'): 'ward site organisation/site code: administrative, no OMOP research meaning '
                                         'without an org dimension (D6)',
 ('msd504assigncareprof', 'CAREPROFLID'): 'care professional / team local identifier: staff identity, no provider '
                                          'table in PREBORN',
 ('msd504assigncareprof', 'ENDDATEASSCAREPROF'): 'consultant-assignment period: staff admin, skipped with msd504',
 ('msd504assigncareprof', 'HOSPPROVSPELLNUM'): 'key of a skipped table (msd504)',
 ('msd504assigncareprof', 'STARTDATEASSCAREPROF'): 'consultant-assignment period: staff admin, skipped with msd504',
 ('msd504assigncareprof', 'TEAMLOCALID'): 'care professional / team local identifier: staff identity, no provider '
                                          'table in PREBORN',
 ('msd504assigncareprof', 'TREATFUNCCODEMAT'): 'treatment function of the assigned consultant: low research value '
                                               'next to msd503 ward stays (plan: skip)'}

# Columns the core build (sections 6 to 8) reads directly: keys, dates and the coded clinical tables.
MSDS_CORE_USED = {
 ('msd001motherdemo', 'ETHNICCATEGORYMOTHER'),
 ('msd001motherdemo', 'LPIDMOTHER'),
 ('msd001motherdemo', 'NHSNUMBERMOTHER'),
 ('msd001motherdemo', 'ORGIDLPID'),
 ('msd001motherdemo', 'PERSONBIRTHDATEMOTHER'),
 ('msd003socperscircum', 'LPIDMOTHER'),
 ('msd004overseasvischarcat', 'LPIDMOTHER'),
 ('msd101pregbook', 'FOLICACIDSUPPLEMENT'),
 ('msd101pregbook', 'LASTMENSTRUALPERIODDATE'),
 ('msd101pregbook', 'LPIDMOTHER'),
 ('msd101pregbook', 'PREGNANCYID'),
 ('msd101pregbook', 'PREVIOUSLIVEBIRTHS'),
 ('msd101pregbook', 'PREVIOUSLOSSESLESSTHAN24WEEKS'),
 ('msd101pregbook', 'PREVIOUSSTILLBIRTHS'),
 ('msd102matcareplan', 'PREGNANCYID'),
 ('msd103datingscan', 'PREGNANCYID'),
 ('msd104codedscoreasspreg', 'PREGNANCYID'),
 ('msd105provdiagnosispreg', 'PREGNANCYID'),
 ('msd106diagnosispreg', 'DIAG'),
 ('msd106diagnosispreg', 'DIAGDATE'),
 ('msd106diagnosispreg', 'DIAGSCHEME'),
 ('msd106diagnosispreg', 'PREGNANCYID'),
 ('msd107medhistory', 'DIAGDATE'),
 ('msd107medhistory', 'DIAGSCHEME'),
 ('msd107medhistory', 'PREGNANCYID'),
 ('msd107medhistory', 'PREVDIAG'),
 ('msd109findobsmother', 'FINDINGCODE'),
 ('msd109findobsmother', 'FINDINGDATE'),
 ('msd109findobsmother', 'OBSCODE'),
 ('msd109findobsmother', 'OBSDATE'),
 ('msd109findobsmother', 'OBSVALUE'),
 ('msd109findobsmother', 'PREGNANCYID'),
 ('msd109findobsmother', 'UCUMUNIT'),
 ('msd201carecontactpreg', 'CARECONID'),
 ('msd201carecontactpreg', 'CCONTACTDATETIME'),
 ('msd201carecontactpreg', 'PREGNANCYID'),
 ('msd202careactivitypreg', 'CAREACTIDMOTHER'),
 ('msd202careactivitypreg', 'CARECONID'),
 ('msd202careactivitypreg', 'FINDINGCODE'),
 ('msd202careactivitypreg', 'OBSCODE'),
 ('msd202careactivitypreg', 'OBSVALUE'),
 ('msd202careactivitypreg', 'PROCEDURECODE'),
 ('msd202careactivitypreg', 'UCUMUNIT'),
 ('msd301labdel', 'LABOURDELIVERYID'),
 ('msd301labdel', 'PREGNANCYID'),
 ('msd302careactlabdel', 'CAREPROFLID'),
 ('msd302careactlabdel', 'CLININTERDATETIMEMOTHER'),
 ('msd302careactlabdel', 'FINDINGCODE'),
 ('msd302careactlabdel', 'LABOURDELIVERYID'),
 ('msd302careactlabdel', 'OBSCODE'),
 ('msd302careactlabdel', 'OBSVALUE'),
 ('msd302careactlabdel', 'PROCEDURECODE'),
 ('msd302careactlabdel', 'UCUMUNIT'),
 ('msd401babydemo', 'APGARSCORE5'),
 ('msd401babydemo', 'BABYBIRTHDATETIME'),
 ('msd401babydemo', 'BIRTHORDERMATERNITYSUS'),
 ('msd401babydemo', 'BIRTHWEIGHT'),
 ('msd401babydemo', 'DELIVERYMETHODCODE'),
 ('msd401babydemo', 'ETHNICCATEGORYBABY'),
 ('msd401babydemo', 'GESTATIONLENGTHBIRTH'),
 ('msd401babydemo', 'LABOURDELIVERYID'),
 ('msd401babydemo', 'LPIDBABY'),
 ('msd401babydemo', 'ORGIDLOCALPATIENTIDBABY'),
 ('msd401babydemo', 'PERSONBIRTHDATETIMEBABY'),
 ('msd401babydemo', 'PERSONPHENSEX'),
 ('msd401babydemo', 'PREGOUTCOME'),
 ('msd402neoadmiss', 'LPIDBABY'),
 ('msd403provdiagneo', 'LPIDBABY'),
 ('msd404diagneo', 'DIAG'),
 ('msd404diagneo', 'DIAGDATE'),
 ('msd404diagneo', 'DIAGSCHEME'),
 ('msd404diagneo', 'LPIDBABY'),
 ('msd405careactbaby', 'CAREACTIDBABY'),
 ('msd405careactbaby', 'CLININTERDATETIMEBABY'),
 ('msd405careactbaby', 'FINDINGCODE'),
 ('msd405careactbaby', 'LPIDBABY'),
 ('msd405careactbaby', 'OBSCODE'),
 ('msd405careactbaby', 'OBSVALUE'),
 ('msd405careactbaby', 'PROCEDURECODE'),
 ('msd405careactbaby', 'UCUMUNIT')}
# <<< SPEC

## 5. Concept lookup

OMOP only allows **standard** concepts in its main `*_concept_id` columns. For any code we follow the same
order, first hit wins:

1. a hand-checked **curated** choice (only where the vocabulary has no usable path);
2. the code's own concept, if it is standard and valid;
3. its **Maps to** link (the vocabulary's own statement of the standard equivalent);
4. for a retired code, its **replacement chain** ('Concept replaced by', then same_as, was_a, poss_eq,
   alt_to), landing on a standard concept;
5. for spec answers that opt in: the **embedding fallback** (closest meaning, accepted only above fixed
   thresholds);
6. otherwise concept 0, with the raw code kept in the `*_source_value` column.

Every choice is recorded with its method in `_msds_concept_map`, which is published.

### 5a. SNOMED lookup table

One row per SNOMED code with everything the later steps need: is it standard and valid, its Maps-to
targets, and (for retired codes) the standard concept its replacement chain lands on. Built once so each
later step joins a small table instead of the whole vocabulary. At Barts the replacement chain recovers 128
retired codes (about 2,000 rows) that have no Maps-to.

In [0]:
def build_snomed_lookup():
    """Create STG._resolved_snomed: per SNOMED code, its own concept, Maps-to list and replacement target."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._resolved_snomed AS
    WITH src AS (
      SELECT concept_code, concept_id AS src_concept_id, standard_concept, domain_id AS src_domain,
             (invalid_reason IS NULL) AS is_valid
      FROM {VOCAB}.concept WHERE vocabulary_id = 'SNOMED'
    ),
    -- every valid Maps-to target of each concept, as a sorted list
    maps AS (
      SELECT cr.concept_id_1 AS from_id,
             array_sort(collect_set(CAST(cr.concept_id_2 AS BIGINT))) AS maps_to
      FROM {VOCAB}.concept_relationship cr
      WHERE cr.relationship_id = 'Maps to' AND cr.invalid_reason IS NULL
      GROUP BY cr.concept_id_1
    ),
    -- one replacement per retired concept: strongest relationship first, then the lowest concept_id
    replacement AS (
      SELECT from_id, rep_id FROM (
        SELECT cr.concept_id_1 AS from_id, cr.concept_id_2 AS rep_id,
               row_number() OVER (PARTITION BY cr.concept_id_1 ORDER BY
                 CASE cr.relationship_id
                   WHEN 'Concept replaced by' THEN 0 WHEN 'Concept same_as to' THEN 1
                   WHEN 'Concept was_a to'    THEN 2 WHEN 'Concept poss_eq to' THEN 3
                   ELSE 4 END,
                 cr.concept_id_2) AS rk
        FROM {VOCAB}.concept_relationship cr
        WHERE cr.relationship_id IN ('Concept replaced by', 'Concept same_as to',
                                     'Concept was_a to', 'Concept poss_eq to', 'Concept alt_to to')
          AND cr.invalid_reason IS NULL
      ) WHERE rk = 1
    ),
    -- land the replacement on a standard concept: itself if standard and valid, else its lowest Maps-to
    replacement_standard AS (
      SELECT r.from_id,
             CASE WHEN c2.standard_concept = 'S' AND c2.invalid_reason IS NULL THEN r.rep_id
                  ELSE mt.min_maps END AS replaced_to,
             CASE WHEN c2.standard_concept = 'S' AND c2.invalid_reason IS NULL THEN c2.domain_id
                  ELSE cmt.domain_id END AS replaced_domain
      FROM replacement r
      JOIN {VOCAB}.concept c2 ON c2.concept_id = r.rep_id
      LEFT JOIN (
        SELECT concept_id_1, min(concept_id_2) AS min_maps
        FROM {VOCAB}.concept_relationship
        WHERE relationship_id = 'Maps to' AND invalid_reason IS NULL
        GROUP BY concept_id_1
      ) mt ON mt.concept_id_1 = r.rep_id
      LEFT JOIN {VOCAB}.concept cmt ON cmt.concept_id = mt.min_maps
    )
    SELECT s.concept_code, s.src_concept_id, s.standard_concept, s.src_domain, s.is_valid,
           m.maps_to, rs.replaced_to, rs.replaced_domain
    FROM src s
    LEFT JOIN maps m ON m.from_id = s.src_concept_id
    LEFT JOIN replacement_standard rs ON rs.from_id = s.src_concept_id
    """)
    recovered = scalar(f"""SELECT count(*) FROM {STG}._resolved_snomed
                           WHERE replaced_to IS NOT NULL AND size(coalesce(maps_to, array())) = 0""")
    print("SNOMED codes:", scalar(f"SELECT count(*) FROM {STG}._resolved_snomed"),
          "| recoverable only through a replacement chain:", recovered)


# The SQL expression that turns one row of _resolved_snomed (alias given) into the list of standard concept ids
# a SNOMED code stands for: its own concept if standard and valid, else its Maps-to list (more than one target
# means the code means several things at once, and one fact row is written per target), else its replacement,
# else [0]. Used by every step that resolves a SNOMED code column.
def snomed_concepts_sql(alias):
    return (f"CASE WHEN {alias}.standard_concept = 'S' AND {alias}.is_valid "
            f"THEN array(CAST({alias}.src_concept_id AS BIGINT)) "
            f"WHEN size({alias}.maps_to) >= 1 THEN array_sort(array_distinct({alias}.maps_to)) "
            f"WHEN {alias}.replaced_to IS NOT NULL THEN array(CAST({alias}.replaced_to AS BIGINT)) "
            f"ELSE array(CAST(0 AS BIGINT)) END")


build_snomed_lookup()

### 5b. Curated SNOMED codes

Retired SNOMED codes that have no Maps-to and no replacement chain, and "complaint of" codes whose
vocabulary path lands in the wrong domain. Each is mapped by hand to a current standard concept whose name
and domain were checked against the live vocabulary (2026-07-07). A curated choice also decides the OMOP
table, whatever MSDS column the code arrived in: a headache is a Condition, not a measurement.

In [0]:
# SNOMED code -> (OMOP table, standard concept_id, what it means)
CURATED_SNOMED = {
    # obstetric: retired "tear during delivery" codes -> current perineal laceration concepts
    "199916005": ("condition_occurrence", 197343, "First degree perineal tear during delivery -> First degree perineal laceration"),
    "199925004": ("condition_occurrence", 198492, "Second degree perineal tear during delivery -> Second degree perineal laceration"),
    "199930000": ("condition_occurrence", 193830, "Third degree perineal tear during delivery -> Third degree perineal laceration"),
    "199607009": ("observation", 4079844, "Intrauterine death - delivered -> Fetal death"),
    "199049003": ("condition_occurrence", 4273560, "Threatened premature labor - not delivered -> Premature labor"),
    "199783004": ("condition_occurrence", 72377, "Shoulder dystocia - delivered -> Shoulder girdle dystocia"),
    # symptoms and complaints ("C/O ...") -> the clinical concept itself
    "272027003": ("condition_occurrence", 378253, "C/O a headache -> Headache"),
    "272044004": ("condition_occurrence", 441408, "C/O vomiting -> Vomiting"),
    "275570002": ("condition_occurrence", 4129155, "Complaining of per vaginam bleeding -> Vaginal bleeding"),
    "162415008": ("condition_occurrence", 140214, "C/O a rash -> Eruption"),
    "268911002": ("condition_occurrence", 140214, "O/E rash present -> Eruption"),
    # administrative and safeguarding
    "281399006": ("observation", 4259518, "Did not attend -> Did not attend"),
    "1077911000000105": ("observation", 605010, "Safeguarding -> Safeguarding concern"),
    "887041000000104": ("observation", 605010, "Safeguarding issues -> Safeguarding concern"),
}


def curated_snomed_values_sql():
    """CURATED_SNOMED as an inline SQL table (code, target_omop_table, concept_id)."""
    rows = []
    for code, (table, concept_id, _) in CURATED_SNOMED.items():
        rows.append(f"('{code}', '{table}', {concept_id})")
    return f"(SELECT * FROM VALUES {', '.join(rows)} AS t(code, target_omop_table, concept_id))"

### 5c. Spec questions, answers and units

Every concept the spec asks for becomes one request row: the rule's question (`target`), a diagnosis
rule's `status`, each code in its `code_map` (`value`), and its `unit`. Requests are resolved through the
vocabulary in the order above. A question or status must resolve to a standard concept in the domain its
kind needs, or the build stops with the list of failures: questions never rest on a guess. Answers and
units may stay unmapped (concept 0).

In [0]:
# The domain each kind's question concept must have.
SPEC_KIND_DOMAIN = {"obs_coded": "Observation", "obs_flag": "Observation", "obs_number": "Observation",
                    "obs_date": "Observation", "meas_number": "Measurement", "procedure": "Procedure",
                    "condition": "Condition"}

# Question concepts chosen by hand. 410543007 'Did not attend' is not standard; this is the same concept the
# curated SNOMED rule for 281399006 uses.
SPEC_CURATED_TARGETS = {"410543007": 4259518}

def spec_requests():
    """One (rule_id, role, src_code, src_text, vocab, code, kind) row per concept the spec asks for."""
    rows = []
    for rule in MSDS_COLUMN_SPEC:
        for role, wanted in (("target", rule["target"]), ("status", rule.get("condition_status"))):
            if wanted:
                rows.append((rule["rule_id"], role, None, wanted.get("name"), wanted["vocab"], wanted["code"],
                             rule["kind"]))
        for code, answer in (rule["code_map"] or {}).items():
            rows.append((rule["rule_id"], "value", code, answer["text"], answer.get("vocab"), answer.get("code"),
                         rule["kind"]))
        if rule.get("unit"):
            # The UCUM code exactly as written; concept 0 is allowed (UCUM has no cigarettes/day).
            rows.append((rule["rule_id"], "unit", None, rule["unit"], "UCUM", rule["unit"], rule["kind"]))
    return rows


def resolve_spec():
    """Create STG._spec_resolved: one row per spec request with its concept and how it was chosen."""
    columns = "rule_id STRING, role STRING, src_code STRING, src_text STRING, vocab STRING, code STRING, kind STRING"
    spark.createDataFrame(spec_requests(), columns).write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{STG}._spec_request")

    curated_cases = []
    for code, concept_id in SPEC_CURATED_TARGETS.items():
        curated_cases.append(f"WHEN q.vocab = 'SNOMED' AND q.code = '{code}' THEN {concept_id}")
    curated = " ".join(curated_cases)

    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._spec_resolved AS
    WITH c AS (
      SELECT concept_id, concept_code, vocabulary_id, domain_id, standard_concept, invalid_reason
      FROM {VOCAB}.concept),
    first_maps_to AS (
      SELECT concept_id_1, min(concept_id_2) AS m FROM {VOCAB}.concept_relationship
      WHERE relationship_id = 'Maps to' AND invalid_reason IS NULL GROUP BY concept_id_1),
    -- every candidate for each request, one column per step of the lookup order
    hit AS (
      SELECT q.*,
             CASE {curated} END AS cur_id,
             CASE WHEN c.standard_concept = 'S' AND c.invalid_reason IS NULL THEN c.concept_id END AS std_id,
             mt.m AS maps_id,
             rs.replaced_to AS rep_id
      FROM {STG}._spec_request q
      LEFT JOIN c ON c.vocabulary_id = q.vocab AND c.concept_code = q.code
      LEFT JOIN first_maps_to mt ON mt.concept_id_1 = c.concept_id
      LEFT JOIN {STG}._resolved_snomed rs ON q.vocab = 'SNOMED' AND rs.concept_code = q.code
                                         AND size(coalesce(rs.maps_to, array())) = 0)
    SELECT h.rule_id, h.role, h.src_code, h.src_text, h.kind,
           CAST(coalesce(h.cur_id, h.std_id, h.maps_id, h.rep_id, 0) AS BIGINT) AS concept_id,
           t.domain_id,
           CASE WHEN h.cur_id IS NOT NULL THEN 'preborn_curated'
                WHEN coalesce(h.std_id, h.maps_id, h.rep_id) IS NOT NULL THEN 'vocab_maps_to'
                ELSE 'unmapped' END AS method,
           CAST(NULL AS STRING) AS model_version, CAST(NULL AS DOUBLE) AS cosine, CAST(NULL AS DOUBLE) AS margin,
           CASE WHEN h.code IS NULL THEN NULL
                ELSE concat(h.vocab, ':', h.code, ' via ',
                            CASE WHEN h.cur_id IS NOT NULL THEN 'curated'
                                 WHEN h.std_id IS NOT NULL THEN 'standard'
                                 WHEN h.maps_id IS NOT NULL THEN 'maps_to'
                                 WHEN h.rep_id IS NOT NULL THEN 'replacement_chain'
                                 ELSE 'none' END) END AS rule_ref
    FROM hit h
    LEFT JOIN c t ON t.concept_id = coalesce(h.cur_id, h.std_id, h.maps_id, h.rep_id)
    """)

    # Questions and statuses must land on a standard concept in the right domain.
    domain_by_kind = ", ".join(f"'{kind}', '{domain}'" for kind, domain in SPEC_KIND_DOMAIN.items())
    bad = spark.sql(f"""
      SELECT rule_id, role, kind, rule_ref, concept_id, domain_id FROM {STG}._spec_resolved
      WHERE role IN ('target', 'status')
        AND (concept_id = 0 OR domain_id IS DISTINCT FROM
             CASE WHEN role = 'status' THEN 'Condition Status'
                  ELSE element_at(map({domain_by_kind}), kind) END)
    """).collect()
    if bad:
        problems = [f"{b.rule_id}/{b.role} {b.rule_ref} -> {b.concept_id} {b.domain_id}" for b in bad]
        raise AssertionError("MSDS spec questions not resolving to a standard concept in the right domain: "
                             + "; ".join(problems))
    show(f"SELECT role, method, count(*) n FROM {STG}._spec_resolved GROUP BY ALL ORDER BY ALL")


resolve_spec()

### 5d. Embedding fallback for answers with no code

Many NHS Data Dictionary answers are text only ("Self referral"). A rule can opt in by naming
`answer_roots`: the candidate answers are then the standard descendants of those concepts, and each
still-unmapped answer text is compared with every candidate name by meaning (text-embedding-3-large
vectors, cosine similarity computed in the warehouse). The best candidate is accepted only when

- its cosine is at least **0.62**, and
- it beats the runner-up by at least **0.05**

(the same thresholds as the pathology LOINC answer mapping). Otherwise the answer stays concept 0 with method
`embed_below_threshold`, or `embed_missing` when a vector is not cached. Nothing is reviewed by hand; the
method, model, cosine and margin are published so users can filter.

Answers below the thresholds go on to section 5e.

The vectors are cached in `3_lookup.embeddings.terms`. The build only reads that cache, unless the
`update_mapping_caches` widget is `yes`: then it first embeds the missing terms through Azure OpenAI and adds
them to the cache. That needs write access to `3_lookup` (a person, not the service principal). Only
needed when the spec gains new answers or `answer_roots`.

In [0]:
EMBED_TERMS = "3_lookup.embeddings.terms"
EMBED_MODEL = "text-embedding-3-large"
EMBED_FLOOR = 0.62    # minimum cosine to accept (= FLOOR_LOINC in map_pathology)
EMBED_MARGIN = 0.05   # minimum lead over the runner-up (= MARGIN_LOINC in map_pathology)


def build_embedding_candidates():
    """Create STG._spec_pool: for each rule with answer_roots, the standard descendants of those roots."""
    roots = []
    for rule in MSDS_COLUMN_SPEC:
        for root in rule.get("answer_roots") or []:
            roots.append((rule["rule_id"], root["vocab"], root["code"]))
    spark.createDataFrame(roots, "rule_id STRING, vocab STRING, code STRING").createOrReplaceTempView("_answer_roots")
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._spec_pool AS
    SELECT DISTINCT q.rule_id, d.concept_id, d.concept_name, d.domain_id
    FROM _answer_roots q
    JOIN {VOCAB}.concept r ON r.vocabulary_id = q.vocab AND r.concept_code = q.code
    JOIN {VOCAB}.concept_ancestor a ON a.ancestor_concept_id = r.concept_id AND a.min_levels_of_separation > 0
    JOIN {VOCAB}.concept d ON d.concept_id = a.descendant_concept_id
    WHERE d.standard_concept = 'S' AND d.invalid_reason IS NULL
    """)
    rules_with_pool = {row.rule_id for row in spark.table(f"{STG}._spec_pool").select("rule_id").distinct().collect()}
    empty = sorted({rule_id for rule_id, _, _ in roots} - rules_with_pool)
    assert not empty, f"answer_roots with no standard descendants: {empty}"


def embedding_terms_needed():
    """Every text the fallback compares: unmapped answers of opted-in rules, and their candidates' names."""
    return spark.sql(f"""
      SELECT src_text AS term FROM {STG}._spec_resolved v
      WHERE v.role = 'value' AND v.method <> 'preborn_curated' AND v.concept_id = 0
        AND v.rule_id IN (SELECT rule_id FROM {STG}._spec_pool)
      UNION SELECT concept_name FROM {STG}._spec_pool""")


def embedding_terms_missing():
    """The needed terms that have no cached vector for EMBED_MODEL."""
    cached = spark.table(EMBED_TERMS).where(f"model_version = '{EMBED_MODEL}'").select("term")
    return [row.term for row in embedding_terms_needed().join(cached, "term", "left_anti").collect()]


def embed_missing_terms():
    """Embed the missing terms with Azure OpenAI and add them to EMBED_TERMS (needs write access to 3_lookup)."""
    import time
    import requests

    endpoint = "https://benja-m6p4lv77-eastus2.cognitiveservices.azure.com"
    url = f"{endpoint}/openai/deployments/{EMBED_MODEL}/embeddings?api-version=2023-05-15"
    headers = {"api-key": dbutils.secrets.get(scope="adc_store", key="barts_global_key"),
               "Content-Type": "application/json"}

    def embed_batch(texts):
        """Vectors for a list of texts, in the same order; retries with back-off on errors."""
        delay = 1.0
        for attempt in range(6):
            response = requests.post(url, headers=headers, json={"input": texts, "encoding_format": "float"},
                                     timeout=120)
            if response.ok:
                data = sorted(response.json()["data"], key=lambda item: item["index"])
                return [item["embedding"] for item in data]
            if attempt == 5:
                response.raise_for_status()
            time.sleep(delay)
            delay = min(delay * 2, 30.0)

    todo = embedding_terms_missing()
    print(len(todo), "terms to embed")
    batch_size = 256
    for start in range(0, len(todo), batch_size):
        batch = todo[start:start + batch_size]
        vectors = embed_batch(batch)
        rows = []
        for term, vector in zip(batch, vectors):
            rows.append((term, [float(x) for x in vector], EMBED_MODEL))
        spark.createDataFrame(rows, "term STRING, embedding_vector ARRAY<FLOAT>, model_version STRING") \
            .createOrReplaceTempView("_new_terms")
        # Insert only what is still missing, so a rerun after a partial failure does no double work.
        spark.sql(f"""
          MERGE INTO {EMBED_TERMS} t USING _new_terms s
          ON t.term = s.term AND t.model_version = s.model_version
          WHEN NOT MATCHED THEN INSERT (term, embedding_vector, model_version, created_at, embedded_at, ADC_UPDT, updated_at)
          VALUES (s.term, s.embedding_vector, s.model_version, current_timestamp(), current_timestamp(),
                  current_timestamp(), current_timestamp())""")
        print(f"embedded {start + len(batch)}/{len(todo)}")
    assert not embedding_terms_missing(), f"terms still missing from {EMBED_TERMS}"


def resolve_answers_by_embedding():
    """Score each unmapped opted-in answer against its candidates and update STG._spec_resolved."""
    embedding_terms_needed().createOrReplaceTempView("_embed_need")
    missing = scalar(f"""SELECT count(*) FROM _embed_need n LEFT ANTI JOIN {EMBED_TERMS} t
                         ON t.term = n.term AND t.model_version = '{EMBED_MODEL}'""")
    # Every candidate of every opted-in unmapped answer, ranked by cosine (section 5e reads the top ten).
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._spec_embed_ranked AS
    WITH vec AS (
      SELECT term, embedding_vector v FROM {EMBED_TERMS}
      WHERE model_version = '{EMBED_MODEL}' AND term IN (SELECT term FROM _embed_need)),
    -- the unmapped answers of opted-in rules, with their vectors (NULL when not cached)
    src AS (
      SELECT r.rule_id, r.src_code, r.src_text, vec.v
      FROM {STG}._spec_resolved r LEFT JOIN vec ON vec.term = r.src_text
      WHERE r.role = 'value' AND r.concept_id = 0 AND r.rule_id IN (SELECT rule_id FROM {STG}._spec_pool)),
    cand AS (
      SELECT /*+ BROADCAST(p) */ p.rule_id, p.concept_id, p.concept_name, p.domain_id, vec.v
      FROM {STG}._spec_pool p JOIN vec ON vec.term = p.concept_name),
    -- cosine similarity = dot product / (length * length)
    scored AS (
      SELECT s.rule_id, s.src_code, c.concept_id, c.concept_name, c.domain_id,
             aggregate(zip_with(s.v, c.v, (a, b) -> a * b), 0D, (acc, x) -> acc + x)
             / (sqrt(aggregate(s.v, 0D, (acc, x) -> acc + x * x)) * sqrt(aggregate(c.v, 0D, (acc, x) -> acc + x * x))) AS cosine
      FROM src s JOIN cand c ON c.rule_id = s.rule_id WHERE s.v IS NOT NULL),
    ranked AS (
      SELECT *, row_number() OVER (PARTITION BY rule_id, src_code ORDER BY cosine DESC, concept_id) rk,
             lead(cosine) OVER (PARTITION BY rule_id, src_code ORDER BY cosine DESC, concept_id) next_cos
      FROM scored)
    SELECT * FROM ranked
    """)
    # The decision for each answer, from its best candidate.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._spec_embed AS
    WITH vec AS (
      SELECT term, embedding_vector v FROM {EMBED_TERMS}
      WHERE model_version = '{EMBED_MODEL}' AND term IN (SELECT term FROM _embed_need)),
    src AS (
      SELECT r.rule_id, r.src_code, r.src_text, vec.v
      FROM {STG}._spec_resolved r LEFT JOIN vec ON vec.term = r.src_text
      WHERE r.role = 'value' AND r.concept_id = 0 AND r.rule_id IN (SELECT rule_id FROM {STG}._spec_pool))
    SELECT s.rule_id, s.src_code, k.concept_id AS top_concept_id, k.concept_name AS top_name, k.domain_id AS top_domain,
           k.cosine, k.cosine - coalesce(k.next_cos, 0D) AS margin,
           CASE WHEN s.v IS NULL OR k.concept_id IS NULL THEN 'embed_missing'
                WHEN k.cosine >= {EMBED_FLOOR} AND k.cosine - coalesce(k.next_cos, 0D) >= {EMBED_MARGIN} THEN 'embed'
                ELSE 'embed_below_threshold' END AS method
    FROM src s LEFT JOIN {STG}._spec_embed_ranked k ON k.rule_id = s.rule_id AND k.src_code = s.src_code AND k.rk = 1
    """)
    spark.sql(f"""
    MERGE INTO {STG}._spec_resolved r
    USING {STG}._spec_embed e ON r.role = 'value' AND r.rule_id = e.rule_id AND r.src_code = e.src_code
    WHEN MATCHED THEN UPDATE SET
      concept_id = CASE WHEN e.method = 'embed' THEN e.top_concept_id ELSE 0 END,
      domain_id = CASE WHEN e.method = 'embed' THEN e.top_domain END,
      method = e.method,
      model_version = CASE WHEN e.method <> 'embed_missing' THEN '{EMBED_MODEL}' END,
      cosine = e.cosine,
      margin = e.margin,
      rule_ref = CASE WHEN e.top_name IS NOT NULL
                      THEN concat('top1 ', e.top_name, ' (floor {EMBED_FLOOR}, margin {EMBED_MARGIN})') END
    """)
    if missing:
        print(f"WARNING: {missing} terms have no cached vector, so their answers are embed_missing. "
              f"Rerun with update_mapping_caches = yes to embed them.")
    show(f"""SELECT rule_id, src_code, src_text, method, round(cosine, 3) cos, round(margin, 3) margin, rule_ref
             FROM {STG}._spec_resolved WHERE method LIKE 'embed%' ORDER BY 1, 2""")


build_embedding_candidates()
if UPDATE_MAPPING_CACHES:
    embed_missing_terms()
resolve_answers_by_embedding()

### 5e. Model adjudication for answers the embedding could not settle

An answer left `embed_below_threshold` is given to Claude Opus 5.5 with the question, the NHS Data Dictionary
answer text, the rule's note and its ten closest candidate concepts (with their cosines). The model picks the
candidate that means the same as the answer, or none.

- a pick is published with method `llm_adjudicated`, model `claude-opus-5-5`, the candidate's cosine and the
  model's one-sentence reason in `rule_ref`;
- "none" stays concept 0 with method `llm_no_match` and the reason.

Nothing is reviewed by hand: users can filter on the method like any other.

**Runs are repeatable.** Decisions are cached in `adjudication_table` (default
`3_lookup.embeddings.llm_adjudications`), keyed on the answer, its candidate list, the model and the prompt
version. The build only reads the cache, so a rerun never changes a mapping by asking again. With
`update_mapping_caches` = `yes`, answers with no cached decision are sent to the model first (this needs
write access to the table: a person, not the service principal). A new candidate list, model or prompt
version means a new decision.

In [0]:
ADJUDICATION_MODEL = "claude-opus-5-5"
ADJUDICATION_PROMPT_VERSION = "msds-answer-v1"
ADJUDICATION_CANDIDATES = 10
FOUNDRY_ANTHROPIC = "https://benja-m6p4lv77-eastus2.openai.azure.com/anthropic"

ADJUDICATION_COLUMNS = """rule_id STRING, src_code STRING, src_text STRING, candidates_key STRING,
  model_version STRING, prompt_version STRING, concept_id BIGINT, reason STRING, adjudicated_at TIMESTAMP"""


def answers_to_adjudicate():
    """The below-threshold answers, each as a dict with its question, note and top candidates."""
    rows = spark.sql(f"""
      SELECT r.rule_id, r.src_code, r.src_text, q.src_text AS question,
             array_sort(collect_list(named_struct('rk', k.rk, 'concept_id', k.concept_id,
                                                  'name', k.concept_name, 'cosine', k.cosine))) AS candidates
      FROM {STG}._spec_resolved r
      JOIN {STG}._spec_embed_ranked k ON k.rule_id = r.rule_id AND k.src_code = r.src_code
                                     AND k.rk <= {ADJUDICATION_CANDIDATES}
      LEFT JOIN {STG}._spec_resolved q ON q.rule_id = r.rule_id AND q.role = 'target'
      WHERE r.role = 'value' AND r.method = 'embed_below_threshold'
      GROUP BY r.rule_id, r.src_code, r.src_text, q.src_text""").collect()
    notes = {rule["rule_id"]: rule.get("note", "") for rule in MSDS_COLUMN_SPEC}
    all_answers = {rule["rule_id"]: rule["code_map"] or {} for rule in MSDS_COLUMN_SPEC}
    answers = []
    for row in rows:
        candidates = [(c.concept_id, c.name, c.cosine) for c in row.candidates]
        answers.append({
            "rule_id": row.rule_id, "src_code": row.src_code, "src_text": row.src_text,
            "question": row.question, "note": notes.get(row.rule_id, ""), "candidates": candidates,
            "other_answers": [f"{code}: {value['text']}" for code, value in all_answers[row.rule_id].items()
                              if code != row.src_code],
            # the decision is tied to exactly this candidate list
            "candidates_key": ",".join(str(concept_id) for concept_id, _, _ in sorted(candidates)),
        })
    return answers


def adjudication_prompt(answer):
    """The question put to the model for one answer."""
    candidate_lines = "\n".join(f"{concept_id}: {name} (cosine {cosine:.3f})"
                                 for concept_id, name, cosine in answer["candidates"])
    return f"""We are mapping a coded answer from the NHS Maternity Services Data Set to an OMOP standard concept.

Question (the MSDS field): {answer['question']} (rule {answer['rule_id']})
Rule note: {answer['note'] or 'none'}
NHS Data Dictionary answer code {answer['src_code']}: "{answer['src_text']}"
The field's other answers, for context: {'; '.join(answer['other_answers'])}

Candidate OMOP concepts, from the answer's concept hierarchy, with their text-embedding similarity:
{candidate_lines}

Choose the candidate that means the same thing as the answer. If none of them means the same thing, choose none:
a clearly broader, narrower or different concept is not a match. Details no candidate could express, such as who
funded or provided the care, do not stop a match: the raw answer code is kept alongside the concept.
Reply with JSON only: {{"concept_id": <one candidate id, or null>, "reason": "<one short sentence>"}}"""


def ask_model(prompt):
    """Send one prompt to ADJUDICATION_MODEL and return (concept_id or None, reason). Retries on errors."""
    import time
    import requests
    headers = {"x-api-key": dbutils.secrets.get(scope="adc_store", key="barts_global_key"),
               "anthropic-version": "2023-06-01", "content-type": "application/json"}
    body = {"model": ADJUDICATION_MODEL, "max_tokens": 4000, "messages": [{"role": "user", "content": prompt}]}
    delay = 2.0
    for attempt in range(6):
        response = requests.post(f"{FOUNDRY_ANTHROPIC}/v1/messages", headers=headers, json=body, timeout=180)
        if response.ok:
            text = "".join(part.get("text", "") for part in response.json()["content"])
            try:
                reply = json.loads(text[text.index("{"):text.rindex("}") + 1])
                return reply.get("concept_id"), str(reply.get("reason", ""))[:500]
            except ValueError:
                if attempt == 5:
                    return None, f"model reply was not the requested JSON: {text[:300]}"
        elif attempt == 5:
            response.raise_for_status()
        time.sleep(delay)
        delay = min(delay * 2, 60.0)


def adjudicate_new_answers(answers):
    """Ask the model about every answer with no cached decision, and add the decisions to ADJUDICATION_TABLE."""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {ADJUDICATION_TABLE} ({ADJUDICATION_COLUMNS}) USING DELTA")
    cached = {(r.rule_id, r.src_code, r.src_text, r.candidates_key) for r in spark.sql(f"""
        SELECT rule_id, src_code, src_text, candidates_key FROM {ADJUDICATION_TABLE}
        WHERE model_version = '{ADJUDICATION_MODEL}' AND prompt_version = '{ADJUDICATION_PROMPT_VERSION}'""").collect()}
    new_rows = []
    for answer in answers:
        key = (answer["rule_id"], answer["src_code"], answer["src_text"], answer["candidates_key"])
        if key in cached:
            continue
        concept_id, reason = ask_model(adjudication_prompt(answer))
        candidate_ids = {c[0] for c in answer["candidates"]}
        if concept_id is not None and concept_id not in candidate_ids:
            reason = f"model picked {concept_id}, which is not a candidate: {reason}"[:500]
            concept_id = None
        new_rows.append(key + (ADJUDICATION_MODEL, ADJUDICATION_PROMPT_VERSION, concept_id, reason,
                               datetime.now(timezone.utc).replace(tzinfo=None)))
        print(answer["rule_id"], answer["src_code"], answer["src_text"], "->", concept_id, "|", reason)
    if new_rows:
        spark.createDataFrame(new_rows, ADJUDICATION_COLUMNS.replace("\n", " ")).createOrReplaceTempView("_new_decisions")
        spark.sql(f"""
          MERGE INTO {ADJUDICATION_TABLE} t USING _new_decisions s
          ON t.rule_id = s.rule_id AND t.src_code = s.src_code AND t.src_text = s.src_text
             AND t.candidates_key = s.candidates_key AND t.model_version = s.model_version
             AND t.prompt_version = s.prompt_version
          WHEN NOT MATCHED THEN INSERT *""")
    print(len(new_rows), "answers adjudicated,", len(answers) - len(new_rows), "already cached")


def apply_cached_adjudications(answers):
    """Update STG._spec_resolved from the cached decisions for exactly these answers and candidate lists."""
    if not answers:
        return
    if not spark.catalog.tableExists(ADJUDICATION_TABLE):
        print(f"WARNING: {ADJUDICATION_TABLE} does not exist, so {len(answers)} below-threshold answers stay "
              f"unadjudicated. Rerun with update_mapping_caches = yes to adjudicate them.")
        return
    keys = [(a["rule_id"], a["src_code"], a["src_text"], a["candidates_key"]) for a in answers]
    spark.createDataFrame(keys, "rule_id STRING, src_code STRING, src_text STRING, candidates_key STRING") \
        .createOrReplaceTempView("_wanted_decisions")
    spark.sql(f"""
    MERGE INTO {STG}._spec_resolved r
    USING (
      SELECT d.rule_id, d.src_code, d.concept_id, d.reason, k.concept_name, k.domain_id, k.cosine
      FROM {ADJUDICATION_TABLE} d
      JOIN _wanted_decisions w ON w.rule_id = d.rule_id AND w.src_code = d.src_code AND w.src_text = d.src_text
                              AND w.candidates_key = d.candidates_key
      LEFT JOIN {STG}._spec_embed_ranked k ON k.rule_id = d.rule_id AND k.src_code = d.src_code
                                          AND k.concept_id = d.concept_id
      WHERE d.model_version = '{ADJUDICATION_MODEL}' AND d.prompt_version = '{ADJUDICATION_PROMPT_VERSION}'
    ) a ON r.role = 'value' AND r.rule_id = a.rule_id AND r.src_code = a.src_code
    WHEN MATCHED THEN UPDATE SET
      concept_id = coalesce(a.concept_id, 0),
      domain_id = a.domain_id,
      method = CASE WHEN a.concept_id IS NOT NULL THEN 'llm_adjudicated' ELSE 'llm_no_match' END,
      model_version = '{ADJUDICATION_MODEL}',
      cosine = CASE WHEN a.concept_id IS NOT NULL THEN a.cosine ELSE r.cosine END,
      margin = CASE WHEN a.concept_id IS NOT NULL THEN NULL ELSE r.margin END,
      rule_ref = concat('{ADJUDICATION_PROMPT_VERSION}: ', coalesce(a.concept_name, 'none'), ' -- ', a.reason)
    """)
    pending = scalar(f"SELECT count(*) FROM {STG}._spec_resolved WHERE method = 'embed_below_threshold'")
    if pending:
        print(f"WARNING: {pending} below-threshold answers have no cached decision. "
              f"Rerun with update_mapping_caches = yes to adjudicate them.")
    show(f"""SELECT rule_id, src_code, src_text, method, concept_id, rule_ref
             FROM {STG}._spec_resolved WHERE method LIKE 'llm%' ORDER BY 1, 2""")


answers = answers_to_adjudicate()
if UPDATE_MAPPING_CACHES and answers:
    adjudicate_new_answers(answers)
apply_cached_adjudications(answers)

## 6. People, pregnancies and babies

- **Mothers** come from `msd001` (one person per local patient id).
- **Pregnancies** come from the booking table `msd101`, with delivery details rolled up from the labour
  (`msd301`) and baby (`msd401`) tables.
- **Babies** come from `msd401`; each becomes both a `person` (role infant) and an `infant` row with the
  same id.

Values are staged raw here; implausible values are removed in section 9.

### 6a. Ethnicity

MSDS records the NHS ethnic category letter (A to Z). OMOP's standard race concepts are US-based, so the fit is
partial. Letters with an honest equivalent are mapped; the Mixed categories (D to G), S (any other) and
Z (not stated) become 0. The letter itself is always kept in `race_source_value`. Used for mothers and
babies. Another site may supply its own expression in `RACE_CASE_OVERRIDE` (section 1).

In [0]:
RACE_CASE = """CASE upper(trim({col}))
  WHEN 'A' THEN 8527 WHEN 'B' THEN 8527 WHEN 'C' THEN 8527
  WHEN 'D' THEN 0 WHEN 'E' THEN 0 WHEN 'F' THEN 0 WHEN 'G' THEN 0
  WHEN 'H' THEN 8515 WHEN 'L' THEN 8515
  WHEN 'J' THEN 38003589 WHEN 'K' THEN 38003575 WHEN 'M' THEN 38003598
  WHEN 'N' THEN 38003600 WHEN 'P' THEN 38003598 WHEN 'R' THEN 38003579
  ELSE 0 END"""

if RACE_CASE_OVERRIDE is not None:
    RACE_CASE = RACE_CASE_OVERRIDE

### 6b. Mothers

In [0]:
def stage_mothers():
    """Mint one person per mother in msd001 and add them to STG.person (role 'mother')."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._stg_mother AS
    SELECT
      {mint_id('person', 'ORGIDLPID', 'LPIDMOTHER')} AS person_id,
      ORGIDLPID, LPIDMOTHER,
      concat_ws('|', '{SITE}', coalesce(nullif(trim(ORGIDLPID), ''), '~'), LPIDMOTHER) AS person_source_value,
      try_cast(PERSONBIRTHDATEMOTHER AS TIMESTAMP) AS birth_datetime,
      upper(trim(ETHNICCATEGORYMOTHER)) AS race_source_value,
      {RACE_CASE.format(col='ETHNICCATEGORYMOTHER')} AS race_concept_id
    FROM {SRC['msd001motherdemo']}
    WHERE LPIDMOTHER IS NOT NULL
    """)
    mothers = assert_unique(f"{STG}._stg_mother", "person_id")

    spark.sql(f"""
    INSERT INTO {STG}.person (person_id, person_source_value, person_role, gender_source_value, birth_datetime,
                              race_concept_id, race_source_value, site_id)
    SELECT person_id, person_source_value, 'mother', CAST(NULL AS STRING), birth_datetime,
           race_concept_id, race_source_value, '{SITE}'
    FROM {STG}._stg_mother
    """)
    print("mothers:", mothers)


stage_mothers()

### 6c. Pregnancies

One pregnancy per booking `PREGNANCYID`. The sentinel id `'0'` means "no pregnancy" and is skipped; a
repeated pregnancy keeps its most recently updated row. Delivery details come from the babies of the
pregnancy (linked through the labour record); the first-born baby decides outcome and delivery method.
Code meanings are from the NHS Data Dictionary.

| Field | Rule |
|---|---|
| start | last menstrual period date |
| end | latest baby birth date (`PERSONBIRTHDATETIMEBABY`; `BABYBIRTHDATETIME` is empty at Barts); NULL if before the start |
| outcome | 01 live birth, 02 antepartum stillbirth, 03 intrapartum stillbirth, 04 stillbirth |
| delivery method | 0/1 spontaneous vaginal, 2 low forceps, 3 forceps, 4 ventouse, 5/6 breech, 7 elective caesarean, 8 emergency caesarean |
| singleton | Yes when exactly one baby |
| parity | nulliparous when no previous live or still births, else multiparous |
| gravidity | previous live births + stillbirths + losses before 24 weeks |
| folic acid | 01/02 Yes, 03 No |

In [0]:
YES, NO = 4188539, 4188540                   # SNOMED Yes / No
NULLIPAROUS, MULTIPAROUS = 4012561, 4102166

PREGNANCY_OUTCOME_CONCEPT = """CASE trim(dr.primary_outcome)
  WHEN '01' THEN 45883764   -- Live birth
  WHEN '02' THEN 37017027   -- Antepartum stillbirth
  WHEN '03' THEN 45773593   -- Intrapartum stillbirth
  WHEN '04' THEN 443213     -- Stillbirth
  ELSE 0 END"""
PREGNANCY_OUTCOME_TEXT = """CASE trim(dr.primary_outcome)
  WHEN '01' THEN 'Live birth' WHEN '02' THEN 'Stillbirth - antepartum'
  WHEN '03' THEN 'Stillbirth - intrapartum' WHEN '04' THEN 'Stillbirth'
  WHEN '05' THEN 'Other' ELSE dr.primary_outcome END"""
DELIVERY_METHOD_CONCEPT = """CASE trim(dr.primary_mode)
  WHEN '0' THEN 45885338 WHEN '1' THEN 45885338   -- Spontaneous vaginal
  WHEN '2' THEN 4156948                            -- Low forceps
  WHEN '3' THEN 4217586                            -- Forceps
  WHEN '4' THEN 440790                             -- Ventouse
  WHEN '5' THEN 4213387 WHEN '6' THEN 4213387      -- Breech
  WHEN '7' THEN 4075182                            -- Elective caesarean
  WHEN '8' THEN 4167089                            -- Emergency caesarean
  ELSE 0 END"""
DELIVERY_METHOD_TEXT = """CASE trim(dr.primary_mode)
  WHEN '0' THEN 'Spontaneous vaginal' WHEN '1' THEN 'Vaginal' WHEN '2' THEN 'Low forceps'
  WHEN '3' THEN 'Forceps' WHEN '4' THEN 'Ventouse' WHEN '5' THEN 'Breech' WHEN '6' THEN 'Breech'
  WHEN '7' THEN 'Elective caesarean' WHEN '8' THEN 'Emergency caesarean' ELSE dr.primary_mode END"""


def stage_pregnancies():
    """Build STG.pregnancy: one row per booked pregnancy, with delivery details from its babies."""
    previous_live = "try_cast(p.PREVIOUSLIVEBIRTHS AS INT)"
    previous_still = "try_cast(p.PREVIOUSSTILLBIRTHS AS INT)"
    previous_losses = "try_cast(p.PREVIOUSLOSSESLESSTHAN24WEEKS AS INT)"

    spark.sql(f"""
    INSERT INTO {STG}.pregnancy (person_id, pregnancy_id, pregnancy_start_date, pregnancy_end_date,
      gestational_length_in_day, pregnancy_outcome_concept_id, pregnancy_mode_delivery_concept_id,
      pregnancy_single_concept_id, pregnancy_marital_status_concept_id, pregnancy_number_fetuses,
      pregnancy_number_liveborn, prev_pregnancy_parity_concept_id, prev_pregnancy_gravidity,
      prev_livebirth_number, prev_still_misc_number, pre_pregnancy_bmi, pregnancy_folic_concept_id,
      pregnancy_outcome_source_value, pregnancy_mode_delivery_source_value, site_id)
    WITH
    -- one booking row per pregnancy: the latest update wins, rows with no update date come last
    booking AS (
      SELECT * FROM (
        SELECT p.*, row_number() OVER (
          PARTITION BY PREGNANCYID
          ORDER BY CASE WHEN ADC_UPDT IS NULL THEN 1 ELSE 0 END, ADC_UPDT DESC) AS rn
        FROM {SRC['msd101pregbook']} p
        WHERE PREGNANCYID IS NOT NULL AND trim(PREGNANCYID) <> '0'
      ) WHERE rn = 1
    ),
    -- one row per baby, linked to its pregnancy through the labour record
    babies AS (
      SELECT l.PREGNANCYID,
             b.LPIDBABY, b.BIRTHORDERMATERNITYSUS, b.LABOURDELIVERYID,
             try_cast(b.PERSONBIRTHDATETIMEBABY AS DATE) AS birth_date,
             try_cast(b.GESTATIONLENGTHBIRTH AS INT) AS gest_days,
             cast(b.PREGOUTCOME AS STRING) AS pregoutcome,
             cast(b.DELIVERYMETHODCODE AS STRING) AS deliverymethod,
             row_number() OVER (PARTITION BY l.PREGNANCYID
               ORDER BY CASE WHEN b.BIRTHORDERMATERNITYSUS IS NULL THEN 1 ELSE 0 END,
                        try_cast(b.BIRTHORDERMATERNITYSUS AS INT), b.LPIDBABY) AS birth_rank
      FROM {SRC['msd401babydemo']} b
      JOIN {SRC['msd301labdel']} l ON l.LABOURDELIVERYID = b.LABOURDELIVERYID
      WHERE l.PREGNANCYID IS NOT NULL
    ),
    -- roll the babies up to one row per pregnancy; the first-born decides outcome and delivery method
    dr AS (
      SELECT PREGNANCYID,
             count(*) AS n_babies,
             count(CASE WHEN trim(pregoutcome) = '01' THEN 1 END) AS n_liveborn,
             max(birth_date) AS end_date,
             max(gest_days) AS gest_days,
             max(CASE WHEN birth_rank = 1 THEN pregoutcome END) AS primary_outcome,
             max(CASE WHEN birth_rank = 1 THEN deliverymethod END) AS primary_mode
      FROM babies GROUP BY PREGNANCYID
    )
    SELECT
      m.person_id,
      {mint_id('pregnancy', 'p.PREGNANCYID')} AS pregnancy_id,
      try_cast(p.LASTMENSTRUALPERIODDATE AS DATE) AS pregnancy_start_date,
      CASE WHEN dr.end_date < try_cast(p.LASTMENSTRUALPERIODDATE AS DATE) THEN CAST(NULL AS DATE)
           ELSE dr.end_date END AS pregnancy_end_date,
      dr.gest_days AS gestational_length_in_day,
      {PREGNANCY_OUTCOME_CONCEPT} AS pregnancy_outcome_concept_id,
      {DELIVERY_METHOD_CONCEPT} AS pregnancy_mode_delivery_concept_id,
      CASE WHEN dr.n_babies IS NULL THEN CAST(NULL AS BIGINT)
           WHEN dr.n_babies = 1 THEN {YES}
           ELSE {NO} END AS pregnancy_single_concept_id,
      CAST(NULL AS BIGINT) AS pregnancy_marital_status_concept_id,    -- not in MSDS
      dr.n_babies AS pregnancy_number_fetuses,
      dr.n_liveborn AS pregnancy_number_liveborn,
      CASE WHEN {previous_live} IS NULL AND {previous_still} IS NULL THEN CAST(NULL AS BIGINT)
           WHEN coalesce({previous_live}, 0) + coalesce({previous_still}, 0) = 0 THEN {NULLIPAROUS}
           ELSE {MULTIPAROUS} END AS prev_pregnancy_parity_concept_id,
      CASE WHEN {previous_live} IS NULL AND {previous_still} IS NULL AND {previous_losses} IS NULL
           THEN CAST(NULL AS INT)
           ELSE coalesce({previous_live}, 0) + coalesce({previous_still}, 0) + coalesce({previous_losses}, 0)
           END AS prev_pregnancy_gravidity,
      {previous_live} AS prev_livebirth_number,
      {previous_still} AS prev_still_misc_number,
      CAST(NULL AS DOUBLE) AS pre_pregnancy_bmi,                       -- not derived from MSDS
      CASE WHEN trim(p.FOLICACIDSUPPLEMENT) IN ('01', '02') THEN {YES}
           WHEN trim(p.FOLICACIDSUPPLEMENT) = '03' THEN {NO}
           ELSE 0 END AS pregnancy_folic_concept_id,
      {PREGNANCY_OUTCOME_TEXT} AS pregnancy_outcome_source_value,
      {DELIVERY_METHOD_TEXT} AS pregnancy_mode_delivery_source_value,
      '{SITE}' AS site_id
    FROM booking p
    JOIN {STG}._stg_mother m ON m.LPIDMOTHER = p.LPIDMOTHER
    LEFT JOIN dr ON dr.PREGNANCYID = p.PREGNANCYID
    """)
    pregnancies = assert_unique(f"{STG}.pregnancy", "pregnancy_id")
    print("pregnancies:", pregnancies)
    spark.sql(f"DROP TABLE IF EXISTS {STG}._stg_mother")


stage_pregnancies()

### 6d. Babies

A baby is identified by its local patient id (`ORGIDLOCALPATIENTIDBABY`, `LPIDBABY`). When `LPIDBABY` is
blank the fallback key is the labour record plus birth order. The same three-part key mints the infant id
everywhere it is needed (here, the fact routing in section 7, and births, neonatal stays and deaths in
section 8), so it is defined once in `BABY_KEY_COLUMNS`.

`birth_con_malformation_concept_id` is set when any neonatal diagnosis (`msd404`) sits under SNOMED
'Congenital disease'. Apgar is not in the MSDS baby table, so it stays empty.

In [0]:
# The infant identity key: (_idkind, _k1, _k2). Selected alongside msd401 columns.
BABY_KEY_COLUMNS = """
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN 'lpid' ELSE 'fallback' END AS _idkind,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN ORGIDLOCALPATIENTIDBABY ELSE LABOURDELIVERYID END AS _k1,
      CASE WHEN nullif(trim(LPIDBABY),'') IS NOT NULL THEN LPIDBABY ELSE cast(BIRTHORDERMATERNITYSUS as string) END AS _k2
    """

INFANT_ID = mint_id("infant", "_idkind", "_k1", "_k2")       # used where msd401 columns are unqualified
INFANT_ID_B = mint_id("infant", "b._idkind", "b._k1", "b._k2")   # used where msd401 is aliased b

CONGENITAL_DISEASE = 440508      # SNOMED 'Congenital disease' (ancestor of every congenital diagnosis)
CONGENITAL_MALFORMATION = 4079975


def stage_infants():
    """Build STG.infant and add each baby to STG.person (role 'infant'); infant_id = person_id."""
    # Babies with any neonatal diagnosis under 'Congenital disease'.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._cong_babies AS
    WITH diagnosis_concepts AS (
      SELECT d.LPIDBABY, {snomed_concepts_sql('r')} AS cids
      FROM {SRC['msd404diagneo']} d
      LEFT JOIN {STG}._resolved_snomed r ON r.concept_code = d.DIAG
      WHERE d.LPIDBABY IS NOT NULL AND d.DIAG IS NOT NULL
    ),
    diagnoses AS (
      SELECT DISTINCT LPIDBABY, explode(cids) AS cid FROM diagnosis_concepts
    )
    SELECT DISTINCT diagnoses.LPIDBABY
    FROM diagnoses JOIN {VOCAB}.concept_ancestor ca
      ON ca.descendant_concept_id = diagnoses.cid AND ca.ancestor_concept_id = {CONGENITAL_DISEASE}
    """)
    print("babies with a congenital diagnosis:", spark.table(f"{STG}._cong_babies").count())

    # One infant row per identity key. pregnancy_id stays NULL unless the pregnancy was staged in 6c.
    spark.sql(f"""
    INSERT INTO {STG}.infant (pregnancy_id, infant_id, birth_outcome_concept_id, birth_weight,
                              birth_con_malformation_concept_id, birth_apgar, labour_delivery_id, site_id)
    WITH ident AS (
      SELECT b.LABOURDELIVERYID, b.LPIDBABY, b.PREGOUTCOME,
             try_cast(b.BIRTHWEIGHT AS DOUBLE) AS BIRTHWEIGHT, {BABY_KEY_COLUMNS}
      FROM {SRC['msd401babydemo']} b
    ),
    one_per_baby AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (PARTITION BY _idkind, _k1, _k2 ORDER BY LABOURDELIVERYID) AS rn
        FROM ident i
      ) WHERE rn = 1
    )
    SELECT
      pr.pregnancy_id,
      {INFANT_ID_B} AS infant_id,
      CASE trim(b.PREGOUTCOME)
        WHEN '01' THEN 45883764 WHEN '02' THEN 37017027 WHEN '03' THEN 45773593
        WHEN '04' THEN 443213 ELSE 0 END AS birth_outcome_concept_id,
      b.BIRTHWEIGHT AS birth_weight,
      CASE WHEN cg.LPIDBABY IS NOT NULL THEN {CONGENITAL_MALFORMATION} ELSE 0 END AS birth_con_malformation_concept_id,
      CAST(NULL AS INT) AS birth_apgar,
      b.LABOURDELIVERYID AS labour_delivery_id,
      '{SITE}' AS site_id
    FROM one_per_baby b
    LEFT JOIN {SRC['msd301labdel']} d301 ON d301.LABOURDELIVERYID = b.LABOURDELIVERYID
    LEFT JOIN {STG}.pregnancy pr ON pr.pregnancy_id = {mint_id('pregnancy', 'd301.PREGNANCYID')}
    LEFT JOIN {STG}._cong_babies cg ON cg.LPIDBABY = b.LPIDBABY
    """)
    infants = assert_unique(f"{STG}.infant", "infant_id")

    # The same babies as people: sex, birth time and ethnicity from msd401.
    spark.sql(f"""
    INSERT INTO {STG}.person (person_id, person_source_value, person_role, gender_source_value, birth_datetime,
                              race_concept_id, race_source_value, site_id)
    WITH ident AS (
      SELECT {BABY_KEY_COLUMNS},
        try_cast(PERSONBIRTHDATETIMEBABY AS TIMESTAMP) AS birth_datetime,
        cast(PERSONPHENSEX AS STRING) AS gender_source_value,
        upper(trim(ETHNICCATEGORYBABY)) AS race_source_value,
        {RACE_CASE.format(col='ETHNICCATEGORYBABY')} AS race_concept_id
      FROM {SRC['msd401babydemo']}
    ),
    one_per_baby AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (PARTITION BY _idkind, _k1, _k2 ORDER BY _idkind) AS rn FROM ident i
      ) WHERE rn = 1
    )
    SELECT
      {INFANT_ID} AS person_id,
      concat_ws('|', '{SITE}', _idkind, coalesce(nullif(trim(_k1), ''), '~'), _k2) AS person_source_value,
      'infant' AS person_role,
      gender_source_value, birth_datetime,
      race_concept_id, race_source_value, '{SITE}' AS site_id
    FROM one_per_baby
    """)
    print("infants:", infants, "| linked to a pregnancy:",
          scalar(f"SELECT count(pregnancy_id) FROM {STG}.infant"))
    spark.sql(f"DROP TABLE IF EXISTS {STG}._cong_babies")


stage_infants()

## 7. Clinical facts

Every clinical fact, whatever its source, is first written as one row of the working table
`STG._all_facts`. A row says which OMOP table it belongs in, its concept and value, its date, and the
source keys that decide whose fact it is (mother, pregnancy, baby, care contact). Section 10 turns these
rows into the OMOP fact tables. Keeping one shape for every fact means routing, cleaning and the
published-or-dropped reconciliation are each written once.

Each row has a `fact_natural_key` built from its source key; the published fact id is minted from it, so
ids are stable between runs.

In [0]:
FACT_COLUMNS = [
    ("source_table", "STRING"),           # the MSDS table it came from
    ("pregnancy_natural_key", "STRING"),  # MSDS PREGNANCYID
    ("baby_natural_key", "STRING"),       # MSDS LPIDBABY, for baby facts
    ("careconid", "STRING"),              # MSDS CARECONID, links the fact to its care contact's visit
    ("fact_natural_key", "STRING"),       # the fact's own source key; the fact id is minted from it
    ("target_omop_table", "STRING"),
    ("concept_id", "BIGINT"),
    ("multi_map_flag", "INT"),            # 1 when one source code maps to several concepts
    ("source_value", "STRING"),
    ("source_concept_id", "BIGINT"),
    ("event_date", "DATE"),
    ("value_as_number", "DOUBLE"),
    ("unit_source_value", "STRING"),
    ("mother_natural_key", "STRING"),     # MSDS LPIDMOTHER, for mother facts with no pregnancy
    ("value_as_concept_id", "BIGINT"),
    ("value_as_datetime", "TIMESTAMP"),
    ("value_as_string", "STRING"),
    ("value_source_value", "STRING"),
    ("unit_concept_id", "BIGINT"),
    ("qualifier_concept_id", "BIGINT"),
    ("qualifier_source_value", "STRING"),
    ("status_concept_id", "BIGINT"),
    ("rule_id", "STRING"),                # the spec rule, or MSDS-CORE-<table> for the core coded tables
    ("person_role", "STRING"),            # 'mother' or 'infant': whose fact it is
]
FACT_COLUMN_NAMES = ", ".join(name for name, _ in FACT_COLUMNS)

spark.sql(f"CREATE TABLE {STG}._all_facts ({', '.join(f'{name} {kind}' for name, kind in FACT_COLUMNS)}) USING DELTA")

### 7a. Care contacts and identity lookups

- `_carecontact`: one row per care contact. A few `CARECONID`s appear more than once in the raw table;
  the kept row is the one with a contact time (earliest first), then the newest update, then the lowest
  pregnancy id. Duplicates are recorded as dropped (`dup_careconid`). `held` says whether the
  appointment happened (section 8): attended (5 or 6), or nothing says otherwise (no code and no
  cancellation date).
- `_baby_id`: `LPIDBABY` -> infant person id, so baby facts reach the right infant.
- `_mother_id`: `LPIDMOTHER` -> mother person id, for mother facts that carry no pregnancy (msd003, msd004).
  An `LPIDMOTHER` shared by two mothers is ambiguous and left out; its facts are then dropped as having
  no eligible person.

In [0]:
def stage_care_contacts():
    """Create STG._carecontact (one row per CARECONID) and record the duplicates dropped."""
    keep_order = """CASE WHEN CCONTACTDATETIME IS NULL THEN 1 ELSE 0 END, CCONTACTDATETIME,
                    CASE WHEN ADC_UPDT IS NULL THEN 1 ELSE 0 END, ADC_UPDT DESC,
                    PREGNANCYID"""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._carecontact AS
    SELECT CARECONID, PREGNANCYID, CCONTACTDATETIME, ATTENDCODE, MEDIUM, CANCELDATE, CONTACTDURATION,
           CONSULTTYPE, CCSUBJECT, LOCCODE, ADMINCATCODE,
           (ATTENDCODE IN ('5', '6') OR (ATTENDCODE IS NULL AND CANCELDATE IS NULL)) AS held
    FROM (
      SELECT CARECONID, PREGNANCYID, CCONTACTDATETIME,
             nullif(trim(ATTENDCODE), '') AS ATTENDCODE, nullif(trim(MEDIUM), '') AS MEDIUM, CANCELDATE,
             CONTACTDURATION, CONSULTTYPE, CCSUBJECT, LOCCODE, ADMINCATCODE,
             row_number() OVER (PARTITION BY CARECONID ORDER BY {keep_order}) AS rn
      FROM {SRC['msd201carecontactpreg']} WHERE CARECONID IS NOT NULL
    ) WHERE rn = 1
    """)
    record_dropped(f"""
    SELECT DISTINCT 'MSDS-201-DEDUP', CAST(CARECONID AS STRING), 'dup_careconid' FROM (
      SELECT CARECONID, row_number() OVER (PARTITION BY CARECONID ORDER BY {keep_order}) AS rn
      FROM {SRC['msd201carecontactpreg']} WHERE CARECONID IS NOT NULL
    ) WHERE rn > 1""")
    raw = scalar(f"SELECT count(*) FROM {SRC['msd201carecontactpreg']} WHERE CARECONID IS NOT NULL")
    kept = spark.table(f"{STG}._carecontact").count()
    print("care contacts:", kept, "| duplicate rows dropped:", raw - kept)


def stage_identity_lookups():
    """Create STG._baby_id (LPIDBABY -> infant person id) and STG._mother_id (LPIDMOTHER -> mother person id)."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._baby_id AS
    WITH ident AS (
      SELECT LPIDBABY, {BABY_KEY_COLUMNS} FROM {SRC['msd401babydemo']}
    ),
    one_per_baby AS (
      SELECT * FROM (
        SELECT i.*, row_number() OVER (PARTITION BY _idkind, _k1, _k2 ORDER BY _idkind) AS rn FROM ident i
      ) WHERE rn = 1
    )
    SELECT LPIDBABY, {INFANT_ID} AS infant_person_id
    FROM one_per_baby WHERE nullif(trim(LPIDBABY), '') IS NOT NULL
    """)
    # One LPIDBABY must mean one baby, or baby facts would be copied to several infants.
    assert_unique(f"{STG}._baby_id", "LPIDBABY")

    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._mother_id AS
    SELECT LPIDMOTHER, min(pid) AS person_id FROM (
      SELECT LPIDMOTHER, {mint_id('person', 'ORGIDLPID', 'LPIDMOTHER')} AS pid
      FROM {SRC['msd001motherdemo']} WHERE LPIDMOTHER IS NOT NULL)
    GROUP BY LPIDMOTHER HAVING count(DISTINCT pid) = 1
    """)


stage_care_contacts()
stage_identity_lookups()

### 7b. Core coded tables

Seven MSDS tables are lists of SNOMED codes: diagnoses (msd106, msd107 previous, msd404 neonatal), findings
and observations (msd109), and care activities (msd202 antenatal, msd302 labour, msd405 baby), where each
activity row can carry a procedure, a finding and an observation code. Each code column is emitted
separately:

1. the code is resolved to standard concepts (curated choice first, then section 5a's lookup);
2. the concept's **domain picks the OMOP table** (Condition, Measurement, Observation, Procedure);
3. when the code does not resolve (concept 0) the column's default table is used:
   diagnoses -> condition, `OBSCODE` -> measurement, `FINDINGCODE` -> observation,
   `PROCEDURECODE` -> procedure.

Truly repeated source rows are kept apart by an occurrence number (`_occ`) so each still gets its own id.

In [0]:
CODE_COLUMN_DEFAULT_TABLE = {
    "DIAG": "condition_occurrence",
    "PREVDIAG": "condition_occurrence",
    "OBSCODE": "measurement",
    "FINDINGCODE": "observation",
    "PROCEDURECODE": "procedure_occurrence",
}

DOMAIN_TABLE_CASE = """CASE WHEN tc.domain_id = 'Condition' THEN 'condition_occurrence'
              WHEN tc.domain_id = 'Measurement' THEN 'measurement'
              WHEN tc.domain_id = 'Observation' THEN 'observation'
              WHEN tc.domain_id = 'Procedure' THEN 'procedure_occurrence'
              ELSE CAST(NULL AS STRING) END"""


def emit_core_facts(msds_table, code_column, date_sql, source_sql, row_key_columns, value_column=None,
                    unit_column=None, baby_key_sql=None, careconid_sql=None, extra_order_columns=None):
    """Append one _all_facts row per (source row, standard concept) for one SNOMED code column.

    source_sql      a SELECT over the raw table that also exposes PREGNANCYID
    row_key_columns the columns identifying a source row; with _occ and the concept they form fact_natural_key
    date_sql        the event date, as SQL over alias m
    value_column    numeric value column (OBSVALUE), unit_column its UCUM unit
    """
    default_table = CODE_COLUMN_DEFAULT_TABLE[code_column]
    person_role = "infant" if msds_table in ("msd404diagneo", "msd405careactbaby") else "mother"

    # _occ numbers truly repeated source rows. Its order must cover every column that affects the output,
    # or the numbering (and so the ids) could change between runs.
    order_columns = list(row_key_columns) + ["PREGNANCYID"] + list(extra_order_columns or [])
    if value_column:
        order_columns.append(value_column)
    if unit_column:
        order_columns.append(unit_column)
    order_columns = list(dict.fromkeys(order_columns))          # drop repeats, keep order
    occurrence_order = ", ".join(f"coalesce(cast({c} as string),'~')" for c in order_columns)
    row_key = ", ".join(row_key_columns)

    value_sql = f"try_cast(m.{value_column} AS DOUBLE)" if value_column else "CAST(NULL AS DOUBLE)"
    unit_sql = f"m.{unit_column}" if unit_column else "CAST(NULL AS STRING)"
    natural_key_sql = ("concat_ws('#', " + ", ".join(f"m.{c}" for c in row_key_columns)
                       + ", cast(m._occ AS string), cast(m.resolved_concept_id AS string))")

    spark.sql(f"""
    WITH
    -- source rows with a non-blank code, numbered within repeats
    occ AS (
      SELECT src.*, row_number() OVER (PARTITION BY {row_key} ORDER BY {occurrence_order}) AS _occ
      FROM (SELECT * FROM ({source_sql}) src0
            WHERE src0.{code_column} IS NOT NULL AND trim(src0.{code_column}) <> '') src
    ),
    -- the standard concepts each code stands for: a curated choice, else the SNOMED lookup
    resolved AS (
      SELECT o.*, cur.target_omop_table AS curated_table, cur.concept_id AS curated_concept_id,
             r.src_concept_id AS resolved_source_concept_id,
             CASE WHEN cur.concept_id IS NOT NULL THEN array(CAST(cur.concept_id AS BIGINT))
                  ELSE {snomed_concepts_sql('r')} END AS resolved_ids
      FROM occ o
      LEFT JOIN {curated_snomed_values_sql()} cur ON cur.code = o.{code_column}
      LEFT JOIN {STG}._resolved_snomed r ON r.concept_code = o.{code_column}
    ),
    -- one row per concept
    expanded AS (
      SELECT x.*, explode(x.resolved_ids) AS resolved_concept_id, size(x.resolved_ids) AS resolved_count
      FROM resolved x
    )
    INSERT INTO {STG}._all_facts (source_table, pregnancy_natural_key, baby_natural_key, careconid,
      fact_natural_key, target_omop_table, concept_id, multi_map_flag, source_value, source_concept_id,
      event_date, value_as_number, unit_source_value, rule_id, person_role)
    SELECT
      '{msds_table}' AS source_table,
      cast(m.PREGNANCYID as string) AS pregnancy_natural_key,
      {baby_key_sql or "CAST(NULL AS STRING)"} AS baby_natural_key,
      {careconid_sql or "CAST(NULL AS STRING)"} AS careconid,
      {natural_key_sql} AS fact_natural_key,
      CASE WHEN m.curated_concept_id IS NOT NULL THEN m.curated_table
           WHEN m.resolved_concept_id = 0 THEN '{default_table}'
           ELSE {DOMAIN_TABLE_CASE} END AS target_omop_table,
      m.resolved_concept_id AS concept_id,
      CASE WHEN m.resolved_count > 1 THEN 1 ELSE 0 END AS multi_map_flag,
      m.{code_column} AS source_value,
      m.resolved_source_concept_id AS source_concept_id,
      {date_sql} AS event_date,
      {value_sql} AS value_as_number,
      {unit_sql} AS unit_source_value,
      'MSDS-CORE-{msds_table}' AS rule_id,
      '{person_role}' AS person_role
    FROM expanded m LEFT JOIN {VOCAB}.concept tc ON tc.concept_id = m.resolved_concept_id
    """)
    print("core facts emitted:", msds_table, code_column)


def emit_all_core_facts():
    """Emit every SNOMED code column of the seven coded MSDS tables."""
    # Baby tables carry LPIDBABY; their pregnancy comes through the baby's labour record.
    baby_labour = (f"LEFT JOIN (SELECT LPIDBABY, max(LABOURDELIVERYID) AS LABOURDELIVERYID FROM {SRC['msd401babydemo']} "
                   "WHERE LPIDBABY IS NOT NULL GROUP BY LPIDBABY) b ON b.LPIDBABY={alias}.LPIDBABY "
                   f"LEFT JOIN {SRC['msd301labdel']} l ON l.LABOURDELIVERYID=b.LABOURDELIVERYID")
    from_404 = (f"SELECT d.*, l.PREGNANCYID FROM {SRC['msd404diagneo']} d "
                + baby_labour.format(alias="d") + " WHERE d.DIAG IS NOT NULL")
    from_405 = f"SELECT a.*, l.PREGNANCYID FROM {SRC['msd405careactbaby']} a " + baby_labour.format(alias="a")
    # msd202 activities take their pregnancy and date from their (deduplicated) care contact.
    from_202 = (f"SELECT a.*, c.PREGNANCYID, c.CCONTACTDATETIME FROM {SRC['msd202careactivitypreg']} a "
                f"LEFT JOIN {STG}._carecontact c ON c.CARECONID=a.CARECONID")
    # msd302 labour activities take their pregnancy from the labour record.
    from_302 = (f"SELECT a.*, l.PREGNANCYID FROM {SRC['msd302careactlabdel']} a "
                f"LEFT JOIN {SRC['msd301labdel']} l ON l.LABOURDELIVERYID=a.LABOURDELIVERYID")

    # Pregnancy diagnoses, previous diagnoses, neonatal diagnoses.
    emit_core_facts("msd106diagnosispreg", "DIAG", "try_cast(m.DIAGDATE AS DATE)",
                    f"SELECT * FROM {SRC['msd106diagnosispreg']} WHERE DIAG IS NOT NULL",
                    ["PREGNANCYID", "DIAG", "DIAGDATE"])
    emit_core_facts("msd107medhistory", "PREVDIAG", "try_cast(m.DIAGDATE AS DATE)",
                    f"SELECT * FROM {SRC['msd107medhistory']} WHERE PREVDIAG IS NOT NULL",
                    ["PREGNANCYID", "PREVDIAG", "DIAGDATE"])
    emit_core_facts("msd404diagneo", "DIAG", "try_cast(m.DIAGDATE AS DATE)", from_404,
                    ["LPIDBABY", "DIAGSCHEME", "DIAG", "DIAGDATE"], baby_key_sql="m.LPIDBABY")

    # Mother's findings and observations.
    emit_core_facts("msd109findobsmother", "OBSCODE", "try_cast(m.OBSDATE AS DATE)",
                    f"SELECT * FROM {SRC['msd109findobsmother']} WHERE OBSCODE IS NOT NULL",
                    ["PREGNANCYID", "OBSCODE", "OBSDATE"], value_column="OBSVALUE", unit_column="UCUMUNIT")
    emit_core_facts("msd109findobsmother", "FINDINGCODE", "try_cast(m.FINDINGDATE AS DATE)",
                    f"SELECT * FROM {SRC['msd109findobsmother']} WHERE FINDINGCODE IS NOT NULL",
                    ["PREGNANCYID", "FINDINGCODE", "FINDINGDATE"])

    # Antenatal care activities, dated by their care contact.
    for code_column in ("PROCEDURECODE", "FINDINGCODE", "OBSCODE"):
        is_obs = code_column == "OBSCODE"
        emit_core_facts("msd202careactivitypreg", code_column, "try_cast(m.CCONTACTDATETIME AS DATE)", from_202,
                        ["CAREACTIDMOTHER", code_column],
                        value_column="OBSVALUE" if is_obs else None, unit_column="UCUMUNIT" if is_obs else None,
                        careconid_sql="m.CARECONID", extra_order_columns=["CARECONID", "CCONTACTDATETIME"])

    # Labour and delivery activities.
    for code_column in ("PROCEDURECODE", "FINDINGCODE", "OBSCODE"):
        is_obs = code_column == "OBSCODE"
        emit_core_facts("msd302careactlabdel", code_column, "try_cast(m.CLININTERDATETIMEMOTHER AS DATE)", from_302,
                        ["LABOURDELIVERYID", "CLININTERDATETIMEMOTHER", "CAREPROFLID", code_column],
                        value_column="OBSVALUE" if is_obs else None, unit_column="UCUMUNIT" if is_obs else None)

    # Baby care activities.
    for code_column in ("PROCEDURECODE", "FINDINGCODE", "OBSCODE"):
        is_obs = code_column == "OBSCODE"
        emit_core_facts("msd405careactbaby", code_column, "try_cast(m.CLININTERDATETIMEBABY AS DATE)", from_405,
                        ["CAREACTIDBABY", code_column],
                        value_column="OBSVALUE" if is_obs else None, unit_column="UCUMUNIT" if is_obs else None,
                        baby_key_sql="m.LPIDBABY", extra_order_columns=["LPIDBABY", "CLININTERDATETIMEBABY"])
    print("_all_facts rows after the core tables:", spark.table(f"{STG}._all_facts").count())


emit_all_core_facts()

### 7c. Spec rules

Each non-structural rule in `MSDS_COLUMN_SPEC` becomes one SELECT; the rules of one MSDS table are
appended together. How a rule's rows are built:

- **Whose fact** (`grain`): mother rules link by `LPIDMOTHER`; pregnancy and care-contact rules by the
  row's `PREGNANCYID`; labour rules through the labour record; baby rules by `LPIDBABY`, with the
  pregnancy found through the baby's labour record.
- **Value**: blank values are skipped; numbers and dates must parse.
- **Concepts** come from section 5: the question (`target`), the decoded answer (`code_map`), the unit and
  status. For a rule whose column *is* a SNOMED code (no target), the code is resolved like the core
  tables and its domain picks the table.
- **source_value** is the rule_id for question rules (the raw answer is in `value_source_value`); the raw
  code for code-column rules; `rule_id:raw` for procedures, which have no value column.
- **fact_natural_key** = rule | grain key | date | value | helper columns | concept. Exact duplicate source
  rows mint the same key and collapse to one fact.

In [0]:
SPEC_FACT_KINDS = {"obs_coded", "obs_flag", "obs_number", "obs_date", "meas_number", "obs_snomed", "meas_snomed",
                   "condition", "procedure"}
KIND_TABLE = {"obs_coded": "observation", "obs_flag": "observation", "obs_number": "observation",
              "obs_date": "observation", "obs_snomed": "observation", "meas_number": "measurement",
              "meas_snomed": "measurement", "condition": "condition_occurrence", "procedure": "procedure_occurrence"}
YES_NO = {"Y": YES, "N": NO}
NULL_STRING = "CAST(NULL AS STRING)"


def spark_literal(value):
    """A Spark SQL string literal (backslash-escaped), for values written into generated SQL."""
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def rule_dates(rule):
    """A rule's event-date columns, tried in order: 'date' is one column, a list, or None."""
    date = rule["date"]
    if date is None:
        return []
    if isinstance(date, str):
        return [date]
    return list(date)


def rule_source_sql(rule):
    """SELECT over the rule's MSDS table adding the link keys: _preg (PREGNANCYID), _mother (LPIDMOTHER),
    _baby (LPIDBABY), _cc (CARECONID) and _gk (the grain key that goes into the natural key)."""
    table, grain = rule["table"], rule["grain"]
    source = SRC[table]
    labour, babies = SRC["msd301labdel"], SRC["msd401babydemo"]

    if rule.get("date_fallback") == "baby_birth":
        # Last-resort event date: the delivery's earliest baby birth time, as column _birth_ts.
        assert grain == "labourdelivery", f"{rule['rule_id']}: date_fallback baby_birth needs grain labourdelivery"
        source = (f"(SELECT s.*, bb._birth_ts FROM {source} s LEFT JOIN (SELECT LABOURDELIVERYID, "
                  f"min(coalesce(PERSONBIRTHDATETIMEBABY, BABYBIRTHDATETIME)) AS _birth_ts FROM {babies} "
                  f"GROUP BY LABOURDELIVERYID) bb ON bb.LABOURDELIVERYID = s.LABOURDELIVERYID)")

    if grain == "mother":
        return (f"SELECT s.*, {NULL_STRING} AS _preg, s.LPIDMOTHER AS _mother, {NULL_STRING} AS _baby, "
                f"{NULL_STRING} AS _cc, s.LPIDMOTHER AS _gk FROM {source} s")
    if grain == "pregnancy":
        return (f"SELECT s.*, s.PREGNANCYID AS _preg, {NULL_STRING} AS _mother, {NULL_STRING} AS _baby, "
                f"{NULL_STRING} AS _cc, s.PREGNANCYID AS _gk FROM {source} s")
    if grain == "carecontact":
        return (f"SELECT s.*, s.PREGNANCYID AS _preg, {NULL_STRING} AS _mother, {NULL_STRING} AS _baby, "
                f"s.CARECONID AS _cc, s.CARECONID AS _gk FROM {source} s")
    if grain == "labourdelivery":
        if table == "msd301labdel":
            return (f"SELECT s.*, s.PREGNANCYID AS _preg, {NULL_STRING} AS _mother, {NULL_STRING} AS _baby, "
                    f"{NULL_STRING} AS _cc, s.LABOURDELIVERYID AS _gk FROM {source} s")
        return (f"SELECT s.*, l.PREGNANCYID AS _preg, {NULL_STRING} AS _mother, {NULL_STRING} AS _baby, "
                f"{NULL_STRING} AS _cc, s.LABOURDELIVERYID AS _gk "
                f"FROM {source} s LEFT JOIN {labour} l ON l.LABOURDELIVERYID = s.LABOURDELIVERYID")
    if grain == "baby":
        if table == "msd401babydemo":
            pregnancy_link = f"LEFT JOIN {labour} l ON l.LABOURDELIVERYID = s.LABOURDELIVERYID"
        else:
            pregnancy_link = (f"LEFT JOIN (SELECT LPIDBABY, max(LABOURDELIVERYID) AS LABOURDELIVERYID FROM {babies} "
                              f"WHERE LPIDBABY IS NOT NULL GROUP BY LPIDBABY) b ON b.LPIDBABY = s.LPIDBABY "
                              f"LEFT JOIN {labour} l ON l.LABOURDELIVERYID = b.LABOURDELIVERYID")
        return (f"SELECT s.*, l.PREGNANCYID AS _preg, {NULL_STRING} AS _mother, s.LPIDBABY AS _baby, "
                f"{NULL_STRING} AS _cc, s.LPIDBABY AS _gk FROM {source} s {pregnancy_link}")
    raise ValueError(f"{rule['rule_id']}: grain {grain} is structural only")


def qualifier_decoder_rule(rule):
    """The rule whose code_map decodes this rule's aux qualifier column: itself for obs_date, else the rule
    on the same table whose value column is the qualifier column. None when there is no qualifier."""
    qualifier_column = (rule.get("aux") or {}).get("qualifier")
    if not qualifier_column:
        return None
    if rule["kind"] == "obs_date":
        return rule["rule_id"]
    for other in MSDS_COLUMN_SPEC:
        if other["table"] == rule["table"] and other["columns"] == [qualifier_column] and other["code_map"]:
            return other["rule_id"]
    return None


def spec_rule_sql(rule):
    """The SELECT producing one rule's _all_facts rows, in FACT_COLUMNS order."""
    rule_id, kind, column = rule["rule_id"], rule["kind"], rule["columns"][0]
    assert kind in SPEC_FACT_KINDS, f"{rule_id}: kind {kind} is structural"
    R = spark_literal(rule_id)
    aux = rule.get("aux") or {}

    # --- 1. the rule's source rows: value _v, event date _d, helper columns _aux_<name> ---
    value = f"nullif(trim(cast(s.{column} as string)), '')"
    date_columns = rule_dates(rule) + (["_birth_ts"] if rule.get("date_fallback") == "baby_birth" else [])
    if date_columns:
        event_date = "coalesce(" + ", ".join(f"try_cast(s.{c} AS DATE)" for c in date_columns) + ")"
    else:
        event_date = "CAST(NULL AS DATE)"
    aux_columns = "".join(f", nullif(trim(cast(s.{c} as string)), '') AS _aux_{name}" for name, c in aux.items())
    filters = [f"{value} IS NOT NULL"]
    if rule.get("only_codes"):
        filters.append(f"{value} IN ({', '.join(spark_literal(code) for code in rule['only_codes'])})")
    if kind in ("obs_number", "meas_number"):
        filters.append(f"try_cast({value} AS DOUBLE) IS NOT NULL")
    if kind == "obs_date":
        filters.append(f"try_cast({value} AS TIMESTAMP) IS NOT NULL")
    rows = (f"SELECT s.*, {value} AS _v, {event_date} AS _d{aux_columns} FROM ({rule_source_sql(rule)}) s "
            f"WHERE {' AND '.join(filters)}")

    # --- 2. the fact's concept _cid (and _scid, _n), and its OMOP table ---
    joins = []
    is_code_column = rule["target"] is None
    if is_code_column:
        # The column is a SNOMED code: one row per standard concept it stands for; the domain picks the table.
        concepts = snomed_concepts_sql("rs")
        with_concept = (f"SELECT i.*, rs.src_concept_id AS _scid, explode({concepts}) AS _cid, size({concepts}) AS _n "
                        f"FROM ({rows}) i LEFT JOIN {STG}._resolved_snomed rs ON rs.concept_code = i._v")
        joins.append(f"LEFT JOIN {VOCAB}.concept tc ON tc.concept_id = x._cid")
        by_domain = " ".join(f"WHEN tc.domain_id = '{d}' THEN '{t}'" for d, t in
                             (("Condition", "condition_occurrence"), ("Measurement", "measurement"),
                              ("Observation", "observation"), ("Procedure", "procedure_occurrence")))
        omop_table = f"CASE WHEN coalesce(x._cid, 0) = 0 THEN '{KIND_TABLE[kind]}' {by_domain} ELSE CAST(NULL AS STRING) END"
        source_value = "x._v"
    else:
        # A question rule: the concept is the rule's resolved question.
        with_concept = (f"SELECT i.*, CAST(NULL AS BIGINT) AS _scid, tg.concept_id AS _cid, 1 AS _n FROM ({rows}) i "
                        f"LEFT JOIN (SELECT concept_id FROM {STG}._spec_resolved WHERE rule_id = {R} AND role = 'target') tg ON true")
        omop_table = spark_literal(KIND_TABLE[kind])
        value_is_the_date = column in rule_dates(rule)
        if kind == "procedure" and not value_is_the_date and not rule.get("only_codes"):
            source_value = f"concat({R}, ':', x._v)"
        else:
            source_value = R

    # --- 3. the answer, by kind ---
    value_concept = value_datetime = value_string = value_source = value_number = "NULL"
    if kind == "obs_coded":
        joins.append(f"LEFT JOIN (SELECT src_code, src_text, concept_id FROM {STG}._spec_resolved "
                     f"WHERE rule_id = {R} AND role = 'value') vr ON vr.src_code = x._v")
        value_concept, value_string, value_source = "coalesce(vr.concept_id, 0)", "vr.src_text", "x._v"
    elif kind == "obs_flag":
        value_concept = "CASE upper(x._v) " + " ".join(f"WHEN '{k}' THEN {v}" for k, v in YES_NO.items()) + " ELSE 0 END"
        value_source = "x._v"
    elif kind in ("obs_number", "meas_number"):
        if rule.get("scale"):
            value_number = f"try_cast(x._v AS DOUBLE) * {float(rule['scale'])}"
        else:
            value_number = "try_cast(x._v AS DOUBLE)"
        value_source = "x._v"
    elif kind == "obs_date":
        value_datetime, value_source = "try_cast(x._v AS TIMESTAMP)", "x._v"
        value_string = "date_format(try_cast(x._v AS TIMESTAMP), 'yyyy-MM-dd')"   # OMOP 5.4 observation has no datetime value
    elif kind == "meas_snomed":
        value_number = f"try_cast(x.{rule['columns'][1]} AS DOUBLE)"
        value_source = f"cast(x.{rule['columns'][1]} as string)"

    # --- 4. unit, qualifier and condition status ---
    unit_source = spark_literal(rule["unit"]) if rule.get("unit") else "NULL"
    unit_concept = "NULL"
    if rule.get("unit"):
        joins.append(f"LEFT JOIN (SELECT concept_id FROM {STG}._spec_resolved WHERE rule_id = {R} AND role = 'unit') un ON true")
        unit_concept = "coalesce(un.concept_id, 0)"

    qualifier_concept = qualifier_source = "NULL"
    if "qualifier" in aux:
        qualifier_source = "x._aux_qualifier"
        decoder = qualifier_decoder_rule(rule)
        if decoder:
            joins.append(f"LEFT JOIN (SELECT src_code, concept_id FROM {STG}._spec_resolved "
                         f"WHERE rule_id = {spark_literal(decoder)} AND role = 'value') qv ON qv.src_code = x._aux_qualifier")
            qualifier_concept = "CASE WHEN x._aux_qualifier IS NULL THEN NULL ELSE coalesce(qv.concept_id, 0) END"
    else:
        # Other helper columns (not the scheme, which is always SNOMED) are kept as 'name=value|name=value'.
        named = [name for name in aux if name != "scheme"]
        if named:
            qualifier_source = "nullif(concat_ws('|', " + ", ".join(
                f"CASE WHEN x._aux_{name} IS NOT NULL THEN concat('{name}=', x._aux_{name}) END" for name in named) + "), '')"

    status_concept = "NULL"
    if rule.get("condition_status"):
        joins.append(f"LEFT JOIN (SELECT concept_id FROM {STG}._spec_resolved WHERE rule_id = {R} AND role = 'status') st ON true")
        status_concept = "st.concept_id"

    # --- 5. whose fact, and its natural key ---
    if rule["grain"] == "baby" and rule.get("person") != "mother":
        person_role = "infant"
    else:
        person_role = "mother"
    key_parts = [R, "x._gk", "cast(x._d as string)", "x._v"] + [f"x._aux_{name}" for name in aux] + ["cast(x._cid as string)"]
    natural_key = "concat_ws('|', " + ", ".join(key_parts) + ")"

    output = {
        "source_table": spark_literal(rule["table"]),
        "pregnancy_natural_key": "cast(x._preg as string)",
        "baby_natural_key": "cast(x._baby as string)",
        "careconid": "cast(x._cc as string)",
        "fact_natural_key": natural_key,
        "target_omop_table": omop_table,
        "concept_id": "CAST(coalesce(x._cid, 0) AS BIGINT)",
        "multi_map_flag": "CASE WHEN x._n > 1 THEN 1 ELSE 0 END",
        "source_value": source_value,
        "source_concept_id": "CAST(x._scid AS BIGINT)",
        "event_date": "x._d",
        "value_as_number": f"CAST({value_number} AS DOUBLE)",
        "unit_source_value": f"CAST({unit_source} AS STRING)",
        "mother_natural_key": "cast(x._mother as string)",
        "value_as_concept_id": f"CAST({value_concept} AS BIGINT)",
        "value_as_datetime": f"CAST({value_datetime} AS TIMESTAMP)",
        "value_as_string": f"CAST({value_string} AS STRING)",
        "value_source_value": f"CAST({value_source} AS STRING)",
        "unit_concept_id": f"CAST({unit_concept} AS BIGINT)",
        "qualifier_concept_id": f"CAST({qualifier_concept} AS BIGINT)",
        "qualifier_source_value": f"CAST({qualifier_source} AS STRING)",
        "status_concept_id": f"CAST({status_concept} AS BIGINT)",
        "rule_id": R,
        "person_role": spark_literal(person_role),
    }
    projection = ",\n  ".join(f"{output[name]} AS {name}" for name, _ in FACT_COLUMNS)

    # --- 6. one row per natural key ---
    # A baby resubmitted under two labour records gives the same fact with two pregnancy keys: keep the linked
    # one, by a fixed order, so the choice never changes between runs.
    keep_first = ("pregnancy_natural_key IS NULL, pregnancy_natural_key ASC NULLS LAST, "
                  "careconid IS NULL, careconid ASC NULLS LAST, "
                  "mother_natural_key ASC NULLS LAST, source_value ASC NULLS LAST, value_source_value ASC NULLS LAST, "
                  "multi_map_flag DESC")
    return (f"SELECT {FACT_COLUMN_NAMES} FROM (SELECT *, row_number() OVER (PARTITION BY fact_natural_key ORDER BY "
            f"{keep_first}) AS _rn FROM (\nSELECT DISTINCT\n  {projection}\nFROM ({with_concept}) x\n"
            + "\n".join(joins) + "\n)) WHERE _rn = 1")


def emit_spec_facts():
    """Append every non-structural spec rule to STG._all_facts, one INSERT per MSDS table."""
    rules_by_table = {}
    skipped = []
    for rule in MSDS_COLUMN_SPEC:
        if rule["kind"] not in SPEC_FACT_KINDS:
            continue
        if SRC[rule["table"]] is None:
            skipped.append(rule["rule_id"])
            continue
        rules_by_table.setdefault(rule["table"], []).append(rule)

    for table, rules in rules_by_table.items():
        union = "\nUNION ALL\n".join(f"({spec_rule_sql(rule)})" for rule in rules)
        spark.sql(f"INSERT INTO {STG}._all_facts ({FACT_COLUMN_NAMES}) SELECT {FACT_COLUMN_NAMES} FROM (\n{union}\n) u")
        print("spec facts emitted:", table, len(rules), "rules")
    if skipped:
        print("spec rules skipped (the site does not hold their table):", skipped)


def drop_caesareans_already_in_msd302():
    """MSDS-301-CS (the caesarean date in the labour record) is a fallback: drop it where the labour activity
    table already has a caesarean procedure for the pregnancy (any descendant of Cesarean section)."""
    CESAREAN_SECTION = 4015701
    record_dropped(f"""
    SELECT f.rule_id, f.fact_natural_key, 'caesarean_already_in_msd302'
    FROM {STG}._all_facts f
    WHERE f.rule_id = 'MSDS-301-CS' AND EXISTS (
      SELECT 1 FROM {STG}._all_facts p JOIN {VOCAB}.concept_ancestor a
        ON a.descendant_concept_id = p.concept_id AND a.ancestor_concept_id = {CESAREAN_SECTION}
      WHERE p.source_table = 'msd302careactlabdel' AND p.target_omop_table = 'procedure_occurrence'
        AND p.pregnancy_natural_key = f.pregnancy_natural_key)""")
    spark.sql(f"""DELETE FROM {STG}._all_facts WHERE rule_id = 'MSDS-301-CS'
                  AND fact_natural_key IN (SELECT fact_natural_key FROM {STG}._dropped WHERE rule_id = 'MSDS-301-CS')""")
    print("caesarean dates dropped (already in msd302):",
          scalar(f"SELECT count(*) FROM {STG}._dropped WHERE rule_id = 'MSDS-301-CS'"))


emit_spec_facts()
drop_caesareans_already_in_msd302()
show(f"""SELECT rule_id, person_role, target_omop_table, count(*) n,
           count_if(concept_id = 0) concept0, count_if(value_as_concept_id = 0) value0
         FROM {STG}._all_facts WHERE rule_id NOT LIKE 'MSDS-CORE-%'
         GROUP BY ALL ORDER BY rule_id""")

## 8. Visits, ward and neonatal stays, deaths and home areas

These rows do not pass through `_all_facts`; each is staged in `STG` already in its published shape
(`_msds_visit`, `_msds_visit_detail`, `_msds_death`, `_msds_location`). Rows that cannot be placed are
recorded in `_dropped`. A visit with no end, or an end before its start, ends at its start (OMOP requires
an end date).

**Visits**

| Rule | Source | Visit |
|---|---|---|
| MSDS-201-VISIT | care contacts that were **held** (attended, or not cancelled and no attendance code) | face to face 9202; phone, video and other remote media (02, 03, 05 to 08) Telehealth 722455 |
| MSDS-501-SPELL | hospital provider spells | inpatient 9201 |
| MSDS-301-ADM | the delivery admission, **unless** a spell of the same pregnancy already covers its start | inpatient 9201 |
| MSDS-401-VISIT | each baby's birth admission | inpatient 9201 |
| MSDS-402-NNU | a neonatal unit admission that starts after the birth admission ended (a readmission) | 32037 if critical care, else 9201 |

Appointments that did not happen are not visits: did-not-attend becomes a "Did not attend" observation
(rule MSDS-201-DNA), and cancelled ones are recorded as dropped. Facts recorded at a contact that was not held
keep their row but have no visit.

**Visit details**: each neonatal unit stay (MSDS-402-NNU) under the birth visit or its readmission visit, and
each ward stay (MSDS-503-WARD) under its spell.

**Deaths**: mother (msd001) and baby (msd401), the earliest date per person.

**Home area**: the mother's latest postcode is looked up in the ONS postcode directory to find its LSOA
(a census area of about 1,500 people). Only the LSOA is published. The postcode is used in that one join
and is never written to any table.

In [0]:
TELEHEALTH_MEDIA = ("02", "03", "05", "06", "07", "08")   # NHS DD contact MEDIUM codes for remote contacts
VISIT_COLUMNS = ", ".join(column_names("visit_occurrence"))
VISIT_DETAIL_COLUMNS = ", ".join(column_names("visit_detail"))


def visit_select(visit_id, person_id, concept, start, end, source_value, pregnancy_id,
                 admitted_from="CAST(NULL AS STRING)", discharged_to="CAST(NULL AS STRING)",
                 preceding_visit="CAST(NULL AS BIGINT)"):
    """The SELECT list for one visit_occurrence row, in table column order, from SQL expressions.
    The end falls back to the start when it is missing or earlier than the start."""
    end_or_start = f"CASE WHEN {end} IS NULL OR {end} < {start} THEN {start} ELSE {end} END"
    return f"""
      {visit_id} AS visit_occurrence_id, {person_id} AS person_id, {concept} AS visit_concept_id,
      CAST({start} AS DATE) AS visit_start_date, CAST({start} AS TIMESTAMP) AS visit_start_datetime,
      CAST({end_or_start} AS DATE) AS visit_end_date, CAST({end_or_start} AS TIMESTAMP) AS visit_end_datetime,
      32817 AS visit_type_concept_id, CAST(NULL AS BIGINT) AS provider_id, CAST(NULL AS BIGINT) AS care_site_id,
      {source_value} AS visit_source_value, 0 AS visit_source_concept_id,
      CASE WHEN {admitted_from} IS NULL THEN NULL ELSE 0 END AS admitted_from_concept_id,
      {admitted_from} AS admitted_from_source_value,
      CASE WHEN {discharged_to} IS NULL THEN NULL ELSE 0 END AS discharged_to_concept_id,
      {discharged_to} AS discharged_to_source_value,
      {preceding_visit} AS preceding_visit_occurrence_id, '{SITE}' AS site_id, {pregnancy_id} AS pregnancy_id"""


def pregnancy_join(alias, pregnancy_id_column):
    """JOIN condition linking a staged pregnancy (alias) to a raw PREGNANCYID column."""
    return f"{alias}.pregnancy_id = {mint_id('pregnancy', pregnancy_id_column)}"

In [0]:
def stage_contact_visits():
    """MSDS-201-VISIT: one visit per held care contact; record the contacts that are not visits."""
    duration = "try_cast(cc.CONTACTDURATION AS INT)"
    end = f"CASE WHEN {duration} BETWEEN 1 AND 1440 THEN cc.CCONTACTDATETIME + make_interval(0,0,0,0,0,{duration},0) END"
    remote_media = ", ".join(f"'{code}'" for code in TELEHEALTH_MEDIA)
    source_value = ("concat_ws('|', coalesce(cc.CONSULTTYPE,''), coalesce(cc.CCSUBJECT,''), "
                    "coalesce(cc.LOCCODE,''), coalesce(cc.ADMINCATCODE,''))")
    columns = visit_select(
        visit_id=mint_id("visit", "cc.CARECONID"),
        person_id="pr.person_id",
        concept=f"CASE WHEN cc.MEDIUM IN ({remote_media}) THEN 722455 ELSE 9202 END",
        start="cc.CCONTACTDATETIME",
        end=end,
        source_value=source_value,
        pregnancy_id="pr.pregnancy_id")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit ({VISIT_COLUMNS})
    SELECT {columns}
    FROM {STG}._carecontact cc JOIN {STG}.pregnancy pr ON {pregnancy_join('pr', 'cc.PREGNANCYID')}
    WHERE cc.held AND cc.CCONTACTDATETIME IS NOT NULL""")
    record_dropped(f"""
    SELECT 'MSDS-201-VISIT', concat_ws('|', 'MSDS-201-VISIT', CARECONID),
           CASE WHEN NOT held THEN concat('contact_not_held:', coalesce(ATTENDCODE, 'cancelled'))
                ELSE 'no_contact_datetime' END
    FROM {STG}._carecontact WHERE NOT held OR CCONTACTDATETIME IS NULL""")


def stage_spell_visits():
    """MSDS-501-SPELL: one inpatient visit per hospital provider spell (latest update of each spell)."""
    if SRC["msd501hospprovspell"] is None:
        spark.sql(f"""CREATE OR REPLACE TABLE {STG}._spell (HOSPPROVSPELLNUM STRING, PREGNANCYID STRING,
                      STARTDATETIME TIMESTAMP, DISCHDATETIME TIMESTAMP)""")
        return
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._spell AS
    SELECT * FROM (
      SELECT s.*, row_number() OVER (PARTITION BY HOSPPROVSPELLNUM
                                     ORDER BY ADC_UPDT DESC NULLS LAST, STARTDATETIME NULLS LAST) AS rn
      FROM {SRC['msd501hospprovspell']} s WHERE nullif(trim(HOSPPROVSPELLNUM),'') IS NOT NULL
    ) WHERE rn = 1""")
    columns = visit_select(
        visit_id=mint_id("spell-visit", "s.HOSPPROVSPELLNUM"),
        person_id="pr.person_id",
        concept=9201,
        start="s.STARTDATETIME",
        end="s.DISCHDATETIME",
        source_value="concat_ws('|', coalesce(s.ADMMETHCODE,''), coalesce(s.PATIENTCLASSCODE,''), coalesce(s.DISCHMETHCODE,''))",
        pregnancy_id="pr.pregnancy_id",
        admitted_from="nullif(trim(s.SOURCEADMCODE),'')",
        discharged_to="nullif(trim(s.DISCHDESTCODE),'')")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit ({VISIT_COLUMNS})
    SELECT {columns}
    FROM {STG}._spell s JOIN {STG}.pregnancy pr ON {pregnancy_join('pr', 's.PREGNANCYID')}
    WHERE s.STARTDATETIME IS NOT NULL""")


def stage_delivery_visits():
    """MSDS-301-ADM: the delivery admission, unless a spell of the same pregnancy covers its start."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._delivery_visit AS
    SELECT d.LABOURDELIVERYID, d.PREGNANCYID, d.STARTDATETIMEMOTHERDELIVERYHPS AS adm_start,
           d.DISCHARGEDATETIMEMOTHERHSP AS adm_end, d.ADMMETHCODEMOTHDELHSP, d.DISCHMETHCODEMOTHPOSTDELHSP,
           d.DISCHDESTCODEMOTHPOSTDELHSP, min(s.HOSPPROVSPELLNUM) AS spell
    FROM {SRC['msd301labdel']} d
    LEFT JOIN {STG}._spell s ON s.PREGNANCYID = d.PREGNANCYID AND s.STARTDATETIME IS NOT NULL
      AND d.STARTDATETIMEMOTHERDELIVERYHPS BETWEEN s.STARTDATETIME AND coalesce(s.DISCHDATETIME, s.STARTDATETIME)
    WHERE d.STARTDATETIMEMOTHERDELIVERYHPS IS NOT NULL
    GROUP BY ALL""")
    columns = visit_select(
        visit_id=mint_id("delivery-visit", "d.LABOURDELIVERYID"),
        person_id="pr.person_id",
        concept=9201,
        start="d.adm_start",
        end="d.adm_end",
        source_value="concat_ws('|', coalesce(d.ADMMETHCODEMOTHDELHSP,''), coalesce(d.DISCHMETHCODEMOTHPOSTDELHSP,''))",
        pregnancy_id="pr.pregnancy_id",
        discharged_to="nullif(trim(d.DISCHDESTCODEMOTHPOSTDELHSP),'')")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit ({VISIT_COLUMNS})
    SELECT {columns}
    FROM {STG}._delivery_visit d JOIN {STG}.pregnancy pr ON {pregnancy_join('pr', 'd.PREGNANCYID')}
    WHERE d.spell IS NULL""")
    record_dropped(f"""
    SELECT 'MSDS-301-ADM', concat_ws('|', 'MSDS-301-ADM', LABOURDELIVERYID), concat('inside_spell:', spell)
    FROM {STG}._delivery_visit WHERE spell IS NOT NULL""")


def stage_birth_visits():
    """MSDS-401-VISIT: one birth admission per infant, starting at the birth time.

    Barts records the birth time in PERSONBIRTHDATETIMEBABY (BABYBIRTHDATETIME is empty); other sites may
    only fill the other one, so both are tried."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._birth_visit AS
    SELECT ipid, LPIDBABY, birth_ts, DISCHARGEDATETIMEBABYHSP, PREGNANCYID,
           {mint_id('birth-visit', 'ipid')} AS visit_occurrence_id
    FROM (
      SELECT i.*, row_number() OVER (PARTITION BY ipid ORDER BY
               CASE WHEN birth_ts IS NULL THEN 1 ELSE 0 END,
               CASE WHEN DISCHARGEDATETIMEBABYHSP IS NULL THEN 1 ELSE 0 END,
               ADC_UPDT DESC NULLS LAST, LABOURDELIVERYID) AS rn
      FROM (SELECT b.*, l.PREGNANCYID, coalesce(b.PERSONBIRTHDATETIMEBABY, b.BABYBIRTHDATETIME) AS birth_ts,
                   {INFANT_ID_B} AS ipid
            FROM (SELECT *, {BABY_KEY_COLUMNS} FROM {SRC['msd401babydemo']}) b
            LEFT JOIN {SRC['msd301labdel']} l ON l.LABOURDELIVERYID = b.LABOURDELIVERYID) i
    ) WHERE rn = 1 AND birth_ts IS NOT NULL""")
    columns = visit_select(
        visit_id="bv.visit_occurrence_id",
        person_id="bv.ipid",
        concept=9201,
        start="bv.birth_ts",
        end="bv.DISCHARGEDATETIMEBABYHSP",
        source_value="'MSDS-401-VISIT'",
        pregnancy_id="pr.pregnancy_id")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit ({VISIT_COLUMNS})
    SELECT {columns}
    FROM {STG}._birth_visit bv LEFT JOIN {STG}.pregnancy pr ON {pregnancy_join('pr', 'bv.PREGNANCYID')}""")


def stage_neonatal_readmission_visits():
    """MSDS-402-NNU: stage every neonatal unit admission in STG._nnu, and give each readmission its own visit.

    An admission is a readmission when it starts after the birth admission ended (about half of Barts' stays,
    median age 3 days). msd402 has no end time, so a readmission visit ends at its start; it is preceded by the
    birth visit. Critical care when any row of the admission says so."""
    if SRC["msd402neoadmiss"] is None:
        spark.sql(f"""CREATE OR REPLACE TABLE {STG}._nnu (LPIDBABY STRING, nnu_start TIMESTAMP, crit BOOLEAN,
                      ipid BIGINT, birth_vid BIGINT, PREGNANCYID STRING, birth_end TIMESTAMP, readmit BOOLEAN,
                      readmit_vid BIGINT)""")
        return
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._nnu AS
    SELECT n.LPIDBABY, n.nnu_start, n.crit, bv.ipid, bv.visit_occurrence_id AS birth_vid, bv.PREGNANCYID,
           bv.DISCHARGEDATETIMEBABYHSP AS birth_end,
           coalesce(n.nnu_start > bv.DISCHARGEDATETIMEBABYHSP, false) AS readmit,
           {mint_id('nnu-visit', 'n.LPIDBABY', 'n.nnu_start')} AS readmit_vid
    FROM (SELECT LPIDBABY, NEONATALTRANSFERSTARTDATETIME AS nnu_start,
                 max(coalesce(upper(trim(NEOCRITCAREIND)) = 'Y', false)) AS crit
          FROM {SRC['msd402neoadmiss']} WHERE NEONATALTRANSFERSTARTDATETIME IS NOT NULL GROUP BY ALL) n
    JOIN {STG}._baby_id b ON b.LPIDBABY = n.LPIDBABY
    JOIN {STG}._birth_visit bv ON bv.ipid = b.infant_person_id""")
    columns = visit_select(
        visit_id="n.readmit_vid",
        person_id="n.ipid",
        concept="CASE WHEN n.crit THEN 32037 ELSE 9201 END",
        start="n.nnu_start",
        end="CAST(NULL AS TIMESTAMP)",
        source_value="'MSDS-402-NNU'",
        pregnancy_id="pr.pregnancy_id",
        preceding_visit="n.birth_vid")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit ({VISIT_COLUMNS})
    SELECT {columns}
    FROM {STG}._nnu n LEFT JOIN {STG}.pregnancy pr ON {pregnancy_join('pr', 'n.PREGNANCYID')}
    WHERE n.readmit""")


spark.sql(f"CREATE OR REPLACE TABLE {STG}._msds_visit ({CDM_TABLES['visit_occurrence']}) USING DELTA")
stage_contact_visits()
stage_spell_visits()
stage_delivery_visits()
stage_birth_visits()
stage_neonatal_readmission_visits()
assert_unique(f"{STG}._msds_visit", "visit_occurrence_id")
show(f"SELECT visit_concept_id, count(*) n FROM {STG}._msds_visit GROUP BY 1 ORDER BY 1")

In [0]:
def visit_detail_select(detail_id, person_id, concept, start, end, source_value, visit_id, pregnancy_id, source_table):
    """The SELECT list for one visit_detail row, in table column order. The end falls back to the start."""
    end_or_start = f"CASE WHEN {end} IS NULL OR {end} < {start} THEN {start} ELSE {end} END"
    return (f"{detail_id}, {person_id}, {concept}, CAST({start} AS DATE), CAST({start} AS TIMESTAMP), "
            f"CAST({end_or_start} AS DATE), CAST({end_or_start} AS TIMESTAMP), 32817, NULL, NULL, {source_value}, 0, "
            f"NULL, NULL, NULL, NULL, NULL, NULL, {visit_id}, '{SITE}', {pregnancy_id}, NULL, '{source_table}'")


def stage_neonatal_stays():
    """MSDS-402-NNU details: every neonatal unit stay, under the birth visit (ending with it) or under its own
    readmission visit (ending at its start). Critical care 32037, else inpatient 9201."""
    if SRC["msd402neoadmiss"] is None:
        return
    columns = visit_detail_select(
        detail_id=mint_id("nnu-detail", "n.LPIDBABY", "n.nnu_start"),
        person_id="n.ipid",
        concept="CASE WHEN n.crit THEN 32037 ELSE 9201 END",
        start="n.nnu_start",
        end="CASE WHEN n.readmit THEN n.nnu_start ELSE n.birth_end END",
        source_value="'MSDS-402-NNU'",
        visit_id="v.visit_occurrence_id",
        pregnancy_id="v.pregnancy_id",
        source_table="msd402neoadmiss")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit_detail_raw ({VISIT_DETAIL_COLUMNS})
    SELECT {columns}
    FROM {STG}._nnu n
    JOIN {STG}._msds_visit v ON v.visit_occurrence_id = CASE WHEN n.readmit THEN n.readmit_vid ELSE n.birth_vid END""")
    record_dropped(f"""
    SELECT 'MSDS-402-NNU', concat_ws('|', 'MSDS-402-NNU', n.LPIDBABY, cast(n.NEONATALTRANSFERSTARTDATETIME AS STRING)),
           'no_birth_visit'
    FROM {SRC['msd402neoadmiss']} n LEFT JOIN {STG}._baby_id b ON b.LPIDBABY = n.LPIDBABY
    LEFT JOIN {STG}._birth_visit bv ON bv.ipid = b.infant_person_id
    WHERE n.NEONATALTRANSFERSTARTDATETIME IS NOT NULL AND bv.ipid IS NULL""")


def stage_ward_stays():
    """MSDS-503-WARD: ward stays under their spell's visit."""
    if SRC["msd503wardstay"] is None:
        return
    spell_visit_id = mint_id("spell-visit", "w.HOSPPROVSPELLNUM")
    columns = visit_detail_select(
        detail_id=mint_id("ward-detail", "w.HOSPPROVSPELLNUM", "w.WARDCODE", "w.STARTDATETIMEWARDSTAY"),
        person_id="v.person_id",
        concept=9201,
        start="w.STARTDATETIMEWARDSTAY",
        end="w.ENDDATETIMEWARDSTAY",
        source_value="nullif(trim(w.WARDCODE),'')",
        visit_id="v.visit_occurrence_id",
        pregnancy_id="v.pregnancy_id",
        source_table="msd503wardstay")
    spark.sql(f"""
    INSERT INTO {STG}._msds_visit_detail_raw ({VISIT_DETAIL_COLUMNS})
    SELECT {columns}
    FROM (SELECT DISTINCT HOSPPROVSPELLNUM, WARDCODE, STARTDATETIMEWARDSTAY, ENDDATETIMEWARDSTAY
          FROM {SRC['msd503wardstay']} WHERE STARTDATETIMEWARDSTAY IS NOT NULL) w
    JOIN {STG}._msds_visit v ON v.visit_occurrence_id = {spell_visit_id}""")
    record_dropped(f"""
    SELECT 'MSDS-503-WARD', concat_ws('|', 'MSDS-503-WARD', w.HOSPPROVSPELLNUM, w.WARDCODE,
           cast(w.STARTDATETIMEWARDSTAY AS STRING)), 'no_spell_visit'
    FROM {SRC['msd503wardstay']} w WHERE w.STARTDATETIMEWARDSTAY IS NOT NULL AND NOT EXISTS (
      SELECT 1 FROM {STG}._msds_visit v WHERE v.visit_occurrence_id = {spell_visit_id})""")


def keep_one_row_per_visit_detail():
    """Two staged rows with one id but a different end or flag would collide: keep the latest end, then
    critical care. The result is STG._msds_visit_detail."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._msds_visit_detail AS
    SELECT {VISIT_DETAIL_COLUMNS} FROM (
      SELECT *, row_number() OVER (PARTITION BY visit_detail_id
                                   ORDER BY visit_detail_end_datetime DESC NULLS LAST,
                                            CASE WHEN visit_detail_concept_id = 32037 THEN 0 ELSE 1 END) AS rn
      FROM {STG}._msds_visit_detail_raw) WHERE rn = 1""")


spark.sql(f"CREATE OR REPLACE TABLE {STG}._msds_visit_detail_raw ({CDM_TABLES['visit_detail']}) USING DELTA")
stage_neonatal_stays()
stage_ward_stays()
keep_one_row_per_visit_detail()
show(f"SELECT source_table, visit_detail_concept_id, count(*) n FROM {STG}._msds_visit_detail GROUP BY ALL ORDER BY 1, 2")

In [0]:
def stage_deaths():
    """MSDS-001-DEATH and MSDS-401-DEATH: the earliest recorded death per mother or baby."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._msds_death AS
    WITH deaths AS (
      SELECT {mint_id('person', 'ORGIDLPID', 'LPIDMOTHER')} AS person_id, PERSONDEATHDATETIMEMOTHER AS dt
      FROM {SRC['msd001motherdemo']} WHERE LPIDMOTHER IS NOT NULL AND PERSONDEATHDATETIMEMOTHER IS NOT NULL
      UNION ALL
      SELECT {INFANT_ID_B}, b.PERSONDEATHDATETIMEBABY
      FROM (SELECT *, {BABY_KEY_COLUMNS} FROM {SRC['msd401babydemo']}) b WHERE b.PERSONDEATHDATETIMEBABY IS NOT NULL
    )
    SELECT person_id, CAST(min(dt) AS DATE) AS death_date, CAST(min(dt) AS TIMESTAMP) AS death_datetime,
           32817 AS death_type_concept_id, 0 AS cause_concept_id, CAST(NULL AS STRING) AS cause_source_value,
           0 AS cause_source_concept_id, '{SITE}' AS site_id
    FROM deaths GROUP BY person_id""")
    print("deaths:", spark.table(f"{STG}._msds_death").count())


# An LSOA code: E01 (England) or W01 (Wales) and six digits.
LSOA_PATTERN = "^(E01|W01)[0-9]{6}$"


def stage_home_areas():
    """MSDS-001-LOC: each mother's latest postcode -> LSOA (ONS postcode directory) -> location row.

    location_id is 8e15 + country (England 1, Wales 2) x 1e9 + the LSOA's six digits, the same scheme as the
    estate's OMOP location table, so the ids agree. The postcode is only joined on, never selected."""
    location_id = (f"CASE WHEN o.lsoa11 RLIKE '{LSOA_PATTERN}' THEN 8000000000000000 "
                   "+ (CASE WHEN o.lsoa11 LIKE 'E01%' THEN 1 ELSE 2 END) * 1000000000 "
                   "+ CAST(substring(o.lsoa11, 4, 6) AS BIGINT) END")
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._person_location AS
    WITH latest AS (
      SELECT person_id, pc FROM (
        SELECT {mint_id('person', 'ORGIDLPID', 'LPIDMOTHER')} AS person_id,
               upper(replace(POSTCODE, ' ', '')) AS pc,
               row_number() OVER (PARTITION BY ORGIDLPID, LPIDMOTHER ORDER BY
                                  CASE WHEN nullif(trim(POSTCODE),'') IS NULL THEN 1 ELSE 0 END,
                                  ADC_UPDT DESC NULLS LAST) AS rn
        FROM {SRC['msd001motherdemo']} WHERE LPIDMOTHER IS NOT NULL) WHERE rn = 1
    )
    SELECT l.person_id, {location_id} AS location_id, o.lsoa11 AS lsoa, o.oslaua AS county
    FROM latest l JOIN `3_lookup`.ons.onspd o ON upper(replace(o.pcds, ' ', '')) = l.pc
    WHERE o.lsoa11 RLIKE '{LSOA_PATTERN}'""")
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._msds_location AS
    SELECT location_id, CAST(NULL AS STRING) AS address_1, CAST(NULL AS STRING) AS address_2,
           CAST(NULL AS STRING) AS city, CAST(NULL AS STRING) AS state, CAST(NULL AS STRING) AS zip,
           max(county) AS county, lsoa AS location_source_value, 42035286 AS country_concept_id,
           'GB' AS country_source_value, CAST(NULL AS DOUBLE) AS latitude, CAST(NULL AS DOUBLE) AS longitude,
           '{SITE}' AS site_id
    FROM {STG}._person_location GROUP BY location_id, lsoa""")
    locations = assert_unique(f"{STG}._msds_location", "location_id")
    print("mothers with an LSOA:", spark.table(f"{STG}._person_location").count(), "| LSOAs:", locations)


stage_deaths()
stage_home_areas()

## 9. Cleaning

This is the **only** place the build decides a staged row cannot be published. Every decision is written to
`STG._dropped` (rule, row key, reason) first, then applied. Nothing is corrected or guessed, and nothing goes
to a review queue: rows are dropped, or one implausible column is blanked, and the reason is counted.

1. **People** with no birth date cannot be placed in time, so they are not eligible; their facts go too.
2. **Facts**: the first matching reason wins:

   | Reason | Meaning |
   |---|---|
   | unsupported_domain | the concept's domain has no fact table here |
   | unresolved_infant | a baby fact whose `LPIDBABY` matches no baby |
   | no_eligible_person | no mother or baby to attach it to, or that person has no birth date |
   | null_event_date | no date (a mother's fact with no date takes its pregnancy's start date first) |
   | out_of_range | an implausible value (`FACT_RANGES`) |
   | before_1900, after_release | outside the dataset's time window |
   | before_birth | before the person was born (babies: more than a day before, as birth times can be a day off) |
   | after_death | more than 30 days after the person's death |

3. **Pregnancy and infant columns** outside a plausible range (`COLUMN_RANGES`) are set to NULL; the rest
   of the row is kept.
4. **Visits, visit details and deaths** outside the time window are dropped, and a visit detail whose
   visit was dropped goes with it.

There is no lower limit on gestation: early losses are real.

In [0]:
# Implausible fact values: the whole fact is dropped. Values are after the spec's scaling (height in cm).
FACT_RANGES = {
    "MSDS-101-WT": (30, 250),       # kg
    "MSDS-101-HT": (120, 220),      # cm
    "MSDS-101-PREVCS": (0, 10),
    "MSDS-103-GA": (35, 315),       # days at the dating scan; late bookers are scanned up to term
    "MSDS-103-NFET": (1, 6),
    "MSDS-401-BW": (200, 7000),     # g
    "MSDS-401-APGAR": (0, 10),
}

# Implausible extension columns: the value is set to NULL, the row is kept.
COLUMN_RANGES = {
    ("pregnancy", "gestational_length_in_day"): (1, 315),
    ("pregnancy", "prev_pregnancy_gravidity"): (0, 29),    # previous pregnancies only
    ("pregnancy", "prev_livebirth_number"): (0, 20),
    ("pregnancy", "prev_still_misc_number"): (0, 20),
    ("infant", "birth_weight"): (200, 7000),
    ("infant", "birth_apgar"): (0, 10),
}
COLUMN_RANGE_KEYS = {"pregnancy": "pregnancy_id", "infant": "infant_id"}

# The fact tables an _all_facts row can go to, and the kind used to mint its id.
FACT_TABLE_ID_KIND = {
    "condition_occurrence": "condition", "procedure_occurrence": "procedure",
    "measurement": "measurement", "observation": "observation",
}


def between(sql_expr, low, high):
    """SQL: sql_expr BETWEEN low AND high, in brackets."""
    return f"({sql_expr} BETWEEN {low} AND {high})"

### Whose fact, and when

The same two SQL expressions decide, for an `_all_facts` row `f`, which person it belongs to and its event
date. Cleaning (here), the table build (section 10) and the reconciliation check (section 12) all use them,
so a fact survives cleaning exactly when the build can place it.

- Infant facts go to the baby found through `LPIDBABY`.
- Mother facts go to the mother of the fact's pregnancy, or, for facts with no pregnancy (msd003, msd004),
  to the mother found through `LPIDMOTHER`.
- A mother fact with no date takes its pregnancy's start date. An infant fact does not (that date is before
  the birth), so it stays undated and is dropped.

In [0]:
# Joins that bind aliases bl (the baby), mo (the mother by LPIDMOTHER) and pr (the fact's pregnancy).
FACT_ROUTING_JOINS = (f"LEFT JOIN {STG}._baby_id bl ON bl.LPIDBABY = f.baby_natural_key "
                      f"LEFT JOIN {STG}._mother_id mo ON mo.LPIDMOTHER = f.mother_natural_key "
                      f"LEFT JOIN {STG}.pregnancy pr ON pr.pregnancy_id = {mint_id('pregnancy', 'f.pregnancy_natural_key')}")
FACT_PERSON = "CASE WHEN f.person_role = 'infant' THEN bl.infant_person_id ELSE coalesce(pr.person_id, mo.person_id) END"
FACT_DATE = "coalesce(f.event_date, CASE WHEN f.person_role <> 'infant' THEN pr.pregnancy_start_date END)"

In [0]:
def stage_eligible_people():
    """STG._eligible = staged people with a birth date. Record the others as dropped."""
    spark.sql(f"""CREATE OR REPLACE TABLE {STG}._eligible AS
      SELECT person_id FROM {STG}.person WHERE site_id = '{SITE}' AND birth_datetime IS NOT NULL""")
    record_dropped(f"""
      SELECT concat('PERSON-', person_role), CAST(person_id AS STRING), 'null_birth_year'
      FROM {STG}.person WHERE site_id = '{SITE}' AND birth_datetime IS NULL""")
    print("eligible people:", spark.table(f"{STG}._eligible").count(),
          "| without a birth date:", scalar(f"SELECT count(*) FROM {STG}.person WHERE birth_datetime IS NULL"))


def drop_unpublishable_facts():
    """Decide, record and delete every _all_facts row that cannot be published (reasons in the table above).

    First records which version of _all_facts was cleaned, and the release date, in STG._clean_state; the
    reconciliation check (section 12) compares the published facts with that snapshot."""
    assert not spark.catalog.tableExists(f"{STG}._clean_state"), "this working schema has already been cleaned"
    version = spark.sql(f"DESCRIBE HISTORY {STG}._all_facts LIMIT 1").first()["version"]
    spark.sql(f"""CREATE TABLE {STG}._clean_state AS
      SELECT CAST({version} AS BIGINT) AS all_facts_version, DATE'{RELEASE_DATE}' AS release_date,
             current_timestamp() AS cleaned_at""")

    fact_tables = ", ".join(f"'{t}'" for t in FACT_TABLE_ID_KIND)
    out_of_range = " OR ".join(f"(rule_id = '{rule}' AND NOT {between('v', low, high)})"
                               for rule, (low, high) in FACT_RANGES.items())
    spark.sql(f"""
    CREATE TABLE {STG}._clean_drop AS
    WITH f AS (
      SELECT f.source_table, f.fact_natural_key, coalesce(f.rule_id, concat('SRC-', f.source_table)) AS rule_id,
             f.target_omop_table AS t, f.person_role, f.value_as_number AS v,
             {FACT_PERSON} AS rp, {FACT_DATE} AS ed, bl.infant_person_id AS bpid
      FROM {STG}._all_facts f {FACT_ROUTING_JOINS}),
    -- add: is the person eligible, their birth date and their death date
    g AS (
      SELECT f.*, el.person_id IS NOT NULL AS elig, CAST(p.b AS DATE) AS bd, d.death_date AS dd
      FROM f
      LEFT JOIN (SELECT DISTINCT person_id FROM {STG}._eligible) el ON el.person_id = f.rp
      LEFT JOIN (SELECT person_id, min(birth_datetime) AS b FROM {STG}.person GROUP BY person_id) p
        ON p.person_id = f.rp
      LEFT JOIN {STG}._msds_death d ON d.person_id = f.rp)
    SELECT source_table, fact_natural_key, rule_id, reason FROM (
      SELECT *, CASE
        WHEN t IS NULL OR t NOT IN ({fact_tables}) THEN 'unsupported_domain'
        WHEN person_role = 'infant' AND bpid IS NULL THEN 'unresolved_infant'
        WHEN rp IS NULL OR NOT elig THEN 'no_eligible_person'
        WHEN ed IS NULL THEN 'null_event_date'
        WHEN {out_of_range} THEN 'out_of_range'
        WHEN ed < DATE'1900-01-01' THEN 'before_1900'
        WHEN ed > DATE'{RELEASE_DATE}' THEN 'after_release'
        WHEN ed < CASE WHEN person_role = 'infant' THEN date_sub(bd, 1) ELSE bd END THEN 'before_birth'
        WHEN ed > date_add(dd, 30) THEN 'after_death'
      END AS reason FROM g)
    WHERE reason IS NOT NULL""")
    record_dropped(f"SELECT rule_id, fact_natural_key, reason FROM {STG}._clean_drop")
    spark.sql(f"""DELETE FROM {STG}._all_facts f WHERE EXISTS (SELECT 1 FROM {STG}._clean_drop d
      WHERE d.source_table = f.source_table AND d.fact_natural_key = f.fact_natural_key)""")


def null_out_of_range_columns():
    """Record, then set to NULL, each pregnancy or infant column value outside COLUMN_RANGES."""
    ranges_by_table = {}
    for (table, column), (low, high) in COLUMN_RANGES.items():
        ranges_by_table.setdefault(table, []).append((column, low, high))

    for table, ranges in ranges_by_table.items():
        key = COLUMN_RANGE_KEYS[table]
        selects = []
        for column, low, high in ranges:
            selects.append(f"""SELECT 'COL:{table}.{column}', CAST({key} AS STRING),
                                      concat('nulled_out_of_range:', CAST({column} AS STRING))
                               FROM {STG}.{table} WHERE {column} IS NOT NULL AND NOT {between(column, low, high)}""")
        record_dropped(" UNION ALL ".join(selects))

        assignments = ", ".join(f"{column} = CASE WHEN {between(column, low, high)} THEN {column} END"
                                for column, low, high in ranges)
        any_bad = " OR ".join(f"({column} IS NOT NULL AND NOT {between(column, low, high)})"
                              for column, low, high in ranges)
        spark.sql(f"UPDATE {STG}.{table} SET {assignments} WHERE {any_bad}")


# Staged structural tables that are windowed: (table, key column, date column).
WINDOWED_TABLES = [("_msds_visit", "visit_occurrence_id", "visit_start_date"),
                   ("_msds_visit_detail", "visit_detail_id", "visit_detail_start_date"),
                   ("_msds_death", "person_id", "death_date")]


def drop_structural_rows_outside_window():
    """Drop staged visits, visit details and deaths dated before 1900 or after the release date, and visit
    details whose visit was dropped (they would point at a visit that does not exist)."""
    for table, key, date_column in WINDOWED_TABLES:
        orphan_check = ""
        if table == "_msds_visit_detail":
            orphan_check = (f" WHEN NOT EXISTS (SELECT 1 FROM {STG}._msds_visit v "
                            f"WHERE v.visit_occurrence_id = x.visit_occurrence_id) THEN 'parent_visit_dropped'")
        spark.sql(f"""
        CREATE OR REPLACE TABLE {STG}._clean_window AS
        SELECT {key} AS k, reason FROM (
          SELECT x.{key}, CASE
            WHEN x.{date_column} < DATE'1900-01-01' THEN 'before_1900'
            WHEN x.{date_column} > DATE'{RELEASE_DATE}' THEN 'after_release'{orphan_check} END AS reason
          FROM {STG}.{table} x)
        WHERE reason IS NOT NULL""")
        record_dropped(f"SELECT 'WINDOW:{table}', CAST(k AS STRING), reason FROM {STG}._clean_window")
        spark.sql(f"DELETE FROM {STG}.{table} WHERE {key} IN (SELECT k FROM {STG}._clean_window)")


stage_eligible_people()
drop_unpublishable_facts()
null_out_of_range_columns()
drop_structural_rows_outside_window()
show(f"SELECT split(reason, ':')[0] AS reason, count(*) n FROM {STG}._dropped GROUP BY 1 ORDER BY 2 DESC")

## 10. Assemble the OMOP tables

The cleaned staging tables are copied into the run's `PUB` schema in the published shape. Only eligible
people (section 9) and their rows are copied. Every fact gets its id here, minted from its source table and
`fact_natural_key`, so the same source row always gets the same id.

### 10a. People

Gender: mothers are Female (8532); babies use `PERSONPHENSEX` (1 Male 8507, 2 Female 8532, anything else
Unknown 8551). Ethnicity stays 0 (MSDS has no OMOP ethnicity source; the ethnic category is in race).
A mother's `location_id` is her LSOA from section 8.

In [0]:
def build_person():
    """PUB.person from eligible staged people, with the full OMOP column set."""
    spark.sql(f"""
    INSERT INTO {PUB}.person (person_id, gender_concept_id, year_of_birth, month_of_birth,
      day_of_birth, birth_datetime, race_concept_id, ethnicity_concept_id, location_id,
      provider_id, care_site_id, person_source_value, gender_source_value,
      gender_source_concept_id, race_source_value, race_source_concept_id,
      ethnicity_source_value, ethnicity_source_concept_id, site_id, person_role)
    SELECT sp.person_id,
      CASE WHEN person_role = 'mother' THEN 8532
           WHEN gender_source_value = '1' THEN 8507
           WHEN gender_source_value = '2' THEN 8532
           ELSE 8551 END AS gender_concept_id,
      year(birth_datetime), month(birth_datetime), day(birth_datetime), birth_datetime,
      race_concept_id, 0 AS ethnicity_concept_id,
      pl.location_id, CAST(NULL AS BIGINT) AS provider_id, CAST(NULL AS BIGINT) AS care_site_id,
      person_source_value,
      CASE WHEN person_role = 'mother' THEN 'mother' ELSE gender_source_value END AS gender_source_value,
      0 AS gender_source_concept_id, race_source_value, 0 AS race_source_concept_id,
      CAST(NULL AS STRING) AS ethnicity_source_value, 0 AS ethnicity_source_concept_id,
      '{SITE}' AS site_id, person_role
    FROM {STG}.person sp
    LEFT JOIN {STG}._person_location pl ON pl.person_id = sp.person_id
    WHERE site_id = '{SITE}' AND birth_datetime IS NOT NULL
    """)
    show(f"""SELECT person_role, gender_concept_id, count(*) n FROM {PUB}.person GROUP BY 1, 2 ORDER BY 1, 2""")


build_person()

### 10b. Conditions, procedures, measurements and observations

Each `_all_facts` row goes to the table chosen in section 7, on the person and date chosen in section 9.
A fact recorded at a care contact links to that contact's visit only if the contact became a visit.
Type concept 32817 (EHR) throughout.

In [0]:
def routed_facts_sql(omop_table):
    """WITH r AS (...): the _all_facts rows for one OMOP table, with rp (person), ed (date), pid (pregnancy id)
    and vid (visit id, when the fact's care contact became a visit)."""
    return f"""
    WITH r AS (
      SELECT f.source_table, f.fact_natural_key, f.pregnancy_natural_key,
             f.concept_id, f.source_value, f.source_concept_id, f.value_as_number, f.unit_source_value,
             f.value_as_concept_id, f.value_as_string, f.value_source_value, f.unit_concept_id,
             f.qualifier_concept_id, f.qualifier_source_value, f.status_concept_id,
             {FACT_PERSON} AS rp, hv.visit_occurrence_id AS vid, {FACT_DATE} AS ed, pr.pregnancy_id AS pid
      FROM {STG}._all_facts f {FACT_ROUTING_JOINS}
      LEFT JOIN {STG}._msds_visit hv ON f.careconid IS NOT NULL AND hv.visit_occurrence_id = {mint_id('visit', 'f.careconid')}
      WHERE f.target_omop_table = '{omop_table}'
    )"""


# Cleaning already removed every fact without an eligible person or a date; this filter only makes sure.
ROUTED_FACT_KEEP = f"""WHERE r.rp IS NOT NULL AND r.ed IS NOT NULL
      AND EXISTS (SELECT 1 FROM {STG}._eligible el WHERE el.person_id = r.rp)"""


def fact_id(omop_table):
    """The id expression for a routed fact going to omop_table."""
    return mint_id(FACT_TABLE_ID_KIND[omop_table], "r.source_table", "r.fact_natural_key")


def build_fact_tables():
    """PUB condition_occurrence, procedure_occurrence, measurement and observation from STG._all_facts."""
    no_role = scalar(f"SELECT count_if(person_role IS NULL OR person_role NOT IN ('mother', 'infant')) FROM {STG}._all_facts")
    assert no_role == 0, f"{no_role} _all_facts rows without a valid person_role"

    spark.sql(f"""{routed_facts_sql('condition_occurrence')}
    INSERT INTO {PUB}.condition_occurrence (condition_occurrence_id, person_id, condition_concept_id,
      condition_start_date, condition_start_datetime, condition_end_date, condition_end_datetime,
      condition_type_concept_id, condition_status_concept_id, stop_reason, provider_id,
      visit_occurrence_id, visit_detail_id, condition_source_value, condition_source_concept_id,
      condition_status_source_value, site_id, pregnancy_id, source_table)
    SELECT {fact_id('condition_occurrence')}, r.rp, r.concept_id,
      r.ed, CAST(r.ed AS TIMESTAMP), CAST(NULL AS DATE), CAST(NULL AS TIMESTAMP),
      32817, r.status_concept_id, CAST(NULL AS STRING), CAST(NULL AS BIGINT),
      r.vid, CAST(NULL AS BIGINT), r.source_value, r.source_concept_id,
      CAST(NULL AS STRING), '{SITE}', r.pid, r.source_table
    FROM r {ROUTED_FACT_KEEP}""")

    spark.sql(f"""{routed_facts_sql('procedure_occurrence')}
    INSERT INTO {PUB}.procedure_occurrence (procedure_occurrence_id, person_id, procedure_concept_id,
      procedure_date, procedure_datetime, procedure_end_date, procedure_end_datetime,
      procedure_type_concept_id, modifier_concept_id, quantity, provider_id, visit_occurrence_id,
      visit_detail_id, procedure_source_value, procedure_source_concept_id, modifier_source_value,
      site_id, pregnancy_id, source_table)
    SELECT {fact_id('procedure_occurrence')}, r.rp, r.concept_id,
      r.ed, CAST(r.ed AS TIMESTAMP), CAST(NULL AS DATE), CAST(NULL AS TIMESTAMP),
      32817, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), r.vid,
      CAST(NULL AS BIGINT), r.source_value, r.source_concept_id, CAST(NULL AS STRING),
      '{SITE}', r.pid, r.source_table
    FROM r {ROUTED_FACT_KEEP}""")

    spark.sql(f"""{routed_facts_sql('measurement')}
    INSERT INTO {PUB}.measurement (measurement_id, person_id, measurement_concept_id, measurement_date,
      measurement_datetime, measurement_time, measurement_type_concept_id, operator_concept_id,
      value_as_number, value_as_concept_id, unit_concept_id, range_low, range_high, provider_id,
      visit_occurrence_id, visit_detail_id, measurement_source_value, measurement_source_concept_id,
      unit_source_value, unit_source_concept_id, value_source_value, measurement_event_id,
      meas_event_field_concept_id, site_id, pregnancy_id, enrichment_tag, source_table)
    SELECT {fact_id('measurement')}, r.rp, r.concept_id, r.ed,
      CAST(r.ed AS TIMESTAMP), CAST(NULL AS STRING), 32817, CAST(NULL AS BIGINT),
      r.value_as_number, r.value_as_concept_id, r.unit_concept_id,
      CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE), CAST(NULL AS BIGINT),
      r.vid, CAST(NULL AS BIGINT), r.source_value, r.source_concept_id,
      r.unit_source_value, CAST(NULL AS BIGINT), r.value_source_value, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      '{SITE}', r.pid, CAST(NULL AS STRING), r.source_table
    FROM r {ROUTED_FACT_KEEP}""")

    spark.sql(f"""{routed_facts_sql('observation')}
    INSERT INTO {PUB}.observation (observation_id, person_id, observation_concept_id, observation_date,
      observation_datetime, observation_type_concept_id, value_as_number, value_as_string,
      value_as_concept_id, qualifier_concept_id, unit_concept_id, provider_id, visit_occurrence_id,
      visit_detail_id, observation_source_value, observation_source_concept_id, unit_source_value,
      qualifier_source_value, value_source_value, observation_event_id, obs_event_field_concept_id,
      site_id, pregnancy_id, enrichment_tag, source_table)
    SELECT {fact_id('observation')}, r.rp, r.concept_id, r.ed,
      CAST(r.ed AS TIMESTAMP), 32817, r.value_as_number, r.value_as_string,
      r.value_as_concept_id, r.qualifier_concept_id, r.unit_concept_id, CAST(NULL AS BIGINT),
      r.vid, CAST(NULL AS BIGINT), r.source_value, r.source_concept_id, r.unit_source_value,
      r.qualifier_source_value, r.value_source_value, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      '{SITE}', r.pid, CAST(NULL AS STRING), r.source_table
    FROM r {ROUTED_FACT_KEEP}""")

    for table in FACT_TABLE_ID_KIND:
        print(table, spark.table(f"{PUB}.{table}").count())


build_fact_tables()

### 10c. Visits, visit details, deaths and locations

Copied from the staged tables for eligible people. A visit detail is copied only if its visit was; a
location only if a published person lives there.

In [0]:
def is_eligible(person_column):
    """SQL condition: person_column is an eligible person."""
    return f"EXISTS (SELECT 1 FROM {STG}._eligible el WHERE el.person_id = {person_column})"


def build_structural_tables():
    """PUB visit_occurrence, visit_detail, death and location from the staged section 8 tables."""
    spark.sql(f"""INSERT INTO {PUB}.visit_occurrence ({VISIT_COLUMNS})
      SELECT {VISIT_COLUMNS} FROM {STG}._msds_visit v WHERE {is_eligible('v.person_id')}""")

    detail_columns = ", ".join("d." + column for column in column_names("visit_detail"))
    spark.sql(f"""INSERT INTO {PUB}.visit_detail ({VISIT_DETAIL_COLUMNS})
      SELECT {detail_columns} FROM {STG}._msds_visit_detail d
      WHERE {is_eligible('d.person_id')}
        AND EXISTS (SELECT 1 FROM {PUB}.visit_occurrence vo
                    WHERE vo.site_id = '{SITE}' AND vo.visit_occurrence_id = d.visit_occurrence_id)""")

    death_columns = ", ".join(column_names("death"))
    spark.sql(f"""INSERT INTO {PUB}.death ({death_columns})
      SELECT {death_columns} FROM {STG}._msds_death d WHERE {is_eligible('d.person_id')}""")

    location_columns = ", ".join(column_names("location"))
    spark.sql(f"""INSERT INTO {PUB}.location ({location_columns})
      SELECT {location_columns} FROM {STG}._msds_location l
      WHERE EXISTS (SELECT 1 FROM {PUB}.person p WHERE p.site_id = '{SITE}' AND p.location_id = l.location_id)""")

    show(f"SELECT visit_concept_id, count(*) n FROM {PUB}.visit_occurrence GROUP BY 1 ORDER BY 1")
    print("visit details:", spark.table(f"{PUB}.visit_detail").count(),
          "| deaths:", spark.table(f"{PUB}.death").count(),
          "| locations:", spark.table(f"{PUB}.location").count())


build_structural_tables()

### 10d. Pregnancy episodes, pregnancy conditions, family links and ethnicity

Each eligible pregnancy gets an **anchor date**, the first that exists of:

1. its start (last menstrual period);
2. its end minus the gestational length;
3. its earliest dated published fact or visit;
4. its end.

With the anchor as start (and the pregnancy end, when not earlier, as end) it becomes:

- an OMOP **episode** (Pregnancy 32533), linked through `episode_event` to every published row of the
  pregnancy;
- a **condition** 'Pregnancy' (4299535, source_table `l3_pregnancy`).

A pregnancy with no possible anchor date is recorded as dropped (`undated_pregnancy_episode`) and kept
only in the pregnancy table.

**fact_relationship** links people: mother and child both ways, twins (same pregnancy), and siblings (same
mother, different pregnancies). Both people must be eligible.

**Ethnicity observation**: the NHS ethnic category as a concept (value_as_concept, source_table
`l3_ethnicity`), one per person with a recognised letter A to R, dated at the person's first published
activity (or first pregnancy, or birth).

In [0]:
PREGNANCY = 4299535              # SNOMED 'Pregnancy'
PREGNANCY_EPISODE = 32533        # OMOP episode 'Pregnancy'
PERSON_ID_FIELD = 1147026        # CDM field concept for person.person_id
MOTHER_OF, CHILD_OF, TWIN_OF, SIBLING_OF = 4248584, 4285883, 4013484, 4292398

# The CDM field concept of each table's id column, for episode_event.
EPISODE_EVENT_LINKS = [
    ("condition_occurrence", "condition_occurrence_id", 1147127),
    ("procedure_occurrence", "procedure_occurrence_id", 1147082),
    ("measurement", "measurement_id", 1147138),
    ("observation", "observation_id", 1147165),
    ("visit_occurrence", "visit_occurrence_id", 1147070),
    ("visit_detail", "visit_detail_id", 1147624),
]

# NHS ethnic category letter -> the matching OMOP ethnic category concept.
ETHNIC_CATEGORY_CONCEPT = """CASE upper(trim(race_source_value))
  WHEN 'A' THEN 700385  -- White British
  WHEN 'B' THEN 700386  -- White Irish
  WHEN 'C' THEN 700387  -- Other White
  WHEN 'D' THEN 700388  -- Mixed White and Black Caribbean
  WHEN 'E' THEN 700389  -- Mixed White and Black African
  WHEN 'F' THEN 700390  -- Mixed White and Asian
  WHEN 'G' THEN 700391  -- Other Mixed
  WHEN 'H' THEN 700362  -- Asian Indian
  WHEN 'J' THEN 700363  -- Asian Pakistani
  WHEN 'K' THEN 700364  -- Asian Bangladeshi
  WHEN 'L' THEN 700365  -- Other Asian
  WHEN 'M' THEN 700366  -- Black Caribbean
  WHEN 'N' THEN 700367  -- Black African
  WHEN 'P' THEN 700368  -- Other Black
  WHEN 'R' THEN 700369  -- Chinese
  ELSE NULL END"""


def stage_pregnancy_anchors():
    """STG._pregnancy_anchor: episode start and end for every eligible pregnancy (see above)."""
    event_dates = " UNION ALL ".join(
        f"SELECT pregnancy_id, {date_column} AS d FROM {PUB}.{table} WHERE site_id = '{SITE}'"
        for table, date_column in [
            ("condition_occurrence", "condition_start_date"), ("procedure_occurrence", "procedure_date"),
            ("measurement", "measurement_date"), ("observation", "observation_date"),
            ("visit_occurrence", "visit_start_date"), ("visit_occurrence", "visit_end_date"),
            ("visit_detail", "visit_detail_start_date"), ("visit_detail", "visit_detail_end_date")])
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._pregnancy_anchor AS
    WITH ev AS ({event_dates}),
    earliest AS (
      SELECT pregnancy_id, min(d) AS earliest_event
      FROM ev WHERE pregnancy_id IS NOT NULL AND d IS NOT NULL GROUP BY pregnancy_id
    ),
    anchored AS (
      SELECT pr.pregnancy_id, pr.person_id, pr.pregnancy_end_date,
        coalesce(
          pr.pregnancy_start_date,
          CASE WHEN pr.pregnancy_end_date IS NOT NULL AND pr.gestational_length_in_day BETWEEN 1 AND 400
               THEN date_sub(pr.pregnancy_end_date, pr.gestational_length_in_day) END,
          a.earliest_event,
          pr.pregnancy_end_date
        ) AS episode_start_date
      FROM {STG}.pregnancy pr
      LEFT JOIN earliest a ON a.pregnancy_id = pr.pregnancy_id
      WHERE pr.site_id = '{SITE}' AND {is_eligible('pr.person_id')}
    )
    SELECT pregnancy_id, person_id, episode_start_date,
      CASE WHEN pregnancy_end_date >= episode_start_date THEN pregnancy_end_date END AS episode_end_date
    FROM anchored
    """)
    record_dropped(f"""SELECT 'L3-EPISODE', CAST(pregnancy_id AS STRING), 'undated_pregnancy_episode'
                       FROM {STG}._pregnancy_anchor WHERE episode_start_date IS NULL""")


def build_pregnancy_conditions():
    """PUB.condition_occurrence 'Pregnancy' rows, one per anchored pregnancy."""
    spark.sql(f"""
    INSERT INTO {PUB}.condition_occurrence (condition_occurrence_id, person_id, condition_concept_id,
      condition_start_date, condition_start_datetime, condition_end_date, condition_end_datetime,
      condition_type_concept_id, condition_status_concept_id, stop_reason, provider_id,
      visit_occurrence_id, visit_detail_id, condition_source_value, condition_source_concept_id,
      condition_status_source_value, site_id, pregnancy_id, source_table)
    SELECT {mint_id('pregnancy-condition', 'a.pregnancy_id')}, a.person_id, {PREGNANCY},
      a.episode_start_date, CAST(a.episode_start_date AS TIMESTAMP),
      a.episode_end_date, CAST(a.episode_end_date AS TIMESTAMP),
      32817, CAST(NULL AS BIGINT), CAST(NULL AS STRING),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      'MSDS pregnancy', 0, CAST(NULL AS STRING),
      '{SITE}', a.pregnancy_id, 'l3_pregnancy'
    FROM {STG}._pregnancy_anchor a
    WHERE a.episode_start_date IS NOT NULL
    """)


def build_episodes():
    """PUB.episode, one Pregnancy episode per anchored pregnancy."""
    spark.sql(f"""
    INSERT INTO {PUB}.episode (episode_id, person_id, episode_concept_id, episode_start_date,
      episode_start_datetime, episode_end_date, episode_end_datetime, episode_parent_id,
      episode_number, episode_object_concept_id, episode_type_concept_id, episode_source_value,
      episode_source_concept_id, pregnancy_id, site_id)
    SELECT {mint_id('episode', 'a.pregnancy_id')}, a.person_id, {PREGNANCY_EPISODE}, a.episode_start_date,
      CAST(a.episode_start_date AS TIMESTAMP), a.episode_end_date,
      CAST(a.episode_end_date AS TIMESTAMP), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      {PREGNANCY}, 32817, 'MSDS pregnancy', 0,
      a.pregnancy_id, '{SITE}'
    FROM {STG}._pregnancy_anchor a
    WHERE a.episode_start_date IS NOT NULL
    """)
    assert_unique(f"{PUB}.episode", "episode_id")


def build_episode_events():
    """PUB.episode_event: every published row that has a pregnancy_id, linked to that pregnancy's episode."""
    for table, id_column, field_concept in EPISODE_EVENT_LINKS:
        spark.sql(f"""
        INSERT INTO {PUB}.episode_event (episode_id, event_id, episode_event_field_concept_id, source_table, site_id)
        SELECT e.episode_id, f.{id_column}, {field_concept}, '{table}', '{SITE}'
        FROM {PUB}.{table} f
        JOIN {PUB}.episode e ON e.pregnancy_id = f.pregnancy_id AND e.site_id = f.site_id
        WHERE f.site_id = '{SITE}' AND f.pregnancy_id IS NOT NULL
        """)


def build_family_links():
    """PUB.fact_relationship: mother/child (both ways), twins and siblings, between eligible people."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._mother_child AS
    SELECT pr.person_id AS mother_id, cast(i.infant_id AS bigint) AS child_id, i.pregnancy_id
    FROM {STG}.infant i
    JOIN {STG}.pregnancy pr ON pr.pregnancy_id = i.pregnancy_id
    WHERE i.pregnancy_id IS NOT NULL AND i.site_id = '{SITE}'
    """)
    spark.sql(f"""
    INSERT INTO {PUB}.fact_relationship (domain_concept_id_1, fact_id_1, domain_concept_id_2, fact_id_2,
                                         relationship_concept_id, site_id)
    WITH links AS (
      -- mother of, and child of
      SELECT mother_id AS a, child_id AS b, {MOTHER_OF} AS rel FROM {STG}._mother_child
      UNION ALL
      SELECT child_id, mother_id, {CHILD_OF} FROM {STG}._mother_child
      UNION ALL
      -- twins: two babies of one pregnancy
      SELECT x.child_id, y.child_id, {TWIN_OF}
      FROM {STG}._mother_child x JOIN {STG}._mother_child y
        ON x.pregnancy_id = y.pregnancy_id AND x.child_id <> y.child_id
      UNION ALL
      -- siblings: babies of one mother from different pregnancies
      SELECT x.child_id, y.child_id, {SIBLING_OF}
      FROM {STG}._mother_child x JOIN {STG}._mother_child y
        ON x.mother_id = y.mother_id AND x.pregnancy_id <> y.pregnancy_id AND x.child_id <> y.child_id
    )
    SELECT {PERSON_ID_FIELD}, a, {PERSON_ID_FIELD}, b, rel, '{SITE}'
    FROM links
    WHERE {is_eligible('a')} AND {is_eligible('b')}
    """)


def build_ethnicity_observations():
    """PUB.observation rows carrying each person's NHS ethnic category (source_table l3_ethnicity)."""
    activity_dates = " UNION ALL ".join(
        f"SELECT person_id, {date_column} AS d FROM {PUB}.{table} WHERE site_id = '{SITE}'{extra}"
        for table, date_column, extra in [
            ("condition_occurrence", "condition_start_date", ""),
            ("measurement", "measurement_date", ""),
            ("observation", "observation_date", " AND (source_table <> 'l3_ethnicity' OR source_table IS NULL)"),
            ("procedure_occurrence", "procedure_date", ""),
            ("visit_occurrence", "visit_start_date", "")])
    observation_date = "coalesce(cl.earliest_clin, pf.ps, pf.pe, CAST(p.birth_datetime AS DATE))"
    spark.sql(f"""
    INSERT INTO {PUB}.observation (observation_id, person_id, observation_concept_id, observation_date,
      observation_datetime, observation_type_concept_id, value_as_number, value_as_string,
      value_as_concept_id, qualifier_concept_id, unit_concept_id, provider_id, visit_occurrence_id,
      visit_detail_id, observation_source_value, observation_source_concept_id, unit_source_value,
      qualifier_source_value, value_source_value, observation_event_id, obs_event_field_concept_id,
      site_id, pregnancy_id, enrichment_tag, source_table)
    WITH ethnicity AS (
      SELECT person_id, {ETHNIC_CATEGORY_CONCEPT} AS value_concept, upper(trim(race_source_value)) AS letter
      FROM {STG}.person
      WHERE site_id = '{SITE}' AND {ETHNIC_CATEGORY_CONCEPT} IS NOT NULL
    ),
    pf AS (
      SELECT person_id, min(pregnancy_start_date) AS ps, min(pregnancy_end_date) AS pe
      FROM {STG}.pregnancy WHERE site_id = '{SITE}' GROUP BY person_id
    ),
    clin AS (
      SELECT person_id, min(d) AS earliest_clin FROM ({activity_dates}) WHERE d IS NOT NULL GROUP BY person_id
    )
    SELECT {mint_id('l3-ethnicity-obs', 'o.person_id')}, o.person_id, 0,
      {observation_date}, CAST({observation_date} AS TIMESTAMP),
      32817, CAST(NULL AS DOUBLE), CAST(NULL AS STRING),
      o.value_concept, CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), CAST(NULL AS BIGINT),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), o.letter,
      CAST(NULL AS BIGINT), CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS STRING),
      CAST(NULL AS BIGINT), CAST(NULL AS BIGINT), '{SITE}', CAST(NULL AS BIGINT),
      CAST(NULL AS STRING), 'l3_ethnicity'
    FROM ethnicity o
    JOIN {PUB}.person p ON p.person_id = o.person_id AND p.site_id = '{SITE}'
    LEFT JOIN pf ON pf.person_id = o.person_id
    LEFT JOIN clin cl ON cl.person_id = o.person_id
    WHERE {is_eligible('o.person_id')}
    """)
    undated = scalar(f"""SELECT count(*) FROM {PUB}.observation
                         WHERE source_table = 'l3_ethnicity' AND observation_date IS NULL""")
    assert undated == 0, f"{undated} ethnicity observations without a date"


stage_pregnancy_anchors()
build_pregnancy_conditions()
build_episodes()
build_episode_events()
build_family_links()
build_ethnicity_observations()
show(f"""SELECT 'pregnancy conditions' what, count(*) n FROM {PUB}.condition_occurrence WHERE source_table = 'l3_pregnancy'
         UNION ALL SELECT 'episodes', count(*) FROM {PUB}.episode
         UNION ALL SELECT 'episode events', count(*) FROM {PUB}.episode_event
         UNION ALL SELECT 'family links', count(*) FROM {PUB}.fact_relationship
         UNION ALL SELECT 'ethnicity observations', count(*) FROM {PUB}.observation WHERE source_table = 'l3_ethnicity'""")

### 10e. Pregnancy and infant tables, and the concept map

The cleaned staged pregnancy and infant rows for eligible people. The `*_source` columns are filled only by
the Barts extension, so they are NULL here. `_msds_concept_map` publishes how every spec question, answer
and unit was mapped (section 5).

In [0]:
def build_extension_tables():
    """PUB.pregnancy, PUB.infant and PUB._msds_concept_map."""
    pregnancy_columns = [c for c in column_names("pregnancy")
                         if c not in ("pre_pregnancy_bmi_source", "pregnancy_marital_status_source_value")]
    spark.sql(f"""
    INSERT INTO {PUB}.pregnancy ({", ".join(pregnancy_columns)}, pre_pregnancy_bmi_source, pregnancy_marital_status_source_value)
    SELECT {", ".join("s." + c for c in pregnancy_columns)}, CAST(NULL AS STRING), CAST(NULL AS STRING)
    FROM {STG}.pregnancy s
    WHERE s.site_id = '{SITE}' AND {is_eligible('s.person_id')}""")

    infant_columns = [c for c in column_names("infant") if c not in ("birth_weight_source", "birth_apgar_source")]
    spark.sql(f"""
    INSERT INTO {PUB}.infant ({", ".join(infant_columns)}, birth_weight_source, birth_apgar_source)
    SELECT {", ".join("s." + c for c in infant_columns)}, CAST(NULL AS STRING), CAST(NULL AS STRING)
    FROM {STG}.infant s
    WHERE s.site_id = '{SITE}' AND {is_eligible('s.infant_id')}""")

    spark.sql(f"""
    INSERT INTO {PUB}._msds_concept_map
    SELECT '{SITE}', rule_id, role, src_code, src_text, concept_id, domain_id, method,
           model_version, cosine, margin, rule_ref
    FROM {STG}._spec_resolved""")
    print("pregnancies:", spark.table(f"{PUB}.pregnancy").count(), "| infants:", spark.table(f"{PUB}.infant").count())


build_extension_tables()

## 11. Observation periods, CDM source and the bundled vocabulary

### 11a. Observation periods

One period per person, from their earliest to their latest published date (facts, visits and pregnancy
episodes). A person with no dated rows gets their pregnancy dates, or else their birth date.

In [0]:
# The dated columns that set a person's observation period: (table, date column).
OBSERVATION_PERIOD_DATES = [
    ("condition_occurrence", "condition_start_date"), ("measurement", "measurement_date"),
    ("observation", "observation_date"), ("procedure_occurrence", "procedure_date"),
    ("visit_occurrence", "visit_start_date"), ("visit_occurrence", "visit_end_date"),
    ("episode", "episode_start_date"), ("episode", "episode_end_date"),
]


def build_observation_periods(schema):
    """schema.observation_period for this site, from the published rows in schema (OBSERVATION_PERIOD_DATES)."""
    dated_rows = " UNION ALL ".join(
        f"SELECT person_id, {date_column} AS d FROM {schema}.{table} WHERE site_id = '{SITE}'"
        for table, date_column in OBSERVATION_PERIOD_DATES)
    spark.sql(f"DELETE FROM {schema}.observation_period WHERE site_id = '{SITE}'")
    spark.sql(f"""
    INSERT INTO {schema}.observation_period (observation_period_id, person_id,
      observation_period_start_date, observation_period_end_date, period_type_concept_id, site_id)
    WITH ev AS ({dated_rows}),
    span AS (
      SELECT person_id, min(d) AS first_date, max(d) AS last_date FROM ev WHERE d IS NOT NULL GROUP BY person_id
    ),
    pregnancies AS (
      SELECT person_id, min(pregnancy_start_date) AS ps,
             max(coalesce(pregnancy_end_date, pregnancy_start_date)) AS pe
      FROM {schema}.pregnancy WHERE site_id = '{SITE}' GROUP BY person_id
    )
    SELECT {mint_id('obs-period', 'p.person_id')}, p.person_id,
      coalesce(s.first_date, pf.ps, CAST(p.birth_datetime AS DATE)),
      coalesce(s.last_date, pf.pe, CAST(p.birth_datetime AS DATE)),
      32817, '{SITE}'
    FROM {schema}.person p
    LEFT JOIN span s ON s.person_id = p.person_id
    LEFT JOIN pregnancies pf ON pf.person_id = p.person_id
    WHERE p.site_id = '{SITE}'
    """)
    periods, people = spark.sql(f"""SELECT
        (SELECT count(*) FROM {schema}.observation_period WHERE site_id = '{SITE}'),
        (SELECT count(*) FROM {schema}.person WHERE site_id = '{SITE}')""").first()
    assert periods == people, f"{periods} observation periods for {people} people"
    print("observation periods:", periods)


build_observation_periods(PUB)

### 11b. CDM source

The single `cdm_source` row. Its dates are the release date, never today's date, so a rebuild of the same
release is identical.

In [0]:
def write_cdm_source(schema):
    """Replace schema.cdm_source with the PREBORN row for RELEASE_DATE."""
    spark.sql(f"""
      INSERT INTO {schema}.cdm_source BY NAME
      REPLACE WHERE true
      SELECT
        'PREBORN multi-site maternity OMOP subset' AS cdm_source_name,
        'PREBORN' AS cdm_source_abbreviation,
        'PREBORN participating NHS organisations' AS cdm_holder,
        'Multi-site maternity-focused OMOP CDM v5.4 subset: MSDS common core with optional site-specific enrichment'
          AS source_description,
        CAST(NULL AS STRING) AS source_documentation_reference,
        CAST(NULL AS STRING) AS cdm_etl_reference,
        DATE'{RELEASE_DATE}' AS source_release_date, DATE'{RELEASE_DATE}' AS cdm_release_date,
        'v5.4' AS cdm_version, 756265 AS cdm_version_concept_id,
        (SELECT vocabulary_version FROM {VOCAB}.vocabulary WHERE vocabulary_id = 'None') AS vocabulary_version
    """)


write_cdm_source(PUB)

### 11c. The bundled vocabulary

Each PREBORN schema carries the part of the OMOP vocabulary it needs, so it can be used on its own:

- every concept used anywhere in the schema (all sites), plus all of their ancestors;
- the concepts the vocabulary metadata tables point at, and the language concepts of synonyms;
- relationships between two included concepts, ancestry of the used concepts, and their synonyms;
- full copies of `vocabulary`, `domain`, `concept_class` and `relationship`.

The build then checks that every vocabulary reference resolves inside the bundle (no dangling links).

In [0]:
# Every concept-carrying column, per table.
CONCEPT_COLUMNS = {
    "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
               "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
    "observation_period": ["period_type_concept_id"],
    "condition_occurrence": ["condition_concept_id", "condition_type_concept_id",
                             "condition_status_concept_id", "condition_source_concept_id"],
    "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id",
                             "modifier_concept_id", "procedure_source_concept_id"],
    "measurement": ["measurement_concept_id", "measurement_type_concept_id", "operator_concept_id",
                    "value_as_concept_id", "unit_concept_id", "measurement_source_concept_id",
                    "unit_source_concept_id", "meas_event_field_concept_id"],
    "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                    "qualifier_concept_id", "unit_concept_id", "observation_source_concept_id",
                    "obs_event_field_concept_id"],
    "visit_occurrence": ["visit_concept_id", "visit_type_concept_id", "visit_source_concept_id",
                         "admitted_from_concept_id", "discharged_to_concept_id"],
    "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id",
                     "visit_detail_source_concept_id", "admitted_from_concept_id",
                     "discharged_to_concept_id"],
    "death": ["death_type_concept_id", "cause_concept_id", "cause_source_concept_id"],
    "location": ["country_concept_id"],
    "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id",
                "episode_source_concept_id"],
    "episode_event": ["episode_event_field_concept_id"],
    "fact_relationship": ["domain_concept_id_1", "domain_concept_id_2", "relationship_concept_id"],
    "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                  "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                  "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
    "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
    "cdm_source": ["cdm_version_concept_id"],
    "_msds_concept_map": ["concept_id"],
}


def missing_reference_checks(schema):
    """(name, SQL counting dangling references) for every vocabulary link inside schema's bundle."""
    def missing(child, column, parent, parent_column, positive_only=False):
        only_positive = f" AND x.{column} > 0" if positive_only else ""
        return (f"(SELECT count(*) FROM {schema}.{child} x LEFT JOIN {schema}.{parent} y "
                f"ON y.{parent_column} = x.{column} WHERE y.{parent_column} IS NULL{only_positive})")
    return [
        ("concept_relationship.concept_id_1 in concept", missing("concept_relationship", "concept_id_1", "concept", "concept_id")),
        ("concept_relationship.concept_id_2 in concept", missing("concept_relationship", "concept_id_2", "concept", "concept_id")),
        ("concept_ancestor.ancestor in concept", missing("concept_ancestor", "ancestor_concept_id", "concept", "concept_id")),
        ("concept_ancestor.descendant in concept", missing("concept_ancestor", "descendant_concept_id", "concept", "concept_id")),
        ("vocabulary.vocabulary_concept_id in concept", missing("vocabulary", "vocabulary_concept_id", "concept", "concept_id", True)),
        ("domain.domain_concept_id in concept", missing("domain", "domain_concept_id", "concept", "concept_id", True)),
        ("concept_class.concept_class_concept_id in concept", missing("concept_class", "concept_class_concept_id", "concept", "concept_id", True)),
        ("relationship.relationship_concept_id in concept", missing("relationship", "relationship_concept_id", "concept", "concept_id", True)),
        ("concept_synonym.concept_id in concept", missing("concept_synonym", "concept_id", "concept", "concept_id")),
        ("concept_synonym.language_concept_id in concept", missing("concept_synonym", "language_concept_id", "concept", "concept_id", True)),
        ("cdm_source.cdm_version_concept_id in concept", missing("cdm_source", "cdm_version_concept_id", "concept", "concept_id", True)),
        ("concept.domain_id in domain", missing("concept", "domain_id", "domain", "domain_id")),
        ("concept.vocabulary_id in vocabulary", missing("concept", "vocabulary_id", "vocabulary", "vocabulary_id")),
        ("concept.concept_class_id in concept_class", missing("concept", "concept_class_id", "concept_class", "concept_class_id")),
    ]


def build_bundled_vocabulary(schema):
    """Rebuild schema's vocabulary tables from VOCAB for every concept the schema uses (see above)."""
    # 1. The concepts the schema uses.
    used = " UNION ALL ".join(f"SELECT CAST({column} AS BIGINT) AS cid FROM {schema}.{table}"
                              for table, columns in CONCEPT_COLUMNS.items() for column in columns)
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_used AS
      SELECT DISTINCT cid FROM ({used}) WHERE cid IS NOT NULL AND cid > 0""")

    # 2. Plus their ancestors and the concepts the metadata tables point at.
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_ids AS
      SELECT cid FROM {schema}._vocab_used
      UNION
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) FROM {VOCAB}.concept_ancestor ca
        JOIN {schema}._vocab_used u ON ca.descendant_concept_id = u.cid
      UNION
      SELECT CAST(vocabulary_concept_id AS BIGINT) FROM {VOCAB}.vocabulary WHERE vocabulary_concept_id > 0
      UNION
      SELECT CAST(domain_concept_id AS BIGINT) FROM {VOCAB}.domain WHERE domain_concept_id > 0
      UNION
      SELECT CAST(concept_class_concept_id AS BIGINT) FROM {VOCAB}.concept_class WHERE concept_class_concept_id > 0
      UNION
      SELECT CAST(relationship_concept_id AS BIGINT) FROM {VOCAB}.relationship WHERE relationship_concept_id > 0""")

    # 3. Plus the language concepts of the included concepts' synonyms, repeated until nothing new is added.
    for _ in range(5):
        before = spark.table(f"{schema}._vocab_ids").count()
        spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_ids_next AS
          SELECT cid FROM {schema}._vocab_ids
          UNION
          SELECT CAST(cs.language_concept_id AS BIGINT)
          FROM {VOCAB}.concept_synonym cs JOIN {schema}._vocab_ids v ON cs.concept_id = v.cid
          WHERE cs.language_concept_id IS NOT NULL AND cs.language_concept_id > 0""")
        spark.sql(f"CREATE OR REPLACE TABLE {schema}._vocab_ids AS SELECT cid FROM {schema}._vocab_ids_next")
        if spark.table(f"{schema}._vocab_ids").count() == before:
            break
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_ids_next")

    # 4. Write the vocabulary tables.
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept AS
      SELECT CAST(c.concept_id AS BIGINT) AS concept_id, c.concept_name, c.domain_id, c.vocabulary_id,
             c.concept_class_id, c.standard_concept, c.concept_code,
             c.valid_start_date, c.valid_end_date, c.invalid_reason
      FROM {VOCAB}.concept c JOIN {schema}._vocab_ids v ON c.concept_id = v.cid""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_relationship AS
      SELECT CAST(cr.concept_id_1 AS BIGINT) AS concept_id_1, CAST(cr.concept_id_2 AS BIGINT) AS concept_id_2,
             cr.relationship_id, cr.valid_start_date, cr.valid_end_date, cr.invalid_reason
      FROM {VOCAB}.concept_relationship cr
      WHERE cr.concept_id_1 IN (SELECT concept_id FROM {schema}.concept)
        AND cr.concept_id_2 IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_ancestor AS
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) AS ancestor_concept_id,
             CAST(ca.descendant_concept_id AS BIGINT) AS descendant_concept_id,
             CAST(ca.min_levels_of_separation AS BIGINT) AS min_levels_of_separation,
             CAST(ca.max_levels_of_separation AS BIGINT) AS max_levels_of_separation
      FROM {VOCAB}.concept_ancestor ca
      WHERE ca.descendant_concept_id IN (SELECT cid FROM {schema}._vocab_used)
        AND ca.ancestor_concept_id IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_synonym AS
      SELECT CAST(cs.concept_id AS BIGINT) AS concept_id, cs.concept_synonym_name,
             CAST(cs.language_concept_id AS BIGINT) AS language_concept_id
      FROM {VOCAB}.concept_synonym cs
      WHERE cs.concept_id IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.vocabulary AS
      SELECT vocabulary_id, vocabulary_name, vocabulary_reference, vocabulary_version,
             CAST(vocabulary_concept_id AS BIGINT) AS vocabulary_concept_id
      FROM {VOCAB}.vocabulary""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.domain AS
      SELECT domain_id, domain_name, CAST(domain_concept_id AS BIGINT) AS domain_concept_id FROM {VOCAB}.domain""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_class AS
      SELECT concept_class_id, concept_class_name, CAST(concept_class_concept_id AS BIGINT) AS concept_class_concept_id
      FROM {VOCAB}.concept_class""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.relationship AS
      SELECT relationship_id, relationship_name, is_hierarchical, defines_ancestry,
             reverse_relationship_id, CAST(relationship_concept_id AS BIGINT) AS relationship_concept_id
      FROM {VOCAB}.relationship""")

    # 5. Check that nothing dangles.
    used_count = spark.table(f"{schema}._vocab_used").count()
    concept_count = spark.table(f"{schema}.concept").count()
    print(f"bundled vocabulary for {schema}: {used_count} concepts used, {concept_count} concepts bundled")
    dangling = 0
    for name, count_sql in missing_reference_checks(schema):
        n = scalar(f"SELECT {count_sql}")
        dangling += n
        print(f"  {'OK  ' if n == 0 else 'FAIL'} {name}: {n} dangling")
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_used")
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_ids")
    assert dangling == 0, f"the bundled vocabulary in {schema} has {dangling} dangling references"


build_bundled_vocabulary(PUB)

## 12. Quality checks

The checks only read. They run twice: on the staged `PUB` schema here, and on the target straight after
publishing (section 13), where a failure rolls the publish back. Each result is appended to
`{target}._qa_results` (which is never rolled back, so the evidence of a failed publish survives).

- **hard** checks stop the run when they fail;
- **report** checks are logged for information.

| Group | Checks |
|---|---|
| structure | ids unique (across all sites); every link points at an existing row; required columns filled; every person has an observation period; every concept used exists in the bundled vocabulary; the vocabulary has no dangling references |
| visits | no visit from a contact that was not held; visits and details end on or after their start; details sit within their visit |
| reconciliation | per rule, every cleaned fact is published or dropped (never both, never neither), on a person of the right role, and nothing is published that was not staged |

In [0]:
QA_RESULTS_COLUMNS = """site_id STRING, run_tag STRING, source_release DATE, scope STRING, check_name STRING,
  severity STRING, ok BOOLEAN, metric STRING, checked_at TIMESTAMP"""

# Per table: (primary key, the references it holds, the columns that must be filled for this site).
TABLE_RULES = {
    "person": ("person_id", [], ["gender_concept_id", "year_of_birth"]),
    "observation_period": ("observation_period_id", ["person"], ["period_type_concept_id"]),
    "condition_occurrence": ("condition_occurrence_id", ["person", "pregnancy", "visit"],
                             ["condition_start_date", "condition_type_concept_id"]),
    "procedure_occurrence": ("procedure_occurrence_id", ["person", "pregnancy", "visit"],
                             ["procedure_date", "procedure_type_concept_id"]),
    "measurement": ("measurement_id", ["person", "pregnancy", "visit"],
                    ["measurement_date", "measurement_type_concept_id"]),
    "observation": ("observation_id", ["person", "pregnancy", "visit"],
                    ["observation_date", "observation_type_concept_id"]),
    "visit_occurrence": ("visit_occurrence_id", ["person", "pregnancy"],
                         ["person_id", "visit_concept_id", "visit_start_date", "visit_end_date",
                          "visit_type_concept_id"]),
    "visit_detail": ("visit_detail_id", ["person", "pregnancy", "visit"],
                     ["visit_occurrence_id", "visit_detail_start_date", "visit_detail_type_concept_id"]),
    "episode": ("episode_id", ["person", "pregnancy"], ["episode_type_concept_id"]),
    "death": ("person_id", ["person"], ["death_date", "death_type_concept_id"]),
}
# A reference: (column holding it, the table it points at, that table's key).
REFERENCES = {
    "person": ("person_id", "person", "person_id"),
    "pregnancy": ("pregnancy_id", "pregnancy", "pregnancy_id"),
    "visit": ("visit_occurrence_id", "visit_occurrence", "visit_occurrence_id"),
}
# The concept columns whose values must exist in the bundled concept table.
QA_CONCEPT_COLUMNS = {
    "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
               "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
    "observation_period": ["period_type_concept_id"],
    "condition_occurrence": ["condition_concept_id", "condition_type_concept_id", "condition_source_concept_id"],
    "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id", "procedure_source_concept_id"],
    "measurement": ["measurement_concept_id", "measurement_type_concept_id", "value_as_concept_id",
                    "unit_concept_id", "measurement_source_concept_id"],
    "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                    "unit_concept_id", "observation_source_concept_id"],
    "visit_occurrence": ["visit_concept_id", "visit_type_concept_id"],
    "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id"],
    "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id"],
    "death": ["death_type_concept_id", "cause_concept_id"],
    "location": ["country_concept_id"],
    "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                  "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                  "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
    "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
    "_msds_concept_map": ["concept_id"],
}

In [0]:
def qa_structure(schema):
    """Keys, references, required columns, observation-period coverage, concepts and vocabulary links.
    Returns a list of (check name, severity, passed, metric). One query per table keeps this quick."""
    results = []
    for table, (key, references, required) in TABLE_RULES.items():
        joins = []
        measures = ["count(*) AS n", f"count(DISTINCT f.{key}) AS d"]
        for ref in references:
            column, ref_table, ref_key = REFERENCES[ref]
            joins.append(f"LEFT JOIN (SELECT DISTINCT {ref_key} FROM {schema}.{ref_table}) j_{ref} "
                         f"ON j_{ref}.{ref_key} = f.{column}")
            measures.append(f"count_if(f.{column} IS NOT NULL AND j_{ref}.{ref_key} IS NULL) AS o_{ref}")
        for column in required:
            measures.append(f"count_if(f.site_id = '{SITE}' AND f.{column} IS NULL) AS nn_{column}")
        if table == "person":
            joins.append(f"LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.observation_period) op ON op.person_id = f.person_id")
            joins.append(f"LEFT JOIN (SELECT DISTINCT location_id FROM {schema}.location) lo ON lo.location_id = f.location_id")
            measures.append(f"count_if(f.site_id = '{SITE}' AND op.person_id IS NULL) AS uncovered")
            measures.append("count_if(f.location_id IS NOT NULL AND lo.location_id IS NULL) AS o_location")
        row = spark.sql(f"SELECT {', '.join(measures)} FROM {schema}.{table} f {' '.join(joins)}").first().asDict()

        results.append((f"{table}.{key} unique (global)", "hard", row["n"] == row["d"], f"{row['n']}/{row['d']}"))
        for ref in references:
            ref_table = REFERENCES[ref][1]
            results.append((f"{table} -> {ref_table}", "hard", row[f"o_{ref}"] == 0, f"{row[f'o_{ref}']} orphans"))
        if table == "person":
            results.append(("person -> location", "hard", row["o_location"] == 0, f"{row['o_location']} orphans"))
        for column in required:
            results.append((f"{table}.{column} not-null", "hard", row[f"nn_{column}"] == 0, f"{row[f'nn_{column}']} null"))
        if table == "person":
            results.append(("observation_period covers every person", "hard", row["uncovered"] == 0,
                            f"{row['uncovered']} uncovered"))

    row = spark.sql(f"""SELECT
      (SELECT count(*) FROM {schema}.episode_event ee
         LEFT JOIN (SELECT DISTINCT episode_id FROM {schema}.episode) e ON e.episode_id = ee.episode_id
         WHERE e.episode_id IS NULL) AS ee_orphan,
      (SELECT count(*) FROM {schema}.fact_relationship fr
         LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.person) p1 ON p1.person_id = fr.fact_id_1
         LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.person) p2 ON p2.person_id = fr.fact_id_2
         WHERE p1.person_id IS NULL OR p2.person_id IS NULL) AS fr_orphan,
      (SELECT count_if(site_id = '{SITE}') - count(DISTINCT CASE WHEN site_id = '{SITE}' THEN location_id END)
         FROM {schema}.location) AS loc_dup_site,
      (SELECT count(*) - count(DISTINCT location_id) FROM {schema}.location) AS loc_dup_global,
      (SELECT count(*) FROM {schema}._msds_concept_map WHERE site_id = '{SITE}') AS concept_map_rows""").first()
    results += [
        ("episode_event -> episode", "hard", row.ee_orphan == 0, f"{row.ee_orphan} orphans"),
        ("fact_relationship endpoints -> person", "hard", row.fr_orphan == 0, f"{row.fr_orphan} orphans"),
        ("location.location_id unique (site)", "hard", row.loc_dup_site == 0, f"{row.loc_dup_site} duplicates"),
        # LSOA ids are shared codes, so two sites can rightly publish the same LSOA row
        ("location.location_id unique (global)", "report", row.loc_dup_global == 0, f"{row.loc_dup_global} duplicates"),
        ("_msds_concept_map populated", "report", row.concept_map_rows > 0, f"{row.concept_map_rows} rows"),
    ]

    used = " UNION ALL ".join(f"SELECT CAST({column} AS BIGINT) AS cid FROM {schema}.{table} WHERE site_id = '{SITE}'"
                              for table, columns in QA_CONCEPT_COLUMNS.items() for column in columns)
    missing = scalar(f"""SELECT count(*) FROM (SELECT DISTINCT cid FROM ({used}) WHERE cid > 0) u
                         LEFT JOIN {schema}.concept c ON c.concept_id = u.cid WHERE c.concept_id IS NULL""")
    results.append(("all used non-zero concept ids exist in concept", "hard", missing == 0, f"{missing} missing"))

    checks = missing_reference_checks(schema)
    counts = spark.sql("SELECT " + ", ".join(f"{sql} AS c{i}" for i, (_, sql) in enumerate(checks))).first()
    for i, (name, _) in enumerate(checks):
        results.append((name, "hard", counts[i] == 0, f"{counts[i]} dangling"))
    return results


def qa_visits(schema):
    """No visit from a contact that was not held; visit and detail dates in order; details inside their visit."""
    row = spark.sql(f"""SELECT
      (SELECT count(*) FROM {schema}.visit_occurrence vo JOIN {STG}._carecontact cc
         ON vo.visit_occurrence_id = {mint_id('visit', 'cc.CARECONID')}
         WHERE vo.site_id = '{SITE}' AND NOT cc.held) AS not_held,
      (SELECT count_if(visit_end_date < visit_start_date) FROM {schema}.visit_occurrence
         WHERE site_id = '{SITE}') AS vo_neg,
      (SELECT count_if(visit_detail_end_date < visit_detail_start_date) FROM {schema}.visit_detail
         WHERE site_id = '{SITE}') AS vd_neg,
      (SELECT count(*) FROM {schema}.visit_detail vd JOIN {schema}.visit_occurrence vo
         ON vo.visit_occurrence_id = vd.visit_occurrence_id
         WHERE vd.site_id = '{SITE}' AND (vd.visit_detail_start_date < date_sub(vo.visit_start_date, 1)
           OR coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date) > date_add(vo.visit_end_date, 1))
      ) AS vd_outside""").first()
    return [
        ("no visit minted from a non-held contact", "hard", row.not_held == 0, f"{row.not_held} visits"),
        ("visit_occurrence end >= start", "hard", row.vo_neg == 0, f"{row.vo_neg} rows"),
        ("visit_detail end >= start", "hard", row.vd_neg == 0, f"{row.vd_neg} rows"),
        ("visit_detail within its visit (+/-1 day)", "report", row.vd_outside == 0, f"{row.vd_outside} rows"),
    ]


def qa_reconcile(schema):
    """Per rule: every fact in the cleaned snapshot of _all_facts is published XOR dropped, on a person of its
    role; and no published fact is missing from the snapshot. Pregnancy conditions and ethnicity rows
    (source_table l3_*) are built in section 10, not from _all_facts, so they are left out."""
    state = spark.table(f"{STG}._clean_state").first()
    fact_id_by_table = "CASE f.target_omop_table " + " ".join(
        f"WHEN '{table}' THEN {mint_id(kind, 'f.source_table', 'f.fact_natural_key')}"
        for table, kind in FACT_TABLE_ID_KIND.items()) + " END"
    published = " UNION ALL ".join(
        f"SELECT '{table}' AS t, {table}_id AS id, person_id FROM {schema}.{table} "
        f"WHERE site_id = '{SITE}' AND left(coalesce(source_table, ''), 3) <> 'l3_'"
        + (" AND enrichment_tag IS NULL" if table in ("measurement", "observation") else "")
        for table in FACT_TABLE_ID_KIND)
    spark.sql(f"""CREATE OR REPLACE TEMP VIEW _qa_recon AS
    WITH snapshot AS (
      SELECT coalesce(f.rule_id, concat('SRC-', f.source_table)) AS rule_id, f.fact_natural_key,
             f.target_omop_table AS t, f.person_role, {fact_id_by_table} AS fid
      FROM {STG}._all_facts VERSION AS OF {state.all_facts_version} f),
    published AS ({published}),
    dropped AS (SELECT DISTINCT rule_id, fact_natural_key FROM {STG}._dropped)
    SELECT s.rule_id, s.fid, s.t, p.id IS NOT NULL AS is_published, d.rule_id IS NOT NULL AS is_dropped,
           p.id IS NOT NULL AND pp.person_role IS DISTINCT FROM s.person_role AS wrong_role
    FROM snapshot s
    LEFT JOIN published p ON p.t = s.t AND p.id = s.fid
    LEFT JOIN dropped d ON d.rule_id = s.rule_id AND d.fact_natural_key = s.fact_natural_key
    LEFT JOIN {schema}.person pp ON pp.person_id = p.person_id""")
    rows = spark.sql("""SELECT rule_id, count(*) n, count_if(is_published) published, count_if(is_dropped) dropped,
        count_if(NOT is_published AND NOT is_dropped) gap, count_if(is_published AND is_dropped) both_,
        count_if(wrong_role) wrong_grain
      FROM _qa_recon GROUP BY rule_id ORDER BY rule_id""").collect()
    results = []
    for x in rows:
        ok = x.gap == 0 and x.both_ == 0 and x.wrong_grain == 0
        results.append((f"reconcile {x.rule_id}", "hard", ok,
                        f"n {x.n} = published {x.published} + dropped {x.dropped}; gap {x.gap}, both {x.both_}, "
                        f"wrong_grain {x.wrong_grain}"))
    extra = scalar(f"""SELECT count(*) FROM ({published}) p
                       LEFT ANTI JOIN (SELECT DISTINCT t, fid FROM _qa_recon) s ON s.t = p.t AND s.fid = p.id""")
    results.append(("no published fact outside the cleaned stg", "hard", extra == 0, f"{extra} rows"))
    for x in spark.sql(f"""SELECT split(reason, ':')[0] AS reason, count(*) n FROM {STG}._dropped
                           GROUP BY 1 ORDER BY 1""").collect():
        results.append((f"dropped: {x.reason}", "report", True, f"{x.n} rows"))
    return results


def run_qa(schema, scope):
    """Run every check on schema, append the results to TARGET._qa_results, print them, and raise if any hard
    check failed. scope is 'staged' (the PUB schema) or 'published' (the target)."""
    assert scope in ("staged", "published"), scope
    results = qa_structure(schema) + qa_visits(schema) + qa_reconcile(schema)

    release = spark.table(f"{STG}._clean_state").first().release_date
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET}")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET}._qa_results ({QA_RESULTS_COLUMNS}) USING DELTA")
    rows = [(SITE_ID, RUN_TAG, release, scope, name, severity, bool(ok), str(metric), now)
            for name, severity, ok, metric in results]
    spark.createDataFrame(rows, QA_RESULTS_COLUMNS.replace("\n", " ")) \
        .write.mode("append").saveAsTable(f"{TARGET}._qa_results")

    for name, severity, ok, metric in results:
        status = "PASS" if ok else ("FAIL" if severity == "hard" else "WARN")
        print(f"{status}  {name}: {metric}")
    failures = [name for name, severity, ok, _ in results if severity == "hard" and not ok]
    if failures:
        raise AssertionError(f"PREBORN QA ({scope}, site {SITE_ID}): {len(failures)} hard failures: {failures}")
    print(f"PREBORN QA ({scope}, site {SITE_ID}): {len(results)} checks, 0 hard failures")


run_qa(PUB, "staged")

## 13. Publish

Publishing copies this site's rows from `PUB` into the target, table by table, replacing the site's previous
rows (other sites are untouched). It is protected three ways:

- **One writer at a time**: a lock row in `{target}._publish_lock`. A lock not refreshed for two hours is
  treated as abandoned.
- **All or nothing**: before writing, each table's current Delta version is recorded in
  `{target}._publish_journal`. If anything fails, including the quality checks re-run on the target
  afterwards, every table is restored to that version.
- **Crash recovery**: if a previous publish died half way (journal still IN_PROGRESS), the next writer
  restores it before doing anything else.

The target's bundled vocabulary is rebuilt across all sites as part of the publish, and the table and column
descriptions below are written to Unity Catalog.

In [0]:
# Unity Catalog descriptions for the published tables and columns. "*" applies to a column name in any table;
# a per-table entry overrides it.
PREBORN_TABLE_COMMENTS = {
    "person": "OMOP CDM PERSON records for pregnant people and infants in PREBORN, with source-site and maternal/infant role lineage.",
    "observation_period": "OMOP CDM OBSERVATION_PERIOD records defining the dates during which each PREBORN person is represented by available clinical data.",
    "condition_occurrence": "OMOP CDM CONDITION_OCCURRENCE records generated from PREBORN maternity and neonatal diagnosis data.",
    "procedure_occurrence": "OMOP CDM PROCEDURE_OCCURRENCE records generated from PREBORN maternity and neonatal care activities.",
    "measurement": "OMOP CDM MEASUREMENT records for structured numeric or coded maternity and neonatal measurements.",
    "observation": "OMOP CDM OBSERVATION records for clinical findings and other non-measurement facts in PREBORN.",
    "visit_occurrence": "OMOP CDM VISIT_OCCURRENCE records representing maternity care contacts linked to people and pregnancies.",
    "visit_detail": "OMOP CDM VISIT_DETAIL records representing detailed neonatal or enrichment encounters within a visit.",
    "death": "OMOP CDM DEATH records for mothers and infants from MSDS, earliest recorded death per person.",
    "location": "OMOP CDM LOCATION records at LSOA 2011 granularity (England/Wales), derived from the mother's MSDS residence; no postcode or address is held.",
    "episode": "OMOP CDM EPISODE records representing pregnancy episodes and their clinical date bounds.",
    "episode_event": "OMOP CDM EPISODE_EVENT bridge linking pregnancy episodes to their constituent clinical events.",
    "fact_relationship": "OMOP CDM FACT_RELATIONSHIP rows linking related PREBORN facts, including pregnancy and infant relationships.",
    "cdm_source": "OMOP CDM_SOURCE metadata describing the PREBORN CDM instance, ETL release and vocabulary versions.",
    "pregnancy": "PREBORN maternity extension containing one row per pregnancy with gestation, outcome, delivery and prior-pregnancy attributes.",
    "infant": "PREBORN maternity extension containing one row per infant with birth outcome, weight, Apgar and congenital-malformation attributes.",
    "concept": "Bundled OMOP CONCEPT vocabulary subset containing concepts used by PREBORN plus the concepts required for referential closure.",
    "concept_relationship": "Bundled OMOP CONCEPT_RELATIONSHIP rows whose source and target concepts are both included in the PREBORN vocabulary subset.",
    "concept_ancestor": "Bundled OMOP CONCEPT_ANCESTOR hierarchy rows needed to provide ancestry for concepts used by PREBORN.",
    "concept_synonym": "Bundled OMOP CONCEPT_SYNONYM rows for concepts included in the PREBORN vocabulary subset.",
    "vocabulary": "OMOP VOCABULARY metadata for the standardised vocabularies represented in the PREBORN concept bundle.",
    "domain": "OMOP DOMAIN metadata defining the clinical and administrative domains used by PREBORN concepts.",
    "concept_class": "OMOP CONCEPT_CLASS metadata defining concept classifications used by the bundled PREBORN vocabulary.",
    "relationship": "OMOP RELATIONSHIP metadata defining relationship types used by the bundled PREBORN vocabulary.",
    "_msds_concept_map": "Every MSDS mapping-spec resolution (question target, answer value, unit, status) used by this site's build, with its method, model version, cosine and margin (best-effort, unreviewed provenance). Code columns carrying SNOMED directly are self-describing via *_source_concept_id.",
    "_concept_set": "Beyond-MSDS maternity concept set: each concept admitted from the EHR and the anchor concepts and groups that admitted it.",
    "_ehr_provenance": "Per-row mapping provenance for beyond-MSDS EHR facts (method, version, cosine and rule id).",
    "_publish_lock": "Operational lock table used to enforce a single PREBORN publisher or vocabulary rebuilder at a time.",
    "_publish_journal": "Durable PREBORN publish journal used to recover or roll back interrupted multi-table publications.",
}

PREBORN_COLUMN_DESCRIPTIONS = {
    "*": {
        "person_id": "PREBORN-minted identifier for the person represented by the record.",
        "gender_concept_id": "Foreign key to CONCEPT representing the person's gender.",
        "year_of_birth": "Four-digit year of birth derived from the available birth datetime.",
        "month_of_birth": "Month of birth, from 1 to 12, when available.",
        "day_of_birth": "Day of month of birth, from 1 to 31, when available.",
        "birth_datetime": "Recorded or derived date and time of birth.",
        "race_concept_id": "Foreign key to CONCEPT representing the person's race.",
        "ethnicity_concept_id": "Foreign key to CONCEPT representing the person's ethnicity.",
        "location_id": "Optional foreign key to the OMOP LOCATION table.",
        "provider_id": "Optional foreign key to the OMOP PROVIDER table.",
        "care_site_id": "Optional foreign key to the OMOP CARE_SITE table.",
        "person_source_value": "Source-system identifier retained for the person.",
        "gender_source_value": "Verbatim source value describing gender or sex.",
        "gender_source_concept_id": "Concept identifier corresponding to the source gender value, or 0 when unmapped.",
        "race_source_value": "Verbatim source value describing race or NHS ethnic category.",
        "race_source_concept_id": "Concept identifier corresponding to the source race value, or 0 when unmapped.",
        "ethnicity_source_value": "Verbatim source value describing ethnicity.",
        "ethnicity_source_concept_id": "Concept identifier corresponding to the source ethnicity value, or 0 when unmapped.",
        "site_id": "Source-site identifier used to partition and publish PREBORN data.",
        "person_role": "PREBORN role of the person, such as mother or infant.",

        "observation_period_id": "Unique identifier for the person's observation period.",
        "observation_period_start_date": "First date on which the person is represented in PREBORN.",
        "observation_period_end_date": "Last date on which the person is represented in PREBORN.",
        "period_type_concept_id": "Foreign key to CONCEPT describing how the observation period was derived.",

        "condition_occurrence_id": "Unique identifier for a condition occurrence.",
        "condition_concept_id": "Foreign key to the standard CONDITION concept represented by the record.",
        "condition_start_date": "Date on which the condition was recorded or inferred to start.",
        "condition_start_datetime": "Date and time on which the condition was recorded or inferred to start.",
        "condition_end_date": "Date on which the condition ended, when available.",
        "condition_end_datetime": "Date and time on which the condition ended, when available.",
        "condition_type_concept_id": "Foreign key to CONCEPT describing the provenance of the condition record.",
        "condition_status_concept_id": "Foreign key to CONCEPT describing the status of the condition.",
        "stop_reason": "Source or derived reason the condition ended.",
        "visit_occurrence_id": "Optional foreign key to the associated VISIT_OCCURRENCE.",
        "visit_detail_id": "Optional foreign key to the associated VISIT_DETAIL.",
        "condition_source_value": "Verbatim diagnosis or condition value from the source data.",
        "condition_source_concept_id": "Concept identifier corresponding to the source condition value, or 0 when unmapped.",
        "condition_status_source_value": "Verbatim source value describing condition status.",

        "procedure_occurrence_id": "Unique identifier for a procedure occurrence.",
        "procedure_concept_id": "Foreign key to the standard PROCEDURE concept represented by the record.",
        "procedure_date": "Date on which the procedure was performed.",
        "procedure_datetime": "Date and time on which the procedure was performed.",
        "procedure_end_date": "Date on which the procedure ended, when available.",
        "procedure_end_datetime": "Date and time on which the procedure ended, when available.",
        "procedure_type_concept_id": "Foreign key to CONCEPT describing the provenance of the procedure record.",
        "modifier_concept_id": "Foreign key to CONCEPT for an optional procedure modifier.",
        "quantity": "Number of procedures or units represented by the record.",
        "procedure_source_value": "Verbatim procedure value from the source data.",
        "procedure_source_concept_id": "Concept identifier corresponding to the source procedure value, or 0 when unmapped.",
        "modifier_source_value": "Verbatim source value for the procedure modifier.",

        "measurement_id": "Unique identifier for a measurement.",
        "measurement_concept_id": "Foreign key to the standard MEASUREMENT concept represented by the record.",
        "measurement_date": "Date on which the measurement was taken.",
        "measurement_datetime": "Date and time on which the measurement was taken.",
        "measurement_time": "Source measurement time when supplied separately from the date.",
        "measurement_type_concept_id": "Foreign key to CONCEPT describing the provenance of the measurement.",
        "operator_concept_id": "Foreign key to CONCEPT representing a comparison operator such as less than or greater than.",
        "value_as_number": "Numeric value of the measurement or observation.",
        "value_as_concept_id": "Foreign key to CONCEPT representing a coded result value.",
        "unit_concept_id": "Foreign key to CONCEPT representing the standard unit.",
        "range_low": "Lower bound of the source reference range.",
        "range_high": "Upper bound of the source reference range.",
        "measurement_source_value": "Verbatim measurement code or label from the source data.",
        "measurement_source_concept_id": "Concept identifier corresponding to the source measurement value, or 0 when unmapped.",
        "unit_source_value": "Verbatim unit supplied by the source data.",
        "unit_source_concept_id": "Concept identifier corresponding to the source unit, or 0 when unmapped.",
        "value_source_value": "Verbatim source representation of the result value.",
        "measurement_event_id": "Identifier of the event to which this measurement is linked.",
        "meas_event_field_concept_id": "Concept identifying the event table field referenced by measurement_event_id.",

        "observation_id": "Unique identifier for an observation.",
        "observation_concept_id": "Foreign key to the standard OBSERVATION concept represented by the record.",
        "observation_date": "Date on which the observation was recorded.",
        "observation_datetime": "Date and time on which the observation was recorded.",
        "observation_type_concept_id": "Foreign key to CONCEPT describing the provenance of the observation.",
        "value_as_string": "Text value of the observation when it is not represented numerically or as a concept.",
        "qualifier_concept_id": "Foreign key to CONCEPT qualifying the observation value.",
        "observation_source_value": "Verbatim observation or finding value from the source data.",
        "observation_source_concept_id": "Concept identifier corresponding to the source observation value, or 0 when unmapped.",
        "qualifier_source_value": "Verbatim source value for the observation qualifier.",
        "observation_event_id": "Identifier of the event to which this observation is linked.",
        "obs_event_field_concept_id": "Concept identifying the event table field referenced by observation_event_id.",

        "visit_concept_id": "Foreign key to CONCEPT representing the standard visit type.",
        "visit_start_date": "Date on which the visit started.",
        "visit_start_datetime": "Date and time on which the visit started.",
        "visit_end_date": "Date on which the visit ended.",
        "visit_end_datetime": "Date and time on which the visit ended.",
        "visit_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit.",
        "visit_source_value": "Verbatim care-contact or visit value from the source data.",
        "visit_source_concept_id": "Concept identifier corresponding to the source visit value, or 0 when unmapped.",
        "admitted_from_concept_id": "Foreign key to CONCEPT representing where the person was admitted from.",
        "admitted_from_source_value": "Verbatim source value describing where the person was admitted from.",
        "discharged_to_concept_id": "Foreign key to CONCEPT representing the discharge destination.",
        "discharged_to_source_value": "Verbatim source value describing the discharge destination.",
        "preceding_visit_occurrence_id": "Identifier of the immediately preceding visit occurrence for the person.",

        "visit_detail_id": "Unique identifier for a detailed visit segment.",
        "visit_detail_concept_id": "Foreign key to CONCEPT representing the standard visit-detail type.",
        "visit_detail_start_date": "Date on which the visit detail started.",
        "visit_detail_start_datetime": "Date and time on which the visit detail started.",
        "visit_detail_end_date": "Date on which the visit detail ended.",
        "visit_detail_end_datetime": "Date and time on which the visit detail ended.",
        "visit_detail_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit detail.",
        "visit_detail_source_value": "Verbatim source value describing the visit detail.",
        "visit_detail_source_concept_id": "Concept identifier corresponding to the source visit-detail value, or 0 when unmapped.",
        "preceding_visit_detail_id": "Identifier of the immediately preceding visit detail for the person.",
        "parent_visit_detail_id": "Identifier of the containing visit detail, when nested.",

        "episode_id": "Unique identifier for a clinical episode.",
        "episode_concept_id": "Foreign key to CONCEPT representing the standard episode type.",
        "episode_start_date": "Date on which the episode started.",
        "episode_start_datetime": "Date and time on which the episode started.",
        "episode_end_date": "Date on which the episode ended.",
        "episode_end_datetime": "Date and time on which the episode ended.",
        "episode_parent_id": "Identifier of a parent episode, when episodes are nested.",
        "episode_number": "Sequence number of the episode for the person.",
        "episode_object_concept_id": "Foreign key to CONCEPT identifying the domain represented by the episode.",
        "episode_type_concept_id": "Foreign key to CONCEPT describing how the episode was derived.",
        "episode_source_value": "Verbatim source value used to derive the episode.",
        "episode_source_concept_id": "Concept identifier corresponding to the source episode value, or 0 when unmapped.",
        "event_id": "Identifier of the clinical event linked to the episode.",
        "episode_event_field_concept_id": "Concept identifying the event-table field referenced by event_id.",

        "domain_concept_id_1": "Concept identifying the domain of the first related fact.",
        "fact_id_1": "Identifier of the first fact in the relationship.",
        "domain_concept_id_2": "Concept identifying the domain of the second related fact.",
        "fact_id_2": "Identifier of the second fact in the relationship.",
        "relationship_concept_id": "Foreign key to CONCEPT describing the relationship between the two facts.",

        "pregnancy_id": "PREBORN-minted identifier for the pregnancy linked to the record.",
        "pregnancy_start_date": "Estimated start date of the pregnancy.",
        "pregnancy_end_date": "Date on which the pregnancy ended.",
        "gestational_length_in_day": "Gestational length in completed days.",
        "pregnancy_outcome_concept_id": "Foreign key to CONCEPT representing the pregnancy outcome.",
        "pregnancy_mode_delivery_concept_id": "Foreign key to CONCEPT representing the mode of delivery.",
        "pregnancy_single_concept_id": "Foreign key to CONCEPT indicating singleton or multiple pregnancy status.",
        "pregnancy_marital_status_concept_id": "Foreign key to CONCEPT representing marital status during pregnancy.",
        "pregnancy_number_fetuses": "Number of fetuses recorded for the pregnancy.",
        "pregnancy_number_liveborn": "Number of liveborn infants recorded for the pregnancy.",
        "prev_pregnancy_parity_concept_id": "Concept representing previous pregnancy parity.",
        "prev_pregnancy_gravidity": "Number of previous pregnancies.",
        "prev_livebirth_number": "Number of previous live births.",
        "prev_still_misc_number": "Number of previous stillbirths or miscarriages.",
        "pre_pregnancy_bmi": "Body mass index recorded or derived before pregnancy.",
        "pregnancy_folic_concept_id": "Concept indicating folic-acid supplementation.",
        "pregnancy_outcome_source_value": "Verbatim source value for pregnancy outcome.",
        "pregnancy_mode_delivery_source_value": "Verbatim source value for mode of delivery.",
        "pre_pregnancy_bmi_source": "Source system or field used for the pre-pregnancy BMI.",
        "pregnancy_marital_status_source_value": "Verbatim source value for marital status during pregnancy.",

        "infant_id": "PREBORN-minted identifier for the infant and corresponding infant person.",
        "birth_outcome_concept_id": "Foreign key to CONCEPT representing the infant birth outcome.",
        "birth_weight": "Recorded birth weight, in grams where supplied by the source.",
        "birth_con_malformation_concept_id": "Concept indicating whether a congenital malformation was recorded.",
        "birth_apgar": "Recorded five-minute Apgar score.",
        "labour_delivery_id": "Source labour or delivery identifier linking the infant to delivery records.",
        "birth_weight_source": "Source system or field used for birth weight.",
        "birth_apgar_source": "Source system or field used for the Apgar score.",

        "source_table": "Name of the source table or transformation arm that produced the record.",
        "enrichment_tag": "Marker distinguishing enrichment-derived rows from common-core rows.",

        "cdm_source_name": "Name of the PREBORN CDM instance.",
        "cdm_source_abbreviation": "Short abbreviation for the PREBORN CDM instance.",
        "cdm_holder": "Organisation responsible for the PREBORN CDM instance.",
        "source_description": "Description of the source data represented by the CDM.",
        "source_documentation_reference": "Reference to documentation describing the source data.",
        "cdm_etl_reference": "Reference or version identifying the PREBORN ETL implementation.",
        "source_release_date": "Release date of the source data used by the ETL.",
        "cdm_release_date": "Release date of the generated PREBORN CDM.",
        "cdm_version": "OMOP Common Data Model version implemented by PREBORN.",
        "cdm_version_concept_id": "Concept identifier representing the OMOP CDM version.",
        "vocabulary_version": "Version of the OMOP vocabulary bundle loaded into PREBORN.",

        "concept_id": "Unique OMOP identifier for the concept.",
        "concept_name": "Unambiguous, human-readable name of the concept.",
        "domain_id": "Identifier of the OMOP domain to which the concept belongs.",
        "vocabulary_id": "Identifier of the source vocabulary from which the concept was drawn.",
        "concept_class_id": "Identifier of the class used to categorise the concept.",
        "standard_concept": "Flag indicating a Standard concept, Classification concept or source concept.",
        "concept_code": "Code assigned to the concept by its source vocabulary.",
        "valid_start_date": "Date on which the vocabulary record became valid.",
        "valid_end_date": "Date on which the vocabulary record ceased to be valid.",
        "invalid_reason": "Reason the vocabulary record was invalidated; null for a currently valid record.",
        "concept_id_1": "Identifier of the source concept in a directional concept relationship.",
        "concept_id_2": "Identifier of the target concept in a directional concept relationship.",
        "relationship_id": "Identifier of the relationship type.",
        "ancestor_concept_id": "Identifier of the ancestor concept in the OMOP hierarchy.",
        "descendant_concept_id": "Identifier of the descendant concept in the OMOP hierarchy.",
        "min_levels_of_separation": "Minimum number of hierarchy levels between ancestor and descendant.",
        "max_levels_of_separation": "Maximum number of hierarchy levels between ancestor and descendant.",
        "concept_synonym_name": "Alternative name or synonym for the concept.",
        "language_concept_id": "Concept identifier representing the synonym language.",
        "vocabulary_name": "Human-readable name of the vocabulary.",
        "vocabulary_reference": "Reference describing the vocabulary or its source.",
        "vocabulary_version": "Loaded version of the vocabulary.",
        "vocabulary_concept_id": "Concept identifier representing the vocabulary.",
        "domain_name": "Human-readable name of the OMOP domain.",
        "domain_concept_id": "Concept identifier representing the domain.",
        "concept_class_name": "Human-readable name of the concept class.",
        "concept_class_concept_id": "Concept identifier representing the concept class.",
        "relationship_name": "Human-readable name of the relationship type.",
        "is_hierarchical": "Flag indicating whether the relationship is hierarchical.",
        "defines_ancestry": "Flag indicating whether the relationship contributes to CONCEPT_ANCESTOR.",
        "reverse_relationship_id": "Identifier of the relationship used in the reverse direction.",

        "death_date": "Date of death; the earliest recorded when MSDS holds more than one.",
        "death_datetime": "Date and time of death; the earliest recorded when MSDS holds more than one.",
        "death_type_concept_id": "Foreign key to CONCEPT describing the provenance of the death record (EHR 32817).",
        "cause_concept_id": "Foreign key to CONCEPT for the cause of death; 0 because MSDS carries no cause.",
        "cause_source_value": "Verbatim cause of death; NULL because MSDS carries no cause.",
        "cause_source_concept_id": "Concept identifier for the source cause of death; 0 when unmapped.",

        "address_1": "Street address line 1; always NULL (PREBORN holds LSOA only).",
        "address_2": "Street address line 2; always NULL (PREBORN holds LSOA only).",
        "city": "City; always NULL (PREBORN holds LSOA only).",
        "state": "State; always NULL (not applicable in the UK).",
        "zip": "Postal code; always NULL (the postcode is never published).",
        "county": "ONS local authority district code (ONSPD oslaua) containing the LSOA.",
        "location_source_value": "ONS LSOA 2011 code of the residence (ONSPD lsoa11, the vintage of the silver IMD deciles).",
        "country_concept_id": "Foreign key to CONCEPT for the country (United Kingdom 42035286).",
        "country_source_value": "Country code of the location ('GB').",
        "latitude": "Latitude; always NULL (a postcode centroid would re-identify the address).",
        "longitude": "Longitude; always NULL (a postcode centroid would re-identify the address).",

        "rule_id": "Mapping-spec or pipeline rule that produced the row.",
        "src_code": "Source code value as recorded in MSDS.",
        "src_text": "Source code description or answer text that was mapped.",
        "method": "How the concept was chosen (spec, snomed_maps_to, embed, ...).",
        "model_version": "Embedding model version for embed-derived mappings; NULL otherwise.",
        "cosine": "Cosine similarity of the chosen concept for embed-derived mappings.",
        "margin": "Cosine margin over the runner-up concept for embed-derived mappings.",
        "rule_ref": "Reference to the spec rule or lookup row the mapping came from.",
        "role": "Role of the concept: target/value/unit/status in _msds_concept_map; anchor/descendant in _concept_set.",
        "anchors": "Anchor concept ids that admitted the concept.",
        "groups": "Maternity concept groups the concept belongs to.",
        "cdm_table": "CDM table holding the row.",
        "cdm_id": "Primary key of the row in cdm_table.",
        "map_method": "Mapping method used for the row.",
        "map_version": "Mapping model or rule-set version.",
        "map_cosine": "Mapping cosine similarity, when embed-derived.",
        "map_rule_id": "Mapping rule id, when rule-derived.",
        "detail": "Human-readable diagnostic, publication or recovery detail.",

        "lock_name": "Name of the operational lock.",
        "token": "Unique token owned by the process holding the lock.",
        "purpose": "Description of the operation protected by the lock.",
        "acquired_at": "Timestamp at which the lock was acquired.",
        "publish_token": "Unique token identifying one publication attempt.",
        "table_name": "Name of the table included in the publication attempt.",
        "version_before": "Delta table version recorded before publication for rollback.",
        "state": "Current state of the publication journal entry.",
        "started_at": "Timestamp at which publication of the table began.",
        "finished_at": "Timestamp at which publication or recovery finished.",
    },
    "person": {
        "person_id": "Primary key for a PREBORN person; minted deterministically within the source site."
    },
    "location": {
        "location_id": "Primary key for an LSOA location: 8e15 + country rank (E01=1, W01=2) x 1e9 + the LSOA digits (the OGR scheme); one row per LSOA per site."
    },
    "pregnancy": {
        "person_id": "Identifier of the pregnant person linked to this pregnancy.",
        "pregnancy_id": "Primary key for a PREBORN pregnancy; minted deterministically within the source site."
    },
    "infant": {
        "pregnancy_id": "Identifier of the pregnancy that produced the infant.",
        "infant_id": "Primary key for the infant extension and identifier of the corresponding infant PERSON row."
    },
    "episode_event": {
        "episode_id": "Identifier of the pregnancy episode containing the linked event."
    },
    "_publish_journal": {
        "site_id": "Source site being published by this journal entry."
    },
}


def apply_table_comments(schema, tables):
    """Write the descriptions above to schema's tables and columns, skipping any that are already current."""
    catalog, schema_name = schema.split(".", 1)
    table_list = ", ".join(f"'{sql_str(t)}'" for t in tables)
    current_table_comments = {
        row["table_name"]: row["comment"] for row in spark.sql(f"""
          SELECT table_name, comment FROM `{catalog}`.information_schema.tables
          WHERE table_schema = '{sql_str(schema_name)}' AND table_name IN ({table_list})""").collect()}
    column_rows = spark.sql(f"""
      SELECT table_name, column_name, comment FROM `{catalog}`.information_schema.columns
      WHERE table_schema = '{sql_str(schema_name)}' AND table_name IN ({table_list})
      ORDER BY table_name, ordinal_position""").collect()

    updates = 0
    for table in tables:
        if table not in current_table_comments:
            continue                                  # the table does not exist in this schema
        wanted = PREBORN_TABLE_COMMENTS.get(table, f"PREBORN table containing {table.replace('_', ' ')} records.")
        if current_table_comments[table] != wanted:
            spark.sql(f"COMMENT ON TABLE `{catalog}`.`{schema_name}`.`{table}` IS '{sql_str(wanted)}'")
            updates += 1
        for row in column_rows:
            if row["table_name"] != table:
                continue
            column = row["column_name"]
            wanted = (PREBORN_COLUMN_DESCRIPTIONS.get(table, {}).get(column)
                      or PREBORN_COLUMN_DESCRIPTIONS["*"].get(column)
                      or f"PREBORN {table.replace('_', ' ')} attribute: {column.replace('_', ' ')}.")
            if row["comment"] != wanted:
                spark.sql(f"ALTER TABLE `{catalog}`.`{schema_name}`.`{table}` ALTER COLUMN `{column}` "
                          f"COMMENT '{sql_str(wanted)}'")
                updates += 1
    print("descriptions updated:", updates)

### The writer lock and the publish journal

In [0]:
def create_publish_controls(target):
    """Create target._publish_lock and target._publish_journal if missing."""
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_lock (
      lock_name STRING, token STRING, purpose STRING, acquired_at TIMESTAMP) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_journal (
      publish_token STRING, site_id STRING, table_name STRING, version_before BIGINT,
      state STRING, started_at TIMESTAMP, finished_at TIMESTAMP, detail STRING) USING DELTA""")
    apply_table_comments(target, ["_publish_lock", "_publish_journal"])


def acquire_write_lock(target, purpose):
    """Take the target's writer lock and return its token; fail if another writer holds it."""
    create_publish_controls(target)
    token = uuid.uuid4().hex
    # The publisher refreshes the lock after every table, so a lock untouched for two hours is abandoned.
    spark.sql(f"""DELETE FROM {target}._publish_lock
      WHERE lock_name = 'exclusive' AND acquired_at < current_timestamp() - INTERVAL 2 HOURS""")
    spark.sql(f"""INSERT INTO {target}._publish_lock
      VALUES ('exclusive', '{token}', '{sql_str(purpose)}', current_timestamp())""")
    holders = spark.sql(f"SELECT token FROM {target}._publish_lock WHERE lock_name = 'exclusive'").collect()
    if len(holders) != 1 or holders[0]["token"] != token:
        spark.sql(f"DELETE FROM {target}._publish_lock WHERE token = '{token}'")
        raise RuntimeError(f"another PREBORN writer holds the lock on {target}")
    return token


def refresh_write_lock(target, token):
    """Show the lock is still in use (see acquire_write_lock)."""
    spark.sql(f"""UPDATE {target}._publish_lock SET acquired_at = current_timestamp()
      WHERE lock_name = 'exclusive' AND token = '{token}'""")


def release_write_lock(target, token):
    """Give up the writer lock taken by acquire_write_lock."""
    spark.sql(f"DELETE FROM {target}._publish_lock WHERE token = '{token}'")


def restore_versions(target, versions):
    """Restore each target table to its recorded version, newest first. Returns any error messages."""
    errors = []
    for table, version in reversed(list(versions.items())):
        try:
            spark.sql(f"RESTORE TABLE {target}.{table} TO VERSION AS OF {version}")
        except Exception as error:
            errors.append(f"{table}: {type(error).__name__}: {error}")
    return errors


def set_journal_state(target, token, state, detail):
    """Mark every journal row of one publish with its final state."""
    spark.sql(f"""UPDATE {target}._publish_journal
      SET state = '{sql_str(state)}', finished_at = current_timestamp(), detail = '{sql_str(detail)[:8000]}'
      WHERE publish_token = '{sql_str(token)}'""")


def recover_interrupted_publishes(target):
    """Restore any earlier publish that stopped before it was marked COMMITTED."""
    rows = spark.sql(f"""SELECT publish_token, table_name, version_before FROM {target}._publish_journal
      WHERE state IN ('IN_PROGRESS', 'RECOVERY_FAILED')
      ORDER BY started_at, publish_token, table_name""").collect()
    tokens = []
    for row in rows:
        if row["publish_token"] not in tokens:
            tokens.append(row["publish_token"])
    for old_token in tokens:
        versions = {row["table_name"]: int(row["version_before"]) for row in rows if row["publish_token"] == old_token}
        errors = restore_versions(target, versions)
        if errors:
            set_journal_state(target, old_token, "RECOVERY_FAILED", "; ".join(errors))
            raise RuntimeError(f"failed to recover interrupted PREBORN publish {old_token}: {errors}")
        set_journal_state(target, old_token, "RECOVERED", "restored automatically before next writer")
        print("recovered interrupted PREBORN publish", old_token)

In [0]:
def publish(target, tables, check_after_publish):
    """Copy this site's rows of `tables` from PUB to target, rebuild the target's vocabulary, run
    check_after_publish(), and commit; on any failure restore every touched table and re-raise."""
    create_cdm_tables(target)
    apply_table_comments(target, list(PREBORN_TABLE_COMMENTS))
    token = acquire_write_lock(target, f"publish {SITE_ID}")
    versions = {}
    journaled = False
    try:
        recover_interrupted_publishes(target)

        # Record where every table we may touch stands now.
        versions = {t: latest_version(f"{target}.{t}") for t in list(tables) + ["cdm_source"] + VOCAB_TABLES}
        journal_rows = ",\n".join(
            f"('{token}', '{SITE}', '{t}', {v}, 'IN_PROGRESS', current_timestamp(), NULL, NULL)"
            for t, v in versions.items())
        spark.sql(f"""INSERT INTO {target}._publish_journal
          (publish_token, site_id, table_name, version_before, state, started_at, finished_at, detail)
          VALUES {journal_rows}""")
        journaled = True

        # Replace this site's rows, table by table, and check the counts match.
        for table in tables:
            columns = ", ".join(table_columns(target, table))
            spark.sql(f"""INSERT INTO {target}.{table} BY NAME
              REPLACE WHERE site_id = '{SITE}'
              SELECT {columns} FROM {PUB}.{table} WHERE site_id = '{SITE}'""")
            expected = scalar(f"SELECT count(*) FROM {PUB}.{table} WHERE site_id = '{SITE}'")
            actual = scalar(f"SELECT count(*) FROM {target}.{table} WHERE site_id = '{SITE}'")
            if actual != expected:
                raise RuntimeError(f"publish count mismatch {table}: target {actual}, staged {expected}")
            refresh_write_lock(target, token)
            print(f"  published {table}: {actual} rows")

        spark.sql(f"INSERT INTO {target}.cdm_source BY NAME REPLACE WHERE true SELECT * FROM {PUB}.cdm_source")
        refresh_write_lock(target, token)

        build_bundled_vocabulary(target)
        apply_table_comments(target, VOCAB_TABLES)
        refresh_write_lock(target, token)

        check_after_publish()
        set_journal_state(target, token, "COMMITTED", "publish and post-validation succeeded")
        print("published", PUB, "->", target, "for site", SITE_ID)
    except Exception as error:
        rollback_errors = restore_versions(target, versions) if versions else []
        if journaled:
            if rollback_errors:
                set_journal_state(target, token, "RECOVERY_FAILED",
                                  f"publish failed: {type(error).__name__}: {error}; rollback errors: {rollback_errors}")
            else:
                set_journal_state(target, token, "ROLLED_BACK",
                                  f"publish failed and was restored: {type(error).__name__}: {error}")
        if rollback_errors:
            raise RuntimeError(f"publish failed ({type(error).__name__}: {error}); "
                               f"rollback errors: {rollback_errors}") from error
        raise
    finally:
        try:
            release_write_lock(target, token)
        except Exception as lock_error:
            print("WARNING: could not release the publish lock; it will be treated as abandoned after two hours:",
                  lock_error)


publish(TARGET, MSDS_PUBLISH_TABLES, check_after_publish=lambda: run_qa(TARGET, "published"))

## 14. Tidy up

The working schemas are dropped, unless `keep_working_schemas` is `yes` (the test fixture keeps them to
inspect `_dropped`).

In [0]:
if KEEP_WORKING_SCHEMAS:
    print("kept working schemas", PUB, "and", STG)
else:
    spark.sql(f"DROP SCHEMA IF EXISTS {STG} CASCADE")
    spark.sql(f"DROP SCHEMA IF EXISTS {PUB} CASCADE")
    print("dropped working schemas for run", RUN_TAG)
print("PREBORN MSDS", SITE_ID, "published to", TARGET)